## 一覽表

In [1]:
# 訓練資料：99組配對破損顱骨對（000_break.xyz ~ 099_break.xyz）
# Groud-truth 輸出：補骨植入物點雲（000_fix.xyz ~ 099_fit.xyz）
# 點雲數量：每組顱骨資料各5000個點
# 模型：
#   Auto Encoder (AE)
#   PointNet++ with Self-Attention
#   3D UNet with Residual Connections and Self-Attention
#   條件Diffusion Model（Latent Diffusion Model）
# 工具：Python（PyTorch、open3d、NumPy）
# 相較於前一版更動：
#               將各階段都加入以下方法：
                        # 每個 batch 成功訓練即重置錯誤計數
                        # 全部 NaN/Inf 檢查（loss、資料、梯度）->若該batch存在NaN/Inf才跳過
                        # 標準梯度裁剪與 grad_norm 記錄
#               將各階段輸出改成英文

## Import Libraries

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.3.0+cu121
12.1
True


In [3]:
import os
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, Subset, DataLoader, WeightedRandomSampler
import torch.optim.lr_scheduler as lr_scheduler
import torchmetrics
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
import numpy as np
import open3d as o3d
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
# from pytorch3d.loss import chamfer_distance
from MTDC_loss import chamfer_distance, adaptive_density_chamfer_distance, torch_normal_consistency_loss, compute_scale_consistency_loss_v2
from MTDC_loss import compute_latent_regularization_v2, compute_local_structure_loss, compute_multiscale_structure_loss_v2, compute_boundary_preservation_loss, point_cloud_uniform_loss, compute_layer_balance_loss
# from MTDC_loss import test
from collections import defaultdict
import json
import time

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
# # 嘗試直接調用 pairwise_chamfer_distance
# if hasattr(torchmetrics.functional, 'pairwise_chamfer_distance'):
#     from torchmetrics.functional import pairwise_chamfer_distance
#     print("pairwise_chamfer_distance 成功導入")
# else:
#     print("torchmetrics 中沒有 pairwise_chamfer_distance，請嘗試其他方法")

# Preparation

In [5]:
# Path handling function
def converted_backslash(original_path):
    converted_path = original_path.replace("\\", "/")
    return converted_path

In [6]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Model

In [7]:
#-------------------------------------------------------------------------------
# Data Preprocessing & Loading
#-------------------------------------------------------------------------------
class SkullDataset2(Dataset):
    def __init__(self, data_dir, num_points=5000, is_test=False):
        self.data_dir = data_dir
        self.num_points = num_points
        self.is_test = is_test
        self.files = [f for f in os.listdir(data_dir) if f.endswith(".xyz")]
        
    def __len__(self):
        return len(self.files)
    
    # 正規化：x_norm = (x_original_points - centroid) / scale
    def normalize_pointcloud(self, break_points, fix_points=None):

        # Center both point clouds at their respective centroids
        break_centroid = np.mean(break_points, axis=0) # 破損點雲的質心
        break_points_centered = break_points - break_centroid #將破損點雲的質心平移到原點

        if fix_points is not None and not self.is_test:
            fix_centroid = np.mean(fix_points, axis=0)
            fix_points_centered = fix_points - fix_centroid

            # Use combined scale for consistent normalization
            combined_points = np.concatenate([break_points_centered, fix_points_centered], axis=0)
            max_dist = np.max(np.sqrt(np.sum(combined_points**2, axis=1)))

            # Apply same scaling to both
            break_points_norm = break_points_centered / max_dist
            fix_points_norm = fix_points_centered / max_dist

            return break_points_norm, fix_points_norm, break_centroid, fix_centroid, max_dist
        
        else:
            # For test data, use only break points for scaling
            max_dist = np.max(np.sqrt(np.sum(break_points_centered**2, axis=1)))
            break_points_norm = break_points_centered / max_dist

            return break_points_norm, None, break_centroid, None, max_dist


    def __getitem__(self, idx):
        break_file = os.path.join(self.data_dir, self.files[idx])
        # fix_file = os.path.join(self.data_dir, self.files[idx].replace("_break.xyz", "_fix.xyz"))

        # Read point cloud data
        break_pcd = o3d.io.read_point_cloud(break_file)
        # fix_pcd = o3d.io.read_point_cloud(fix_file)

        # Get points
        break_points = np.asarray(break_pcd.points)
        # fix_points = np.asarray(fix_pcd.points)

        fix_points = None
        if not self.is_test:
            fix_file = os.path.join(self.data_dir, self.files[idx].replace("_break.xyz", "_fix.xyz"))
            fix_pcd = o3d.io.read_point_cloud(fix_file)
            fix_points = np.asarray(fix_pcd.points)
        # Standardize point count
        if len(break_points) > self.num_points:
            idx = np.random.choice(len(break_points), self.num_points, replace=False)
            break_points = break_points[idx]
        if fix_points is not None and len(fix_points) > self.num_points:
            idx = np.random.choice(len(fix_points), self.num_points, replace=False)
            fix_points = fix_points[idx]
            
        # Normalize point clouds using the same scale
        break_points, fix_points, break_centroid, fix_centroid, max_dist = self.normalize_pointcloud(break_points, fix_points)
        
        # Store normalization parameters
        norm_params = {
            'break_centroid': break_centroid, 
            'scale': max_dist,
            'original_break_size': len(break_points)
        }
        if fix_centroid is not None:
            norm_params['fix_centroid'] = fix_centroid
            norm_params['original_fix_size'] = len(fix_points)

        # Convert to tensors
        break_points = torch.tensor(break_points, dtype=torch.float32)
        fix_points = torch.tensor(fix_points, dtype=torch.float32) if fix_points is not None else None
        norm_params = {k: torch.tensor(v, dtype=torch.float32) if isinstance(v, (int, float, np.ndarray)) else v 
                        for k, v in norm_params.items()}

        return break_points, fix_points, norm_params
    

def collate_fn(batch):
    break_points, fix_points, norm_params = zip(*batch)
    
    # Stack all data
    break_points = torch.stack(break_points)
    fix_points = torch.stack(fix_points)
    
    # Combine all normalization parameters
    combined_norm_params = {}
    for key in norm_params[0].keys():
        combined_norm_params[key] = torch.stack([p[key] for p in norm_params])
    
    return break_points, fix_points, combined_norm_params


#-------------------------------------------------------------------------------
# PointCloud Autoencoder
#-------------------------------------------------------------------------------
class PointCloudAutoencoder(nn.Module):
    def __init__(self, num_points=5000, latent_dim=128, feature_dim=128): # latent_dim: 16->512->256->64 / feature_dim: 64->128->64
        super(PointCloudAutoencoder, self).__init__()
        self.num_points = num_points
        self.latent_dim = latent_dim
        self.feature_dim = feature_dim
        
        # 改進編碼器架構 - 增加更多層以支援更大的潛在維度
        self.encoder = nn.Sequential(
            ResidualBlock(3, 32), 
            ResidualBlock(32, 64), # 08再給他增加一層
            ResidualBlock(64, 128),
            ResidualBlock(128, 256),
            ResidualBlock(256, 512),
            ResidualBlock(512, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(latent_dim), # [B, 64, 512]
        )

        # 強化Transformer
        self.transformer = nn.Sequential(
            PointTransformerBlock(dim=feature_dim, num_heads=8),
            PointTransformerBlock(dim=feature_dim, num_heads=8)
            # for _ in range(2) # 增加層數
        )

        # 改進解碼器 - 支援更大的潛在維度
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim * feature_dim, 4096), # # [64*64=4096, 4096]
            nn.LayerNorm(4096),  # 替換BatchNorm1d為LayerNorm
            nn.GELU(), # ReLU->GELU
            nn.Dropout(p=0.1), # 減少一點dropout
            nn.Linear(4096, 8192),
            nn.LayerNorm(8192),
            nn.GELU(),
            nn.Dropout(p=0.05),
            nn.Linear(8192, 4096),
            nn.LayerNorm(4096),
            nn.GELU(),
            nn.Linear(4096, num_points * 3), # [4096, 5000*3=15000]
        )

        # 尺度約束改進 
        self.scale_constraint = nn.Sequential(
            nn.Linear(latent_dim * feature_dim, 512),  # [512*64=32768, 512]
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Tanh()  # 輸出-1到1的調整因子
        )
    
    def encode(self, x):
        x = x.transpose(1, 2)  # [B, 3, N] = [B, 3, 5000]
        x = self.encoder(x)    # [B, feature_dim, latent_dim] = [B, 64, 512]
        x = x.transpose(1, 2)  # [B, latent_dim, feature_dim] = [B, 512, 64]
        # print(f"encode output shape: {x.shape}")  # 除錯
        # 多層Transformer處理
        for block in self.transformer:
            x = block(x) + x * 0.1  # 殘差連接
            
        return x.contiguous()
    
    def decode(self, z, apply_scale_constraint=True):
        # z shape: [B, latent_dim, feature_dim] = [B, 512, 64]
        batch_size = z.size(0) 
        # print(batch_size)
        z_flat = z.reshape(batch_size, -1)  # [B, latent_dim * feature_dim] = [B, 512*64]
        # print(f"z_flat shape: {z_flat.shape}")  # 除錯
        
        x = self.decoder(z_flat)  # [B, num_points * 3] = [B, 15000]
        x = x.view(batch_size, self.num_points, 3)  # [B, 5000, 3]
        # print(f"x.shape: {x.shape}") # 除錯

        if apply_scale_constraint:
            # 應用尺度約束
            scale_factor = self.scale_constraint(z_flat) * 0.2 + 1.0  # 0.8~1.2的範圍縮放 # [B, 1]
            x = x * scale_factor.unsqueeze(1)  # [B, 5000, 3]
            
        # print(f"decode output shape: {x.shape}")  # 除錯，應為 [16, 5000, 3]
        return x
    
    def forward(self, x):
        # x: [B, N, 3]
        z = self.encode(x) # [B, latent_dim, feature_dim] = [B, 512, 64]
        # print(f"z shape: {z.shape}")  # 除錯，確認 z 形狀
        x_recon = self.decode(z) # [B, 5000, 3]
        # print(f"forward x_recon shape: {x_recon.shape}") # 除錯
        return x_recon, z

    def compute_scale_loss(self, generated_points, target_points):
        """計算尺度一致性損失"""
        # 計算生成點雲和目標點雲的尺度
        gen_scale = torch.sqrt(torch.sum(generated_points**2, dim=-1)).max(dim=-1)[0]
        target_scale = torch.sqrt(torch.sum(target_points**2, dim=-1)).max(dim=-1)[0]
        
        # 尺度比例損失
        scale_ratio = gen_scale / (target_scale + 1e-8)
        scale_loss = F.mse_loss(scale_ratio, torch.ones_like(scale_ratio))
        
        return scale_loss
    
#-------------------------------------------------------------------------------
# PointNet++ Modules
#-------------------------------------------------------------------------------

def square_distance(src, dst):
    """Calculate Euclid distance between each two points."""
    B, N, _ = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, -1).view(B, N, 1)
    dist += torch.sum(dst ** 2, -1).view(B, 1, M)
    return dist


def index_points(points, idx):
    """Group points by index"""
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points


def farthest_point_sample(xyz, npoint):
    """Farthest Point Sampling (FPS)"""
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids


def query_ball_point(radius, nsample, xyz, new_xyz):
    """Group points within a radius"""
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long).to(device).view(1, 1, N).repeat([B, S, 1])
    sqrdists = square_distance(new_xyz, xyz)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat([1, 1, nsample])
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx


class PointNetSetAbstraction(nn.Module):
    """PointNet++ Set Abstraction Layer"""
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all=False):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample
        self.group_all = group_all
        
        # MLP layers with residual connections
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        self.mlp_residuals = nn.ModuleList()
        
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            
            # Add residual connection if dimensions match
            if last_channel == out_channel:
                self.mlp_residuals.append(nn.Identity())
            else:
                self.mlp_residuals.append(nn.Conv2d(last_channel, out_channel, 1))
                
            last_channel = out_channel

    def forward(self, xyz, points):
        xyz = xyz.contiguous()
        if points is not None:
            points = points.contiguous()

        if self.group_all:
            new_xyz = torch.mean(xyz, dim=1, keepdim=True)
            grouped_xyz = xyz.view(xyz.shape[0], 1, xyz.shape[1], 3) - new_xyz.view(xyz.shape[0], 1, 1, 3)
            if points is not None:
                grouped_points = points.view(points.shape[0], 1, points.shape[1], -1).repeat(1, 1, 1, 1)
            else:
                grouped_points = grouped_xyz
        else:
            # Farthest point sampling for new centroids
            fps_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, fps_idx)
            
            # Ball query to group points
            idx = query_ball_point(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, idx)
            grouped_xyz -= new_xyz.view(xyz.shape[0], self.npoint, 1, 3)
            
            if points is not None:
                grouped_points = index_points(points, idx)
                # grouped_points = torch.cat([grouped_xyz, grouped_points], dim=-1) # 註解掉以防止通道數不一致，只使用特徵不拼接
            else:
                grouped_points = grouped_xyz
        
        # Reshape for 2D convolution
        grouped_points = grouped_points.permute(0, 3, 2, 1)  # [B, C, nsample, npoint]
        
        # Apply MLPs with residual connections
        for i, conv in enumerate(self.mlp_convs):
            identity = grouped_points
            grouped_points = F.relu(self.mlp_bns[i](conv(grouped_points)))
            
            # Add residual connection
            res = self.mlp_residuals[i](identity)
            if res.shape == grouped_points.shape:
                grouped_points = grouped_points + res
        
        # Max pooling
        new_points = torch.max(grouped_points, 2)[0]  # [B, C, npoint]
        new_points = new_points.permute(0, 2, 1)  # [B, npoint, C]
        
        return new_xyz, new_points


#-------------------------------------------------------------------------------
# Self-Attention Module
#-------------------------------------------------------------------------------

class SelfAttention(nn.Module):
    """Self-attention module for point clouds"""
    def __init__(self, dim, num_heads=4):
        super(SelfAttention, self).__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
    def forward(self, x):
        B, N, C = x.shape
        
        # Generate query, key, value
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # [B, heads, N, head_dim]
        
        # Compute attention
        attn = (q @ k.transpose(-2, -1)) * self.scale  # [B, heads, N, N]
        attn = attn.softmax(dim=-1)
        
        # Apply attention
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # [B, N, C]
        x = self.proj(x)
        
        return x


#-------------------------------------------------------------------------------
# Enhanced PointNet++ Encoder with Self-Attention
#-------------------------------------------------------------------------------

class PointNetPlusPlusEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super(PointNetPlusPlusEncoder, self).__init__()
        
        # Set Abstraction layers
        self.sa1 = PointNetSetAbstraction(npoint=512, radius=0.2, nsample=32, in_channel=3, mlp=[32, 64])
        self.sa2 = PointNetSetAbstraction(npoint=128, radius=0.4, nsample=64, in_channel=64, mlp=[64, 128])    # ← 修正：in_channel=64
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=128, mlp=[128, feature_dim], group_all=True)  # ← 修正：in_channel=128
        
        # Self-attention modules for each level of abstraction
        self.attn1 = SelfAttention(64)
        self.attn2 = SelfAttention(128)
        self.attn3 = SelfAttention(feature_dim)
        
        # Multi-scale feature fusion
        self.fusion = nn.Sequential(
            nn.Linear(64 + 128 + feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, feature_dim)
        )
        
    def forward(self, xyz):
        # 檢查並轉換輸入格式
        # if xyz.shape[-1] == 3:  # [B, N, 3] -> [B, 3, N]
        #     xyz = xyz.permute(0, 2, 1)
        
        B, _, N = xyz.shape  # 現在是 [B, 3, N]
        
        # First set abstraction
        xyz1, points1 = self.sa1(xyz, None)
        points1 = self.attn1(points1)  # Apply self-attention
        points1_global = torch.max(points1, dim=1, keepdim=True)[0].expand(-1, N, -1)
        
        # Second set abstraction
        xyz2, points2 = self.sa2(xyz1, points1)
        points2 = self.attn2(points2)  # Apply self-attention
        points2_global = torch.max(points2, dim=1, keepdim=True)[0].expand(-1, N, -1)
        
        # Third set abstraction (global)
        _, points3 = self.sa3(xyz2, points2)
        points3 = self.attn3(points3)  # Apply self-attention
        points3_global = points3.expand(-1, N, -1)
        
        # Multi-scale feature fusion
        multi_scale_features = torch.cat([points1_global, points2_global, points3_global], dim=-1)
        fused_features = self.fusion(multi_scale_features)
        
        # Return global feature vector
        global_feature = torch.max(fused_features, dim=1)[0]
        
        return global_feature

#-------------------------------------------------------------------------------
# Point Transformer Encoder Block
#-------------------------------------------------------------------------------
class PointTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, N, C]
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


#-------------------------------------------------------------------------------
# Modified 3D UNet (with Residual Connections and Attention) for Latent Space
#-------------------------------------------------------------------------------

class Diffusion3DUNet(nn.Module):
    """Enhanced 3D UNet with residual connections and attention for diffusion model"""
    def __init__(self, feature_dim=128, time_dim=128, latent_feature_dim=128):
        super(Diffusion3DUNet, self).__init__()
        self.latent_feature_dim = latent_feature_dim
        base_c = latent_feature_dim
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim * 2), # 4縮減至2
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim)
        )
        
        # 單層線性投影 -> 改成2層結構 
        self.condition_proj = nn.Sequential(
            nn.Linear(feature_dim, latent_feature_dim), # [64, 64]
            nn.ReLU(),
            nn.Linear(latent_feature_dim, latent_feature_dim) # [64, 64]
        )
        # time embeddings
        self.time_proj = nn.Linear(time_dim, latent_feature_dim)
        
        # 增加尺度預測分支
        self.scale_predictor = nn.Sequential(
            nn.Linear(feature_dim + time_dim, base_c * 2),
            nn.ReLU(),
            nn.Linear(base_c * 2, base_c),
            nn.ReLU(),
            nn.Linear(base_c, 1),
            nn.Tanh()  # 輸出-1~1的尺度調整因子
        )

        # dynamic channels for encoder and decoder blocks
        chs = [base_c//1, base_c *2, base_c *4, base_c *8, base_c *8] # [64, 128, 256, 512,512]

        # Encoder blocks with residual connections
        self.enc1 = ResidualBlock(latent_feature_dim, chs[0])
        self.enc2 = ResidualBlock(chs[0], chs[1])
        self.enc3 = ResidualBlock(chs[1], chs[2])
        self.enc4 = ResidualBlock(chs[2], chs[3])
        self.enc5 = ResidualBlock(chs[3], chs[4])

        # Middle block with attention
        self.mid_attn1 = SelfAttention(chs[4])
        self.mid_block1 = ResidualBlock(chs[4], chs[4])
        self.mid_attn2 = SelfAttention(chs[4])
        self.mid_ptblock1 = PointTransformerBlock(dim=chs[4], num_heads=4)
        self.mid_block2 = ResidualBlock(chs[4], chs[4])
        self.mid_ptblock2 = PointTransformerBlock(dim=chs[4], num_heads=4)
        
        # Decoder blocks with residual connections and skip connections
        self.dec5 = ResidualBlock(chs[4] + chs[3], chs[4]) # Skip connection from enc4
        self.dec4 = ResidualBlock(chs[3] + chs[2], chs[2]) # Skip connection from enc3
        self.dec3 = ResidualBlock(chs[2] + chs[1], chs[1]) # Skip connection from enc2
        self.dec2 = ResidualBlock(chs[1] + chs[0], latent_feature_dim) # Skip connection from enc1
        self.dec1 = nn.Sequential(
            nn.Conv1d(latent_feature_dim, latent_feature_dim, kernel_size=1),
            nn.Tanh()  # 限制輸出範圍
        )
        
        # Additional attention layers 
        self.attn1 = SelfAttention(chs[0])
        self.attn2 = SelfAttention(chs[1])
        self.attn3 = SelfAttention(chs[2])
        self.attn4 = SelfAttention(chs[3])

    def forward(self, z, t, condition):
        """
        z: [B, latent_dim, latent_feature_dim] - latent representation
        t: [B] - Timestep
        condition: [B, feature_dim] - Condition feature from encoder
        """
        B, N, C = z.shape  # [B, 64, 64]
        
        # Time embedding
        t_emb = self.time_mlp(t.unsqueeze(-1))  # [B] -> [B, time_dim]
        
        # Project condition and time embeddings
        condition_feat = self.condition_proj(condition)  # [B, latent_feature_dim]
        time_feat = self.time_proj(t_emb)  # [B, latent_feature_dim]
        
        # 預測尺度調整因子
        scale_adjustment = self.scale_predictor(torch.cat([condition, t_emb], dim=-1))

        # Reshape z for processing
        z = z.transpose(1, 2)  # [B, latent_feature_dim, latent_dim] (B, 128, 256)
        
        # 條件注入 - 更精細的方式
        condition_broadcast = (condition_feat + time_feat).unsqueeze(-1).expand(-1, -1, N)
        z = z + condition_broadcast * 0.5  # 降低條件影響強度

        # # Initialize feature with latent vector, modified by condition and time
        # # We don't concatenate, just add them as influences
        # for i in range(N):
        #     z[:, :, i] = z[:, :, i] + condition_feat + time_feat
        
        # Encoder path with enhanced skip connections (UNet pipeline)
        skip_connections = []
        
        e1 = self.enc1(z)  # [B, 128, 256] -> [B, 64, 256]
        e1 = e1 + self.attn1(e1.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e1)
        
        e2 = self.enc2(e1) # [B, 64, 256] -> [B, 128, 256]
        e2 = e2 + self.attn2(e2.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e2)
        
        e3 = self.enc3(e2) # [B, 128, 256] -> [B, 256, 256]
        e3 = e3 + self.attn3(e3.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e3)
        
        e4 = self.enc4(e3) # [B, 256, 256] -> [B, 512, 256]
        e4 = e4 + self.attn4(e4.transpose(1, 2)).transpose(1, 2) * 0.1
        skip_connections.append(e4)
        
        e5 = self.enc5(e4) # [B, 512, 256] -> [B, 512, 256]
        
        # Middle block with attention
        mid = e5
        mid = mid + self.mid_attn1(mid.transpose(1, 2)).transpose(1, 2) * 0.1
        mid = self.mid_block1(mid)
        mid = mid + self.mid_attn2(mid.transpose(1, 2)).transpose(1, 2) * 0.1
        mid = self.mid_block2(mid) 
        mid_ptin = mid.transpose(1,2) # [B, 256, 512]
        mid_ptout = self.mid_ptblock1(mid_ptin) # [B, N, C]
        mid_ptout = self.mid_ptblock2(mid_ptout) # [B, N, C]
        mid = mid_ptout.transpose(1,2) # 回到[B, C, N](B, 512, 256)，後續給decoder使用
        
        # Decoder path with skip connections
        d5 = self.dec5(torch.cat([mid, skip_connections[3]], dim=1))
        d4 = self.dec4(torch.cat([d5, skip_connections[2]], dim=1))
        d3 = self.dec3(torch.cat([d4, skip_connections[1]], dim=1))
        d2 = self.dec2(torch.cat([d3, skip_connections[0]], dim=1))
        d1 = self.dec1(d2) # [B, 128, 256]

        # 應用尺度調整
        scale_factor = 1.0 + scale_adjustment.unsqueeze(-1) * 0.1  # 小幅度調整
        d1 = d1 * scale_factor

        # Output
        output = d1.transpose(1, 2)  # [B, 256, 128]
        
        return output
 
#-------------------------------------------------------------------------------
# Revised ResidualBlock with proper initialization
#-------------------------------------------------------------------------------
class ResidualBlock(nn.Module):
    """Residual block for 3D UNet"""
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        
        # Print channels for debugging
        # print(f"Creating ResidualBlock with in_channels={in_channels}, out_channels={out_channels}")
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        # Residual connection
        self.shortcut = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
        # # Residual connection
        # self.shortcut = nn.Sequential()
        # if in_channels != out_channels:
        #     self.shortcut = nn.Sequential(
        #         nn.Conv1d(in_channels, out_channels, kernel_size=1),
        #         nn.BatchNorm1d(out_channels)
        #     )
    
    # def forward(self, x):
    #     # Print shape for debugging
    #     # print(f"ResidualBlock input shape: {x.shape}")
        
    #     residual = self.shortcut(x)
    #     x = F.relu(self.bn1(self.conv1(x)))
    #     x = self.bn2(self.conv2(x))
    #     x += residual # Add residual connection
    #     x = F.relu(x)
        
    #     # Print shape for debugging
    #     # print(f"ResidualBlock output shape: {x.shape}")
        
    #     return x
    
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.relu(out)
        return out
    
#-------------------------------------------------------------------------------
# Latent Diffusion Model
#-------------------------------------------------------------------------------

class EnhancedConditionalDiffusionModel(nn.Module):
    def __init__(self, autoencoder=None, feature_dim=128, beta_schedule='cosine', num_points=5000, num_timesteps=500):
        super(EnhancedConditionalDiffusionModel, self).__init__()
        self.num_points = num_points

        if autoencoder is None:
            raise ValueError("預訓練的autoencoder必須提供")
        self.autoencoder = autoencoder
        self.latent_dim = autoencoder.latent_dim        # 64
        self.latent_feature_dim = autoencoder.feature_dim   # 64        
        self.encoder = PointNetPlusPlusEncoder(feature_dim=feature_dim)
        self.unet = Diffusion3DUNet(feature_dim=feature_dim, time_dim=128, latent_feature_dim=self.latent_feature_dim)
        self.num_timesteps = num_timesteps
        self.latent_dim = autoencoder.latent_dim
        self.latent_feature_dim = autoencoder.feature_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        if beta_schedule == 'linear':
            self.beta = torch.linspace(1e-4, 0.02, self.num_timesteps, device=self.device)
        elif beta_schedule == 'cosine':
            steps = self.num_timesteps + 1
            x = torch.linspace(0, self.num_timesteps, steps, device=self.device)
            alphas_cumprod = torch.cos(((x / self.num_timesteps) + 0.008) * np.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            self.beta = torch.clip(betas, 0.0001, 0.9999)
        
        self.alpha = 1. - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1. - self.alpha_cumprod)

    def forward(self, x, t, condition_points):
        """
        Modified forward method to match the expected shapes and debug the flow
        
        x: The noisy latent representation [B, latent_dim, latent_feature_dim]
        t: Timestep [B]
        condition_points: Condition point cloud [B, N, 3]
        """
        # Print shapes for debugging
        # print(f"EnhancedConditionalDiffusionModel forward:")
        # print(f"  x shape: {x.shape}")
        # print(f"  t shape: {t.shape}")
        # print(f"  condition_points shape: {condition_points.shape}")
        
        # Get condition features from encoder
        condition = self.encoder(condition_points)
        # print(f"  condition shape after encoding: {condition.shape}")
        
        # Predict noise using UNet
        pred_noise = self.unet(x, t / self.num_timesteps, condition)
        # print(f"  pred_noise shape: {pred_noise.shape}")
        
        return pred_noise

    def add_noise(self, z_0, t):
        """Add noise to the latent representation at specified timestep"""
        # Print shapes for debugging
        # print(f"add_noise:")
        # print(f"  z_0 shape: {z_0.shape}")
        # print(f"  t shape: {t.shape}")
        
        noise = torch.randn_like(z_0) * 0.8 # 降低噪聲強度
        sqrt_alpha_cumprod_t = self.sqrt_alpha_cumprod[t].view(-1, 1, 1)
        sqrt_one_minus_alpha_cumprod_t = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1)
        z_t = sqrt_alpha_cumprod_t * z_0 + sqrt_one_minus_alpha_cumprod_t * noise
        
        # print(f"  z_t shape: {z_t.shape}")
        # print(f"  noise shape: {noise.shape}")
        
        return z_t, noise

    def sample(self, condition_points, num_points=5000, denormalize_params=None, is_test=False, guidance_scale=1.0):
        self.eval()
        with torch.no_grad():
            # 處理形狀
            if condition_points.dim() == 2:
                # [N, 3] -> [1, N, 3]
                condition_points = condition_points.unsqueeze(0)
            elif condition_points.dim() == 4 and condition_points.shape[0] == 1:
                # [1, B, N, 3] -> [B, N, 3]
                condition_points = condition_points.squeeze(0)
            # print shape for debug
            print("condition_points shape inside model.sample():", condition_points.shape)
            B, N, _ = condition_points.shape
            device = condition_points.device

            # 初始化潛在向量時使用更小的標準差
            z = torch.randn(B, self.latent_dim, self.latent_feature_dim).to(device) * 0.8
            condition = self.encoder(condition_points)

            # 計算條件點雲的尺度作為參考
            condition_scale = torch.sqrt(torch.sum(condition_points**2, dim=-1)).max(dim=-1)[0]

            for t in tqdm(reversed(range(self.num_timesteps)), desc="Denoise Progress", total=self.num_timesteps):
                t_tensor = torch.full((B,), t, device=device, dtype=torch.long)
                
                # 預測噪聲
                predicted_noise = self.unet(z, t_tensor / self.num_timesteps, condition)
                """
                 # Classifier-free guidance (可選)
                if guidance_scale > 1.0 and t > 50:  # 只在早期時間步應用
                    uncond_noise = self.unet(z, t_tensor / self.num_timesteps, torch.zeros_like(condition))
                    predicted_noise = uncond_noise + guidance_scale * (predicted_noise - uncond_noise)
                """
                # 去噪步驟
                alpha_t = self.alpha[t]
                alpha_cumprod_t = self.alpha_cumprod[t]
                beta_t = self.beta[t]

                # 添加噪聲時使用更小的標準差
                noise = torch.randn_like(z) * 0.5 if t > 0 else torch.zeros_like(z)

                z = (1 / torch.sqrt(alpha_t)) * (
                    z - (beta_t / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise
                ) + torch.sqrt(beta_t) * noise

                # 尺度約束 -> 在後期時間步進行
                if t < 100:
                    current_scale = torch.sqrt(torch.sum(z**2, dim=-1)).max(dim=-1)[0] # B
                    target_scale = 1.0 # 歸一化空間的目標尺度
                    factor = torch.clamp(target_scale / current_scale, max=1.0) # 若大於target_scale*2，才需要縮放，其餘維持原值（因 clamp）
                    z = z * factor.view(-1, 1, 1) # [B, latent_dim, latent_feature_dim]
                    # # 建立布林遮罩(此方法依舊報錯)
                    # mask = current_scale > (target_scale * 2)
                    # if mask.any():
                    #     # 只修正需要修正的batch
                    #     scales = current_scale[mask].unsqueeze(-1).unsqueeze(-1)
                    #     z[mask] = z[mask] * (target_scale / scales)
            
            # 解碼到點雲空間
            x = self.autoencoder.decode(z, apply_scale_constraint=True)
            

            # 反歸一化
            if denormalize_params is not None:
                scale = denormalize_params['scale']
                if scale.dim() == 1:
                    scale = scale.view(-1, 1, 1)
                elif scale.dim() == 0:
                    scale = scale.view(1, 1, 1).expand(B, 1, 1)
                centroid_key = 'fix_centroid' if (not is_test and 'fix_centroid' in denormalize_params) else 'break_centroid'
                centroid = denormalize_params[centroid_key]
                if centroid.dim() == 1:
                    centroid = centroid.view(1, 1, 3).expand(B, 1, 3)
                elif centroid.dim() == 2:
                    centroid = centroid.view(B, 1, 3)
                x = x * scale + centroid

                target_scale = condition_scale.view(B, 1, 1) * 0.8
                current_scale = torch.sqrt(torch.sum(x**2, dim=-1)).max(dim=-1)[0].view(B, 1, 1)
                scale_correction = target_scale / (current_scale + 1e-8)
                x = x * scale_correction * scale + centroid

            return x.squeeze(0) if B == 1 else x
        

    def forward_encoder_only(self, latent_or_image):
        """僅使用 encoder 的前向傳播"""
        return self.forward_encoder_only_impl(latent_or_image)
    

    def forward_encoder_only_impl(self, latent_or_image):
        """
        根據輸入型態取得條件特徵嵌入。
        支援：
        - 潛在空間 (形狀 [B, latent_dim, latent_feature_dim])
        - 點雲 (形狀 [B, N, 3])
        - 可擴展其他型態（如圖片，依encoder設計）
        """
        # 一般 3D point cloud: [B, N, 3]
        if latent_or_image.dim() == 3:
            # 潛在空間：維度通常為 [B, latent_dim, latent_feature_dim]
            if latent_or_image.shape[1] == self.latent_dim and latent_or_image.shape[2] == self.latent_feature_dim:
                # 潛在 → decode → encoder
                with torch.no_grad():
                    decoded = self.autoencoder.decode(latent_or_image)    # 取出點雲 [B, N, 3]
                return self.encoder(decoded)                              # 抽取條件特徵
            # 點雲直接處理：[B, N, 3]
            elif latent_or_image.shape[2] == 3:
                return self.encoder(latent_or_image)
            else:
                raise ValueError(f"forward_encoder_only_impl: 不支援的 [B, {latent_or_image.shape[1]}, {latent_or_image.shape[2]}] 輸入型態")
        # 圖像/其他張量格式（假設 encoder 支援）：[B, C, H, W]
        elif latent_or_image.dim() == 4:
            # 此處 encoder likely 支援影像輸入，可直接丟給 encoder
            return self.encoder(latent_or_image)
        else:
            raise ValueError(f"forward_encoder_only_impl: 不支援的輸入 dim={latent_or_image.dim()}, shape={latent_or_image.shape}")

        
    def compute_loss(self, break_points, fix_points, norm_params):
        """Enhanced loss computation with scale regularization"""
        B = break_points.shape[0]
        device = break_points.device
        
        # Encode target points to latent space
        z_0, _ = self.autoencoder(fix_points)
        
        # Sample random timesteps
        t = torch.randint(0, self.num_timesteps, (B,), device=device)
        
        # Add noise
        z_t, noise = self.add_noise(z_0, t)
        
        # Predict noise
        predicted_noise = self.forward(z_t, t, break_points)
        
        # Compute main diffusion loss
        diff_loss = F.mse_loss(predicted_noise, noise)
        recon, _ = self.autoencoder(fix_points)
        ae_loss = F.mse_loss(recon, fix_points)
        
        weight_lambda = 0.2
        total_loss = diff_loss + weight_lambda *ae_loss
        
        # 添加尺度一致性損失
        if t.mean() < self.num_timesteps * 0.3:  # 在後期時間步添加尺度約束
            reconstructed = self.autoencoder.decode(z_t - predicted_noise)
            scale_loss = self.autoencoder.compute_scale_loss(reconstructed, fix_points)
            diff_loss = diff_loss + 0.1 * scale_loss
        
        return diff_loss   

In [8]:
def compute_sample_weights(dataset):
    """
    功能：計算每筆樣本的「重要性/難度」分數，並轉化成一維權重列表 (weights)，
        用來給 WeightedRandomSampler 讓訓練時更頻繁采樣重要/難樣本。

    權重範例指標：樣本的點雲破損程度（缺失比例）、Chamfer距離（或其它損失值）、局部密度分布、法向量異常等。

    簡單範例：根據樣本點雲的密度或其它特徵計算權重

    dataset: 自訂的點雲Dataset，須能取得單筆點雲數據

    return: list or np.array, 跟dataset長度等長，數值為sampling權重
    """

    weights = []
    for idx in range(len(dataset)):
        # points, _ = dataset[idx]  # 假設 dataset[idx] 回傳 (pointcloud, label) 或類似
        sample = dataset[idx]  # 假設 dataset[idx] 回傳 (pointcloud, label) 或類似
        # print((type(sample)))
        # print((len(sample)) if hasattr(sample, '__len__') else 'no length attribute')
        # print(sample)
        break_points = sample[0] # 取第一個tensor
        break_points = break_points.cpu().numpy() if hasattr(break_points, 'cpu') else break_points


        # 範例1：計算點雲的局部密度(簡單proxy: 反轉點雲的平均距離)
        # 這裡用點之間平均距離的倒數當作難度代表
        # 若點間距小（密集），權重較小；若點間距大（稀疏出現破洞），權重大
        dist_sum = 0
        cnt = 0
        
        for i in range(len(break_points)):
            # 計算與其他點距離(簡單忽略自己)
            dists = np.linalg.norm(break_points - break_points[i], axis=1)
            dists = dists[dists > 0]  # 避免距離0點
            if len(dists) > 0:
                dist_sum += np.min(dists)
                cnt += 1
        
        avg_min_dist = dist_sum / cnt if cnt > 0 else 0.001
        weight = avg_min_dist  # 距離越大，權重越大
        weights.append(weight)

    # 正規化權重(讓所有權重加總=1)
    weights = np.array(weights)
    weights = weights / weights.sum()

    return weights

潛在空間對齊機制

In [9]:
class EnhancedLatentAlignment(nn.Module):
    """增強版潛在空間對齊機制"""
    def __init__(self, autoencoder, alignment_strength=0.1):
        super(EnhancedLatentAlignment, self).__init__()
        self.autoencoder = autoencoder
        self.alignment_strength = alignment_strength
        
        # 獲取AE的維度
        self.latent_dim = autoencoder.latent_dim  # 256
        self.feature_dim = autoencoder.feature_dim  # 128
        total_dim = self.latent_dim * self.feature_dim  # 256 * 128 = 32768
        
        # 潛在空間對齊適配器
        self.latent_adapter = nn.Sequential(
            nn.Linear(total_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, total_dim),
        )
        
        # 分佈對齊層
        self.distribution_aligner = nn.Sequential(
            nn.Linear(total_dim, total_dim),
            nn.Tanh()  # 限制輸出範圍
        )
        
        # 統計資訊估計器（用於學習潛在空間分佈）
        self.mean_estimator = nn.Parameter(torch.zeros(total_dim))
        self.std_estimator = nn.Parameter(torch.ones(total_dim))
        
        # 初始化權重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化權重"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)  # 小的初始化
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def align_latent_space(self, z):
        """對齊潛在空間分佈"""
        B, N, C = z.shape  # [B, 256, 128]
        z_original = z.clone()
        
        # 展平
        z_flat = z.reshape(B, -1)  # [B, 32768]
        
        # 第一階段：適配器處理
        z_adapted = self.latent_adapter(z_flat)
        
        # 第二階段：分佈對齊
        z_aligned = self.distribution_aligner(z_adapted)
        
        # 第三階段：統計對齊（可選）
        z_normalized = self._statistical_alignment(z_aligned)
        
        # 殘差連接，控制對齊強度
        z_final = z_flat + self.alignment_strength * (z_normalized - z_flat)
        
        return z_final.reshape(B, N, C)
    
    def _statistical_alignment(self, z_flat):
        """統計對齊：將潛在向量對齊到學習的分佈"""
        # 計算當前批次統計
        batch_mean = z_flat.mean(dim=0, keepdim=True)
        batch_std = z_flat.std(dim=0, keepdim=True) + 1e-6
        
        # 標準化到單位分佈
        z_standardized = (z_flat - batch_mean) / batch_std
        
        # 重新縮放到學習的目標分佈
        z_aligned = z_standardized * self.std_estimator.unsqueeze(0) + self.mean_estimator.unsqueeze(0)
        
        return z_aligned
    
    def update_statistics(self, z_batch):
        """更新分佈統計（在訓練過程中調用）"""
        with torch.no_grad():
            B, N, C = z_batch.shape
            z_flat = z_batch.reshape(B, -1)
            
            # 指數移動平均更新
            momentum = 0.1
            batch_mean = z_flat.mean(dim=0)
            batch_std = z_flat.std(dim=0)
            
            self.mean_estimator.data = (1 - momentum) * self.mean_estimator.data + momentum * batch_mean
            self.std_estimator.data = (1 - momentum) * self.std_estimator.data + momentum * batch_std

class AlignmentAwareDiffusionModel(EnhancedConditionalDiffusionModel):
    """整合對齊機制的擴散模型"""
    def __init__(self, autoencoder=None, feature_dim=256, beta_schedule='cosine', 
                 num_points=5000, num_timesteps=500):
        super().__init__(autoencoder, feature_dim, beta_schedule, num_points, num_timesteps)
        
        # 添加對齊機制
        self.latent_aligner = EnhancedLatentAlignment(autoencoder)
        
    def forward_with_alignment(self, x, t, condition_points, use_alignment=True):
        """帶對齊的前向傳播"""
        if use_alignment:
            # 對齊潛在表示
            x_aligned = self.latent_aligner.align_latent_space(x)
            return self.forward(x_aligned, t, condition_points)
        else:
            return self.forward(x, t, condition_points)

In [10]:
def integrate_alignment_to_existing_model(diffusion_model, device):
    """將對齊機制集成到現有模型"""
    
    diffusion_model = diffusion_model.to(device)
    # 添加對齊組件
    diffusion_model.latent_aligner = EnhancedLatentAlignment(diffusion_model.autoencoder)
    diffusion_model.latent_aligner = diffusion_model.latent_aligner.to(device)
    # 驗證所有組件都在同一設備
    main_device = next(diffusion_model.parameters()).device
    aligner_device = next(diffusion_model.latent_aligner.parameters()).device
    # print(f"主模型設備: {main_device}")
    # print(f"對齊器設備: {aligner_device}")
    if main_device != aligner_device:
        print("警告: 設備不匹配，正在修正...")
        diffusion_model.latent_aligner = diffusion_model.latent_aligner.to(main_device)
        print(f"對齊器已移至: {main_device}")
        
    # 修改前向傳播方法
    original_forward = diffusion_model.forward
    
    def forward_with_alignment(x, t, condition_points, use_alignment=True):
        if use_alignment and hasattr(diffusion_model, 'latent_aligner'):
            x_aligned = diffusion_model.latent_aligner.align_latent_space(x)
            return original_forward(x_aligned, t, condition_points)
        else:
            return original_forward(x, t, condition_points)
    
    diffusion_model.forward_with_alignment = forward_with_alignment
    return diffusion_model

In [11]:
def process_batch_data(data, device):
    """處理批次數據，確保在正確設備上"""
    try:
        if data is None:
            return None
            
        if isinstance(data, list):
            data = [d for d in data if d is not None]
            if len(data) == 0:
                return None
            data = torch.stack(data)
        
        # 確保數據在正確設備
        data = data.to(device)
        
        # 檢查數據維度
        if len(data.shape) != 3 or data.shape[-1] != 3:
            print(f"警告: 數據維度異常 {data.shape}")
            return None
            
        return data
        
    except Exception as e:
        print(f"數據處理錯誤: {e}")
        return None

## -----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Data Preporcessing

In [12]:
xyz_training_data = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000")
dataset = SkullDataset2(data_dir = xyz_training_data, num_points = 5000)
weights = compute_sample_weights(dataset) # 回傳一個跟dataset長度相同的權重list/array

dataset_indices = list(range(len(dataset)))
train_indices, val_indices = train_test_split(dataset_indices, test_size=0.2, random_state=42)

# 針對訓練集與驗證集分別抽取對應權重
train_weights = [weights[i] for i in train_indices]
val_weights = [weights[i] for i in val_indices]

train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True
)
val_sampler = WeightedRandomSampler(
    weights=val_weights,
    num_samples=len(val_weights),
    replacement=True
)

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = 64,
    sampler = train_sampler,
    shuffle = False, 
    collate_fn = collate_fn,
    # pin_memory=True, # 加速CPU -> GPU數據傳輸
    num_workers=0
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_size = 64,
    sampler = val_sampler,
    shuffle = False, 
    collate_fn = collate_fn,
    num_workers=0
)
print("len(dataset):", len(dataset))
print("max(train_indices):", max(train_indices))
print("max(val_indices):", max(val_indices))
print("min(train_indices):", min(train_indices))
print("min(val_indices):", min(val_indices))
print(f"訓練迭代器長度: {len(train_dataloader)}")
print(f"驗證迭代器長度: {len(val_dataloader)}")

len(dataset): 200
max(train_indices): 199
max(val_indices): 186
min(train_indices): 0
min(val_indices): 9
訓練迭代器長度: 3
驗證迭代器長度: 1


Data Augmentation

In [13]:
def augment(batch_points):
    theta = np.random.uniform(0, 2*np.pi)
    rot = torch.tensor([[np.cos(theta), -np.sin(theta), 0],
                        [np.sin(theta),  np.cos(theta), 0],[0, 0, 1]], dtype=batch_points.dtype, device=batch_points.device)
    return torch.matmul(batch_points, rot)

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Define Training AE Progress

自適應梯度裁剪

In [14]:
class AdaptiveGradientClipper:
    def __init__(self, model, max_norm=1.0, percentile=95):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        
    def clip_gradients(self, loss):
        """自適應梯度裁剪"""
        loss.backward()
        
        # 計算當前梯度範數
        total_norm = 0
        param_count = 0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        total_norm = total_norm ** (1. / 2)
        
        # 更新歷史記錄
        self.grad_history.append(total_norm)
        if len(self.grad_history) > 100:  # 保持最近100步
            self.grad_history.pop(0)
        
        # 動態調整裁剪閾值
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 1.2)
        else:
            clip_norm = self.max_norm
        
        # 執行梯度裁剪
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class StabilizedGradientClipper:
    """更嚴格的梯度裁剪"""
    def __init__(self, model, max_norm=0.8, percentile=90):  # 降低閾值
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        self.stability_counter = 0
        
    def clip_gradients_enhanced(self):
        total_norm = 0
        param_count = 0
        
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        
        if param_count == 0:
            return 0.0, self.max_norm
            
        total_norm = total_norm ** 0.5
        self.grad_history.append(total_norm)
        
        if len(self.grad_history) > 100:  # 縮短歷史窗口
            self.grad_history.pop(0)
        
        # 更嚴格的自適應閾值
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 0.9)  # 更保守，降低係數
        else:
            clip_norm = self.max_norm
        
        # 檢測不穩定性
        if total_norm > clip_norm * 2.0: # 擴散模型對不穩定更敏感：1.5->2.0
            self.stability_counter += 1
        else:
            self.stability_counter = max(0, self.stability_counter - 1)
        
        # 執行裁剪
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

# 在訓練循環中使用
class EnhancedTrainingLoop:
    def __init__(self, model, optimizer, scheduler=None):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.grad_clipper = StabilizedGradientClipper(model, max_norm=1.0)
        self.loss_history = []
        
    def training_step(self, break_points, fix_points, norm_params):
        """增強的訓練步驟"""
        self.optimizer.zero_grad()
        
        # 計算損失
        loss = self.compute_enhanced_loss(break_points, fix_points, norm_params)
        
        # 檢測異常損失
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"Warning: Invalid loss detected: {loss}")
            return None
        
        # 梯度裁剪
        grad_norm, clip_norm = self.grad_clipper.clip_gradients(loss)
        
        # 檢測梯度爆炸
        if grad_norm > 5.0: # 調整閾值
            print(f"Warning: Large gradient norm: {grad_norm:.4f}")
            # 跳過這個批次或降低學習率
            self.optimizer.zero_grad()
            for param_group in self.optimizer.param_groups:
                param_group['lr'] *= 0.95
            return None
        
        # 優化步驟
        self.optimizer.step()
        if self.scheduler:
            self.scheduler.step()
        
        # 記錄損失歷史
        self.loss_history.append(loss.item())
        if len(self.loss_history) > 1000:
            self.loss_history.pop(0)
        
        return {
            'loss': loss.item(),
            'grad_norm': grad_norm,
            'clip_norm': clip_norm,
            'lr': self.optimizer.param_groups['lr']
        }
    

    def progressive_timestep_sampling(self, batch_size, device):
        """漸進式時間步采樣"""
        # 根據訓練進度調整時間步分佈
        # 早期訓練專注於大時間步，後期逐漸增加小時間步
        progress = min(len(self.loss_history) / 10000, 1.0)
        
        if progress < 0.3:
            # 早期：主要采樣大時間步
            min_t = int(self.model.num_timesteps * 0.5)
        elif progress < 0.7:
            # 中期：均勻采樣
            min_t = 0
        else:
            # 後期：增加小時間步權重
            min_t = 0
            # 使用加權采樣
            weights = torch.exp(-torch.arange(self.model.num_timesteps) / 100.0)
            t = torch.multinomial(weights, batch_size, replacement=True)
            return t.to(device)
        
        return torch.randint(min_t, self.model.num_timesteps, (batch_size,), device=device)
    
    def compute_time_weights(self, t):
        """計算時間步權重"""
        # 困難時間步獲得更高權重
        weights = 1.0 / (t.float() + 1e-8)
        weights = weights / weights.max()  # 歸一化
        return weights
    
    def balance_losses(self, diff_loss, recon_loss, scale_loss):
        """動態損失平衡"""
        # 根據訓練進度動態調整權重
        progress = min(len(self.loss_history) / 5000, 1.0)
        
        # 擴散損失權重隨進度增加
        diff_weight = 0.6 + 0.3 * progress
        
        # 重構損失權重隨進度減少
        recon_weight = 0.4 - 0.2 * progress
        
        # 尺度損失權重保持穩定
        scale_weight = 0.1
        
        total_loss = (diff_weight * diff_loss + 
                     recon_weight * recon_loss + 
                     scale_weight * scale_loss)
        
        return total_loss

Warmup + Restarts 學習率調度器

In [27]:
class CosineWarmRestarts:
    def __init__(self, optimizer, T_0=1000, T_mult=2, eta_min=1e-6, base_lr=2e-5):
        self.optimizer = optimizer
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        self.base_lr = base_lr
        self.T_cur = 0
        self.T_i = T_0
        self.cycle = 0
        
    def step(self):
        """執行學習率調度步驟"""
        self.T_cur += 1
        
        if self.T_cur >= self.T_i:
            self.cycle += 1
            self.T_cur = 0
            self.T_i = self.T_i * self.T_mult
            
        # 餘弦退火公式
        lr = self.eta_min + (self.base_lr - self.eta_min) * \
             (1 + np.cos(np.pi * self.T_cur / self.T_i)) / 2
        
        # 添加溫暖重啟機制
        if self.T_cur == 0 and self.cycle > 0:
            lr = self.base_lr * (0.8 ** self.cycle)  # 每次重啟降低最大學習率
            
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
        return lr

In [28]:
def get_linear_warmup_scheduler(optimizer, warmup_epochs):
    def lr_lambda(epoch):
        return min((epoch+1)/warmup_epochs, 1.0)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

Early Stop 早停法

In [29]:
class EarlyStoppingWithStability:
    """結合穩定性監控的早停機制"""
    def __init__(self, patience=7, min_delta=0.001, stability_threshold=5):
        self.patience = patience
        self.min_delta = min_delta
        self.stability_threshold = stability_threshold
        self.counter = 0
        self.best_loss = float('inf')
        self.unstable_epochs = 0
        
    def __call__(self, val_loss, grad_norm):
        # 檢查損失改善
        improved = val_loss < self.best_loss - self.min_delta
        
        if improved:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        
        # 檢查訓練穩定性
        if grad_norm > 2.0:  # 梯度範數過大
            self.unstable_epochs += 1
        else:
            self.unstable_epochs = max(0, self.unstable_epochs - 1)
        
        # 決定是否停止
        patience_exceeded = self.counter >= self.patience
        too_unstable = self.unstable_epochs >= self.stability_threshold
        
        return patience_exceeded or too_unstable


Loss Function

In [30]:
def compute_enhanced_losses(recon_points, fix_points, latent_z, autoencoder):
    """計算增強的多種損失"""
    losses = {}
    
    try:
        # === 核心幾何損失 ===
        # 1-1. 主要距離損失 - DCD
        losses['dcd_loss'], _ = adaptive_density_chamfer_distance(
            recon_points, fix_points, k=8, density_method='knn', adaptive_weight=True
        )

        # 1-2. 傳統Chamfer距離 (僅用於評估)
        losses['cd_loss'], _ = chamfer_distance(recon_points, fix_points)

        # 2-1. 穩健點對點損失 - Huber 
        # losses['huber_loss'] = F.smooth_l1_loss(recon_points, fix_points)

        # 2-2. MSE損失 (僅用於評估)
        losses['mse_loss'] = F.mse_loss(recon_points, fix_points)
        
        # === 結構保持損失 ===
        # 3. 多尺度結構損失
        losses['structure_loss'] = compute_multiscale_structure_loss_v2(recon_points, fix_points)
        #  # 局部結構保持損失 (與multiscale重複)
        # losses['local_structure_loss'] = compute_local_structure_loss(recon_points, fix_points)

        # 4. 邊界保持損失：針對雙層結構
        losses['boundary_loss'] = compute_boundary_preservation_loss(recon_points, fix_points)

        # # === 幾何約束損失 ===
        # # 5. 尺度一致性
        # losses['scale_loss'] = compute_scale_consistency_loss_v2(recon_points, fix_points)
        
        # # 6. 法向量一致性
        # losses['normal_loss'] = torch_normal_consistency_loss(recon_points, fix_points, k=8)
        
        # # === 正則化損失 ===
        # # 7. 潛在空間正則化
        # losses['latent_reg'] = compute_latent_regularization_v2(latent_z)

        # # === 點雲均勻性 ===
        # losses['uniform_loss'] = point_cloud_uniform_loss(recon_points)

        # # ===雙層結構上下均衡===
        # losses['layer_balance_loss'] = compute_layer_balance_loss(recon_points, fix_points)

        

        
       
        
    except Exception as e:
        print(f"損失計算錯誤: {e}")
        # 安全默認值
        device = recon_points.device
        losses = {
            'dcd_loss': torch.tensor(0.1, device=device, requires_grad=True),
            # 'huber_loss': F.smooth_l1_loss(recon_points, fix_points),
            'structure_loss': torch.tensor(0.01, device=device, requires_grad=True),
            'boundary_loss': torch.tensor(0.01, device=device, requires_grad=True),
            # 'scale_loss': torch.tensor(0.01, device=device, requires_grad=True),
            # 'normal_loss': torch.tensor(0.01, device=device, requires_grad=True),
            # 'latent_reg': torch.tensor(0.01, device=device, requires_grad=True),
            # 'uniform_loss': torch.tensor(0.01, device=device, requires_grad=True),
            # 'layer_balance_loss': torch.tensor(0.01, device=device, requires_grad=True),
        }
    
    return losses

def compute_enhanced_losses_amp_compatible(recon_points, fix_points, latent_z, autoencoder):
    """AMP兼容的損失計算"""
    losses = {}
    
    try:
        # 確保所有計算都在autocast外進行需要FP32的操作
        with torch.cuda.amp.autocast(enabled=False):
            # 將張量轉換為FP32進行某些計算
            recon_fp32 = recon_points.float()
            fix_fp32 = fix_points.float()
            
            # 主要幾何損失 - 使用FP32
            losses['dcd_loss'], _ = adaptive_density_chamfer_distance(
                recon_fp32, fix_fp32, k=8, density_method='knn', adaptive_weight=True
            )
            
            # 傳統Chamfer距離
            losses['cd_loss'], _ = chamfer_distance(recon_fp32, fix_fp32)
            
            # 法向一致性損失 - 可能包含特徵值計算
            losses['normal_loss'] = torch_normal_consistency_loss(recon_fp32, fix_fp32, k=8)
        
        # 這些損失可以使用混合精度
        losses['mse_loss'] = F.mse_loss(recon_points, fix_points)
        losses['huber_loss'] = F.smooth_l1_loss(recon_points, fix_points)
        
        # 尺度一致性損失
        losses['scale_loss'] = compute_scale_consistency_loss_v2(recon_points, fix_points)
        
        # 潛在空間正則化
        losses['latent_reg'] = compute_latent_regularization_v2(latent_z)
        
        # 局部結構保持損失 - 使用FP32
        with torch.cuda.amp.autocast(enabled=False):
            losses['local_structure_loss'] = compute_local_structure_loss(recon_fp32, fix_fp32)
        
        # 確保所有損失都有梯度
        for key, loss in losses.items():
            if not loss.requires_grad:
                print(f"警告: {key} 沒有梯度信息")
                losses[key] = loss.clone().requires_grad_(True)
        
    except Exception as e:
        print(f"損失計算錯誤: {e}")
        # 返回安全的默認值，確保有梯度
        device = recon_points.device
        losses = {
            'dcd_loss': torch.tensor(1.0, device=device, requires_grad=True),
            'cd_loss': torch.tensor(0.1, device=device, requires_grad=True),
            'mse_loss': F.mse_loss(recon_points, fix_points),
            'huber_loss': F.smooth_l1_loss(recon_points, fix_points),
            'scale_loss': torch.tensor(0.01, device=device, requires_grad=True),
            'normal_loss': torch.tensor(0.01, device=device, requires_grad=True),
            'latent_reg': compute_latent_regularization_v2(latent_z),
            'local_structure_loss': torch.tensor(0.01, device=device, requires_grad=True)
        }
    
    return losses

def compute_weighted_total_loss(losses, current_epoch, total_epochs):
    """動態權重損失計算"""
    progress = min(current_epoch / total_epochs, 1.0)
    
    # 動態權重策略
    weights = {                       # 遵守7822法則 
        # 主要損失(78%)
        'dcd_loss': 0.78,             # 主要幾何損失，強調主結構與幾何對齊
        # 輔助損失(22%)
        'structure_loss': 0.22,       # 保持層內點雲結構合理鋪展
        # 'boundary_loss': 0.07,        # 邊界保持，針對雙層結構問題
        # 'scale_loss': 0.08,         # 尺度約束，防點雲圓形化、縮扁或膨脹

        # 輔助損失(22%)
        # 'uniform_loss': 0.10,       # 提升，繼續改善左右集中       # 均勻性/密度限制
        # 'huber_loss': 0.06,         # 微升，加強點對點穩定性       # 穩健點對點
        # 'layer_balance_loss':0.03,  # 新增：專門平衡上下層密度
        # 'latent_reg': 0.02,         # 維持                        # 潛在變數正則化(後期漸增)
        # 'normal_loss': 0.01,        # 略減，僅保溫                 # 輕微法向平滑約束，方向一致性
    }
    
    total_loss = sum(weights[key] * loss for key, loss in losses.items() 
                    if key in weights and torch.is_tensor(loss))
    
    return total_loss

In [31]:
def train_enhanced_autoencoder(autoencoder, train_dataloader, val_dataloader, 
                              num_epochs=35, lr=2e-5, device='cuda',
                              save_path_ae="autoencoder_pretrained__v8_0824_0035.pth",
                              log_dir=converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0824_0035\logs\autoencoder_v8_0824_0142")):
    """
    增強版AutoEncoder預訓練函數
    整合了所有改進方案：穩定化訓練、多損失函數、完整監控
    """
    
    # ================================
    # 初始化設置
    # ================================
    autoencoder.to(device)
    
    # 創建日誌目錄
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(os.path.dirname(save_path_ae), exist_ok=True)
    
    # 保存路徑處理  
    save_dir = os.path.dirname(save_path_ae)
    if save_dir:  # 只有當目錄路徑不為空時才創建
        os.makedirs(save_dir, exist_ok=True)
    else:
        # 如果save_path_ae只是文件名，設置默認保存目錄
        default_save_dir = os.path.join(log_dir, 'models')
        os.makedirs(default_save_dir, exist_ok=True)
        save_path_ae = os.path.join(default_save_dir, save_path_ae)
        print(f"保存路徑調整為: {save_path_ae}")
    
    # TensorBoard設置
    writer = SummaryWriter(log_dir=log_dir, flush_secs=30)
    
    # 優化器設置 - 使用AdamW替代Adam
    optimizer = torch.optim.AdamW(
        autoencoder.parameters(), 
        lr=1e-4, 
        betas=(0.9, 0.999),
        weight_decay=1e-4, # 增強正則化
        eps=1e-8
    )

    # 改進的學習率調度器
    warmup_epochs = 15
    scheduler = StableCosineLR(optimizer, T_max=num_epochs, eta_min=lr*0.01)
    # Warm-up 10 epoch
    warmup_sched = get_linear_warmup_scheduler(optimizer, warmup_epochs=warmup_epochs)
    # CosineAnnealing 不重啟
    main_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs-warmup_epochs, eta_min=1e-6
    )


    # # 早停機制
    # early_stopping = EarlyStoppingWithStability(patience=5, min_delta=0.001)

    # 梯度裁剪器
    grad_clipper = StabilizedGradientClipper(autoencoder, max_norm=2.0)

    # 混合精度
    scaler = GradScaler() if device == 'cuda' else None
    scaler = None
    # 訓練監控器
    monitor = EnhancedTrainingMonitor(log_dir)

    
    
    # ================================
    # 損失記錄初始化
    # ================================
    metrics_history = {
        'train': defaultdict(list),
        'val': defaultdict(list),
        'lr': [],
        'grad_norm': [],
        'best_metrics': {'epoch': 0, 'loss': float('inf')}
    }
    
    # ================================
    # 主訓練循環
    # ================================
    print("開始Enhanced AutoEncoder預訓練...")
    print(f"訓練數據批次: {len(train_dataloader)}, 驗證數據批次: {len(val_dataloader)}")
    print(f"設備: {device}, 學習率: {lr}, 訓練輪數: {num_epochs}")
    print("-" * 40)
    
    for epoch in range(num_epochs):
        
        # ============================
        # 訓練階段
        # ============================
        autoencoder.train()
        train_metrics = train_epoch_enhanced(
            autoencoder, train_dataloader, optimizer, 
            grad_clipper, device, epoch, num_epochs, scaler
        )
        
        # ============================
        # 驗證階段
        # ============================
        autoencoder.eval()
        val_metrics = validate_epoch_enhanced(
            autoencoder, val_dataloader, device, epoch
        )
        
        # ============================
        # 學習率調度
        # ============================
        # current_lr = scheduler.step()
        if epoch < warmup_epochs:
            # Warm-up 階段
            warmup_sched.step()
            current_lr = optimizer.param_groups[0]['lr']
        else:
            # 主要調度階段
            main_sched.step()
            current_lr = optimizer.param_groups[0]['lr']
        
        # ============================
        # 記錄和監控
        # ============================
        # 保存指標到歷史記錄
        for key, value in train_metrics.items():
            metrics_history['train'][key].append(value)
        for key, value in val_metrics.items():
            metrics_history['val'][key].append(value)
        
        metrics_history['lr'].append(current_lr)
        if 'grad_norm' in train_metrics:
            metrics_history['grad_norm'].append(train_metrics['grad_norm'])
        
        # 寫入TensorBoard
        log_to_tensorboard(writer, train_metrics, val_metrics, current_lr, epoch)
        
        # 控制台輸出
        print_epoch_summary(epoch, num_epochs, train_metrics, val_metrics, current_lr)
        
        # 監控訓練穩定性
        monitor.check_training_stability(train_metrics, val_metrics, epoch)
        
        # ============================
        # 模型保存策略
        # ============================
        # 檢查是否為最佳模型
        current_loss = val_metrics.get('total_loss', train_metrics.get('total_loss', float('inf')))
        if current_loss < metrics_history['best_metrics']['loss']:
            metrics_history['best_metrics']['loss'] = current_loss
            metrics_history['best_metrics']['epoch'] = epoch
            
            # 保存最佳模型
            best_model_path = save_path_ae.replace('.pth', '_best.pth')
            save_checkpoint(autoencoder, optimizer, scheduler, metrics_history, 
                          best_model_path, epoch, is_best=True)
            print(f"新的最佳模型已保存: {best_model_path}")

        # 定期保存檢查點
        if (epoch + 1) % 30 == 0 or epoch == num_epochs - 1:
            checkpoint_path = save_path_ae.replace('.pth', f'_epoch_{epoch+1}.pth')
            torch.save(autoencoder.state_dict(), checkpoint_path) # 只保存權重
            # save_checkpoint(autoencoder, optimizer, scheduler, metrics_history, 
            #               checkpoint_path, epoch, is_best=False)

        # # 早停檢查
        # if early_stopping(avg_val_loss, grad_norm):
        #     print(f"早停於 Epoch {epoch+1}")
        #     break
        
        # ============================
        # 訓練診斷和警告
        # ============================
        diagnose_training(train_metrics, val_metrics, epoch, metrics_history)
        
        # 動態調整訓練策略
        if epoch > warmup_epochs:
            adjust_training_strategy(optimizer, scheduler, metrics_history, epoch)

        # CPU清理與顯存維護
        grad_clipper.grad_history.clear()
        if device == "cuda":
            torch.cuda.empty_cache()
    
    # ================================
    # 訓練完成後處理
    # ================================
    print("\n" + "="*80)
    print("AutoEncoder預訓練完成!")
    
    # 保存最終模型
    final_save_checkpoint(autoencoder, optimizer, scheduler, metrics_history, save_path_ae)
    
    # 生成訓練報告
    generate_training_report(metrics_history, log_dir, num_epochs)
    
    # 繪製訓練曲線
    plot_training_curves(metrics_history, log_dir)
    
    # 關閉TensorBoard
    writer.close()
    
    print(f"最佳模型 (Epoch {metrics_history['best_metrics']['epoch']}): "
          f"損失 = {metrics_history['best_metrics']['loss']:.6f}")
    print(f"模型已保存至: {save_path_ae}")
    print(f"日誌已保存至: {log_dir}")
    
    return extract_metrics_for_return(metrics_history)

# ================================
# 核心訓練函數
# ================================

def train_epoch_enhanced(autoencoder, train_dataloader, optimizer, grad_clipper, 
                        device, epoch, total_epochs, scaler=None):
    """增強版訓練epoch函數"""
    
    # 累積指標
    epoch_metrics = defaultdict(list)
    
    progress_bar = tqdm(train_dataloader, desc=f"訓練 Epoch {epoch+1}/{total_epochs}")
    
    for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
        
        # 數據預處理和驗證
        if fix_points is None:
            continue
        
        # 處理批次數據
        fix_points = process_batch_data(fix_points, device)
        if fix_points is None:
            continue
            
        # 前向傳播
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                recon_points, latent_z = autoencoder(fix_points)
                # 計算多種損失
                losses = compute_enhanced_losses_amp_compatible(recon_points, fix_points, latent_z, autoencoder)
                # 總損失計算 - 動態權重
                total_loss = compute_weighted_total_loss(losses, epoch, total_epochs)
        
            # 檢查損失有效性
            if not torch.isfinite(total_loss):
                print(f"警告: 檢測到無效損失 (Epoch {epoch}, Batch {batch_idx})")
                continue
        
            # 混和精度反向傳播
            scaler.scale(total_loss).backward()
            
            # 混和精度梯度裁剪
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(autoencoder.parameters(), max_norm = 2.0)
            clip_norm = 2.0

            # 檢測梯度異常
            if grad_norm > 50.0:
                print(f"警告: 梯度爆炸 (Epoch {epoch}, Batch {batch_idx}, GradNorm: {grad_norm:.2f})")
                optimizer.zero_grad()
                continue
            
            # 優化器步驟
            scaler.step(optimizer)
            scaler.update()
        
        else:
            # 傳統訓練（保持原有邏輯作為後備）
            recon_points, latent_z = autoencoder(fix_points)
            losses = compute_enhanced_losses(recon_points, fix_points, latent_z, autoencoder)
            total_loss = compute_weighted_total_loss(losses, epoch, total_epochs)
            
            if not torch.isfinite(total_loss):
                print(f"警告: 檢測到無效損失 (Epoch {epoch}, Batch {batch_idx})")
                continue
            
            total_loss.backward()
            grad_norm, clip_norm = grad_clipper.clip_gradients_enhanced()
            
            if grad_norm > 50.0:
                print(f"警告: 梯度爆炸 (Epoch {epoch}, Batch {batch_idx}, GradNorm: {grad_norm:.2f})")
                optimizer.zero_grad()
                continue
            
            optimizer.step()

        # 記錄指標
        batch_metrics = {
            'total_loss': total_loss.item(),
            'grad_norm': grad_norm,
            'clip_norm': clip_norm,
            **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
        }
        
        for key, value in batch_metrics.items():
            epoch_metrics[key].append(value)
        
        # 更新進度條
        if batch_idx % 10 == 0:
            progress_bar.set_postfix({
                'Loss': f"{total_loss.item():.4f}",
                'DCD': f"{losses['dcd_loss'].item():.4f}",
                'GradNorm': f"{grad_norm:.2f}"
            })
    
    # 計算epoch平均指標
    return {key: np.mean(values) for key, values in epoch_metrics.items()}

def validate_epoch_enhanced(autoencoder, val_dataloader, device, epoch):
    """增強版驗證epoch函數"""
    
    epoch_metrics = defaultdict(list)
    
    with torch.no_grad():
        progress_bar = tqdm(val_dataloader, desc=f"驗證 Epoch {epoch+1}")
        
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue
            
            fix_points = process_batch_data(fix_points, device)
            if fix_points is None:
                continue
            
            # 前向傳播
            recon_points, latent_z = autoencoder(fix_points)
            
            # 計算損失
            losses = compute_enhanced_losses(recon_points, fix_points, latent_z, autoencoder)
            total_loss = compute_weighted_total_loss(losses, epoch, 50)  # 假設總epoch為50
            
            # 記錄指標
            batch_metrics = {
                'total_loss': total_loss.item(),
                **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
            }
            
            for key, value in batch_metrics.items():
                epoch_metrics[key].append(value)
            
            # 更新進度條
            if batch_idx % 10 == 0:
                progress_bar.set_postfix({
                    'Val_Loss': f"{total_loss.item():.4f}",
                    'Val_DCD': f"{losses['dcd_loss'].item():.4f}"
                })
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}

# ================================
# 學習率調度器
# ================================

class StableCosineLR:
    """穩定的餘弦退火調度器"""
    def __init__(self, optimizer, T_max=50, eta_min=1e-6, warmup_epochs=5):
        self.optimizer = optimizer
        self.T_max = T_max
        self.eta_min = eta_min
        self.warmup_epochs = warmup_epochs
        self.base_lr = optimizer.param_groups[0]['lr']
        self.current_epoch = 0
        
    def step(self):
        if self.current_epoch < self.warmup_epochs:
            # Warmup階段
            lr = self.base_lr * (self.current_epoch + 1) / self.warmup_epochs
        else:
            # 餘弦退火，但避免重啟
            progress = (self.current_epoch - self.warmup_epochs) / (self.T_max - self.warmup_epochs)
            lr = self.eta_min + (self.base_lr - self.eta_min) * \
                 (1 + np.cos(np.pi * progress)) / 2
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
        self.current_epoch += 1
        return lr

# ================================
# 訓練監控器
# ================================

class EnhancedTrainingMonitor:
    """增強版訓練監控器"""
    def __init__(self, log_dir):
        self.log_dir = log_dir
        self.stability_window = 50
        
    def check_training_stability(self, train_metrics, val_metrics, epoch):
        """檢查訓練穩定性"""
        warnings = []
        
        # 檢查損失是否過大
        if train_metrics.get('total_loss', 0) > 10.0:
            warnings.append("訓練損失過大")
        
        # 檢查梯度範數
        if train_metrics.get('grad_norm', 0) > 10.0:
            warnings.append("梯度範數過大")
        
        # 檢查驗證損失是否遠大於訓練損失
        train_loss = train_metrics.get('total_loss', 0)
        val_loss = val_metrics.get('total_loss', 0)
        if val_loss > train_loss * 2:
            warnings.append("可能存在過擬合")
        
        if warnings:
            print(f"Epoch {epoch+1} 警告: {', '.join(warnings)}")

# ================================
# 日誌和可視化函數
# ================================

def log_to_tensorboard(writer, train_metrics, val_metrics, lr, epoch):
    """記錄到TensorBoard"""
    
    # 訓練指標
    for key, value in train_metrics.items():
        writer.add_scalar(f'Train/{key}', value, epoch)
    
    # 驗證指標
    for key, value in val_metrics.items():
        writer.add_scalar(f'Val/{key}', value, epoch)
    
    # 學習率
    writer.add_scalar('Learning_Rate', lr, epoch)
    
    # 比較指標
    if 'total_loss' in train_metrics and 'total_loss' in val_metrics:
        writer.add_scalars('Loss_Comparison', {
            'Train': train_metrics['total_loss'],
            'Val': val_metrics['total_loss']
        }, epoch)

def print_epoch_summary(epoch, total_epochs, train_metrics, val_metrics, lr):
    """打印epoch總結"""
    print(f"\nEpoch {epoch+1}/{total_epochs} 總結:")
    print(f"學習率: {lr:.2e}")
    
    # 訓練指標
    print(f"訓練 - 總損失: {train_metrics.get('total_loss', 0):.4f}, "
          f"DCD: {train_metrics.get('dcd_loss', 0):.4f}, "
          f"梯度範數: {train_metrics.get('grad_norm', 0):.2f}")
    
    # 驗證指標
    print(f"驗證 - 總損失: {val_metrics.get('total_loss', 0):.4f}, "
          f"DCD: {val_metrics.get('dcd_loss', 0):.4f}")
    
    # CD和MSE作為參考
    print(f"參考 - 訓練CD: {train_metrics.get('cd_loss', 0):.4f}, "
          f"驗證CD: {val_metrics.get('cd_loss', 0):.4f}")

def save_checkpoint(model, optimizer, scheduler, metrics_history, 
                   save_path, epoch, is_best=False):
    """保存檢查點"""
    checkpoint = {
        # 'epoch': epoch,
        'model_state_dict': model.state_dict(),
        # 'optimizer_state_dict': optimizer.state_dict(),
        # 'scheduler_state_dict': scheduler.__dict__ if hasattr(scheduler, '__dict__') else None,
        # 'metrics_history': metrics_history,
        # 'is_best': is_best
    }
    
    torch.save(checkpoint, save_path)

def final_save_checkpoint(model, optimizer, scheduler, metrics_history, save_path):
    """最終保存"""
    save_checkpoint(model, optimizer, scheduler, metrics_history, save_path, 
                   metrics_history['best_metrics']['epoch'], is_best=False)

def diagnose_training(train_metrics, val_metrics, epoch, metrics_history):
    """訓練診斷"""
    if epoch < 5:
        return
    
    # 檢查損失趨勢
    recent_train_losses = metrics_history['train']['total_loss'][-5:]
    if len(recent_train_losses) >= 5:
        if all(recent_train_losses[i] >= recent_train_losses[i-1] for i in range(1, 5)):
            print("警告: 訓練損失連續上升，可能需要調整學習率")

def adjust_training_strategy(optimizer, scheduler, metrics_history, epoch):
    """動態調整訓練策略"""
    # 如果損失停滯，可以考慮調整學習率
    if epoch > 20:
        recent_losses = metrics_history['train']['total_loss'][-10:]
        if len(recent_losses) >= 10:
            loss_variance = np.var(recent_losses)
            if loss_variance < 1e-6:  # 損失變化很小
                current_lr = optimizer.param_groups[0]['lr']
                new_lr = current_lr * 0.8
                for param_group in optimizer.param_groups:
                    param_group['lr'] = new_lr
                print(f"學習率調整: {current_lr:.2e} → {new_lr:.2e}")

def generate_training_report(metrics_history, log_dir, num_epochs):
    """生成訓練報告"""
    report = {
        'training_summary': {
            'total_epochs': num_epochs,
            'best_epoch': metrics_history['best_metrics']['epoch'],
            'best_loss': metrics_history['best_metrics']['loss'],
        },
        'final_metrics': {
            'train': {k: v[-1] if v else 0 for k, v in metrics_history['train'].items()},
            'val': {k: v[-1] if v else 0 for k, v in metrics_history['val'].items()}
        },
        'training_stability': {
            'loss_variance': np.var(metrics_history['train']['total_loss'][-20:]) if len(metrics_history['train']['total_loss']) >= 20 else 0,
            'avg_grad_norm': np.mean(metrics_history['grad_norm'][-20:]) if len(metrics_history['grad_norm']) >= 20 else 0
        }
    }
    
    with open(f'{log_dir}/training_report.json', 'w') as f:
        json.dump(report, f, indent=2)

def plot_training_curves(metrics_history, log_dir):
    """繪製訓練曲線"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 損失曲線
    axes[0,0].plot(metrics_history['train']['total_loss'], label='Train', color='blue')
    axes[0,0].plot(metrics_history['val']['total_loss'], label='Val', color='red')
    axes[0,0].set_title('Total Loss')
    axes[0,0].legend()
    axes[0,0].grid(True)
    
    # DCD損失
    axes[0,1].plot(metrics_history['train']['dcd_loss'], label='Train DCD', color='green')
    axes[0,1].plot(metrics_history['val']['dcd_loss'], label='Val DCD', color='orange')
    axes[0,1].set_title('DCD Loss')
    axes[0,1].legend()
    axes[0,1].grid(True)
    
    # CD損失對比
    axes[0,2].plot(metrics_history['train']['cd_loss'], label='Train CD', color='purple')
    axes[0,2].plot(metrics_history['val']['cd_loss'], label='Val CD', color='brown')
    axes[0,2].set_title('Chamfer Distance')
    axes[0,2].legend()
    axes[0,2].grid(True)
    
    # 梯度範數
    axes[1,0].plot(metrics_history['grad_norm'], color='red', alpha=0.7)
    axes[1,0].set_title('Gradient Norm')
    axes[1,0].grid(True)
    
    # 學習率
    axes[1,1].plot(metrics_history['lr'], color='black')
    axes[1,1].set_title('Learning Rate')
    axes[1,1].grid(True)
    
    # MSE損失
    if 'mse_loss' in metrics_history['train']:
        axes[1,2].plot(metrics_history['train']['mse_loss'], label='Train MSE', color='cyan')
        axes[1,2].plot(metrics_history['val']['mse_loss'], label='Val MSE', color='magenta')
        axes[1,2].set_title('MSE Loss')
        axes[1,2].legend()
        axes[1,2].grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{log_dir}/training_curves.png', dpi=300, bbox_inches='tight')
    plt.close()

def extract_metrics_for_return(metrics_history):
    """提取指標用於返回"""
    return (
        metrics_history['train']['cd_loss'],      # train_cd_ae
        metrics_history['train']['mse_loss'],     # train_mse_ae  
        metrics_history['train']['dcd_loss'],     # train_dcd_ae
        metrics_history['val']['cd_loss'],        # val_cd_ae
        metrics_history['val']['mse_loss'],       # val_mse_ae
        metrics_history['val']['dcd_loss']        # val_dcd_ae
    )

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Training AE

In [32]:
torch.cuda.empty_cache()
# train_model(EnhancedConditionalDiffusionModel, dataloader)

In [33]:
if __name__ == "__main__":
    """使用範例"""
    
    # os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

    # 初始化增強版AutoEncoder
    autoencoder = PointCloudAutoencoder(
        num_points=5000, 
        latent_dim=128,  # 修正後的維度
        feature_dim=128
    )
    
    # 設定路徑
    base_dir = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107")
    models_dir = os.path.join(base_dir, "models")
    os.makedirs(models_dir, exist_ok=True)  # 確保models目錄存在
    save_path = os.path.join(models_dir, "autoencoder_pretrained_v8_0912_2137.pth")
    log_dir = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107\logs\autoencoder_v8_0912_2137")
    
    # 執行訓練
    train_results = train_enhanced_autoencoder(
        autoencoder=autoencoder,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        num_epochs=60,
        lr=2e-4,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        save_path_ae=save_path,
        log_dir=log_dir
    )
    
    print("AutoEncoder預訓練完成！")

開始Enhanced AutoEncoder預訓練...
訓練數據批次: 3, 驗證數據批次: 1
設備: cuda, 學習率: 0.0002, 訓練輪數: 60
----------------------------------------


驗證 Epoch 1: 100%|██████████| 1/1 [00:11<00:00, 11.48s/it, Val_Loss=0.2739, Val_DCD=0.3483]



Epoch 1/60 總結:
學習率: 1.33e-05
訓練 - 總損失: 0.2275, DCD: 0.2896, 梯度範數: 2.50
驗證 - 總損失: 0.2739, DCD: 0.3483
參考 - 訓練CD: 0.2896, 驗證CD: 0.3483
新的最佳模型已保存: C:/SHAWN/MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty/LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137_best.pth


驗證 Epoch 2: 100%|██████████| 1/1 [00:13<00:00, 13.29s/it, Val_Loss=0.2332, Val_DCD=0.2969]



Epoch 2/60 總結:
學習率: 2.00e-05
訓練 - 總損失: 0.1559, DCD: 0.1988, 梯度範數: 0.68
驗證 - 總損失: 0.2332, DCD: 0.2969
參考 - 訓練CD: 0.1988, 驗證CD: 0.2969
新的最佳模型已保存: C:/SHAWN/MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty/LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137_best.pth


...


Epoch 58/60 總結:
學習率: 1.09e-06
訓練 - 總損失: 0.0370, DCD: 0.0474, 梯度範數: 0.08
驗證 - 總損失: 0.0315, DCD: 0.0404
參考 - 訓練CD: 0.0474, 驗證CD: 0.0404
新的最佳模型已保存: C:/SHAWN/MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty/LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137_best.pth
警告: 訓練損失連續上升，可能需要調整學習率


驗證 Epoch 59: 100%|██████████| 1/1 [00:12<00:00, 12.02s/it, Val_Loss=0.0314, Val_DCD=0.0402]



Epoch 59/60 總結:
學習率: 1.02e-06
訓練 - 總損失: 0.0352, DCD: 0.0451, 梯度範數: 0.07
驗證 - 總損失: 0.0314, DCD: 0.0402
參考 - 訓練CD: 0.0451, 驗證CD: 0.0402
新的最佳模型已保存: C:/SHAWN/MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty/LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137_best.pth


驗證 Epoch 60: 100%|██████████| 1/1 [00:12<00:00, 12.84s/it, Val_Loss=0.0332, Val_DCD=0.0425]



Epoch 60/60 總結:
學習率: 1.00e-06
訓練 - 總損失: 0.0341, DCD: 0.0436, 梯度範數: 0.08
驗證 - 總損失: 0.0332, DCD: 0.0425
參考 - 訓練CD: 0.0436, 驗證CD: 0.0425

AutoEncoder預訓練完成!
最佳模型 (Epoch 58): 損失 = 0.031364
模型已保存至: C:/SHAWN/MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty/LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137.pth
日誌已保存至: C:/SHAWN/MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty/LDM_training_v8_0904_1107/logs/autoencoder_v8_0912_2137
AutoEncoder預訓練完成！


## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Define Training Diffusion Model Progress

In [14]:
class DiffusionGradientClipper:
    """針對擴散模型優化的梯度裁剪器"""
    def __init__(self, model, max_norm=1.0, percentile=95):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        
    def clip_gradients(self, loss):
        """擴散模型專用梯度裁剪"""
        loss.backward()
        
        # 計算當前梯度範數
        total_norm = 0
        param_count = 0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        total_norm = total_norm ** (1. / 2)
        
        # 更新歷史記錄
        self.grad_history.append(total_norm)
        if len(self.grad_history) > 100:
            self.grad_history.pop(0)
        
        # 擴散模型的自適應閾值調整
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            # 擴散模型訓練通常需要更嚴格的梯度控制
            clip_norm = min(self.max_norm, adaptive_threshold * 1.2)
        else:
            clip_norm = self.max_norm
        
        # 執行梯度裁剪
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class StabilizedDiffusionGradientClipper:
    """更穩定的擴散模型梯度裁剪器"""
    def __init__(self, model, max_norm=0.8, percentile=90):
        self.model = model
        self.max_norm = max_norm
        self.percentile = percentile
        self.grad_history = []
        self.stability_counter = 0
        
    def clip_gradients_enhanced(self):
        total_norm = 0
        param_count = 0
        
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
                param_count += 1
        
        if param_count == 0:
            return 0.0, self.max_norm
            
        total_norm = total_norm ** 0.5
        self.grad_history.append(total_norm)
        
        if len(self.grad_history) > 100:
            self.grad_history.pop(0)
        
        # 擴散模型的嚴格自適應閾值
        if len(self.grad_history) >= 10:
            adaptive_threshold = np.percentile(self.grad_history, self.percentile)
            clip_norm = min(self.max_norm, adaptive_threshold * 0.9)  # 更保守
        else:
            clip_norm = self.max_norm
        
        # 檢測訓練不穩定性
        if total_norm > clip_norm * 2.0:  # 擴散模型對不穩定更敏感
            self.stability_counter += 1
        else:
            self.stability_counter = max(0, self.stability_counter - 1)
        
        # 執行裁剪
        if total_norm > clip_norm:
            clip_coeff = clip_norm / (total_norm + 1e-6)
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coeff)
        
        return total_norm, clip_norm

class EnhancedDiffusionTrainingLoop:
    """擴散模型訓練循環"""
    def __init__(self, diffusion_model, optimizer, scheduler=None, num_timesteps=1000):
        self.diffusion_model = diffusion_model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.grad_clipper = StabilizedDiffusionGradientClipper(diffusion_model, max_norm=1.0)
        self.loss_history = []
        self.num_timesteps = num_timesteps
        
    def training_step(self, break_points, fix_points, norm_params):
        """擴散模型訓練步驟 - 核心改動：預測噪聲而非重建"""
        self.optimizer.zero_grad()
        
        # 計算擴散損失
        loss_dict = self.compute_diffusion_loss(break_points, fix_points, norm_params)
        total_loss = loss_dict['total_loss']
        
        # 檢測異常損失
        if torch.isnan(total_loss) or torch.isinf(total_loss):
            print(f"Warning: Invalid diffusion loss detected: {total_loss}")
            return None
        
        # 梯度裁剪
        grad_norm, clip_norm = self.grad_clipper.clip_gradients(total_loss)
        
        # 檢測梯度爆炸 - 擴散模型更敏感
        if grad_norm > 3.0:  # 降低閾值
            print(f"Warning: Large gradient norm in diffusion training: {grad_norm:.4f}")
            self.optimizer.zero_grad()
            # 動態學習率調整
            for param_group in self.optimizer.param_groups:
                param_group['lr'] *= 0.9
            return None
        
        # 優化步驟
        self.optimizer.step()
        if self.scheduler:
            self.scheduler.step()
        
        # 記錄損失歷史
        self.loss_history.append(total_loss.item())
        if len(self.loss_history) > 1000:
            self.loss_history.pop(0)
        
        return {
            'total_loss': total_loss.item(),
            'noise_loss': loss_dict['noise_loss'].item(),
            'grad_norm': grad_norm,
            'clip_norm': clip_norm,
            'lr': self.optimizer.param_groups[0]['lr'],
            **{k: v.item() if torch.is_tensor(v) else v 
               for k, v in loss_dict.items() if k != 'total_loss'}
        }
    
    def compute_diffusion_loss(self, break_points, fix_points, norm_params):
        """計算擴散模型專用損失函數"""
        B = fix_points.shape[0]
        device = fix_points.device
        
        # 使用預訓練的AE編碼器獲得潛在表示
        with torch.no_grad():
            z_0 = self.diffusion_model.autoencoder.encode(fix_points)
        
        # 時間步采樣 - 擴散模型核心
        t = self.progressive_timestep_sampling(B, device)
        
        # 添加噪聲到潛在空間 - 擴散過程
        z_t, noise= self.diffusion_model.add_noise(z_0, t)
        
        # 模型預測噪聲 - 擴散模型的核心任務
        predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
        
        # 計算各種擴散損失
        losses = self.compute_diffusion_losses(predicted_noise, noise, z_t, z_0, t, fix_points)
        
        # 動態權重平衡
        total_loss = self.balance_diffusion_losses(losses, t)
        
        return {
            'total_loss': total_loss,
            'noise_loss': losses['noise_loss'],
            **losses
        }
    
    def progressive_timestep_sampling(self, batch_size, device):
        """漸進式時間步采樣策略"""
        progress = min(len(self.loss_history) / 5000, 1.0)
        
        if progress < 0.2:
            # 早期：重點訓練大時間步（高噪聲）
            min_t = int(self.num_timesteps * 0.7)
            max_t = self.num_timesteps
        elif progress < 0.6:
            # 中期：均勻采樣全範圍
            min_t = 0
            max_t = self.num_timesteps
        else:
            # 後期：更多小時間步（低噪聲，更困難）
            # 使用加權采樣，偏向小時間步
            weights = torch.exp(-torch.arange(self.num_timesteps, dtype=torch.float) / 200.0)
            t = torch.multinomial(weights, batch_size, replacement=True)
            return t.to(device)
        
        return torch.randint(min_t, max_t, (batch_size,), device=device)
    
    def compute_diffusion_losses(self, predicted_noise, true_noise, z_t, z_0, t, x_0):
        """計算擴散模型的各種損失"""
        losses = {}
        
        # 主要損失：噪聲預測MSE - 擴散模型核心
        losses['noise_loss'] = F.mse_loss(predicted_noise, true_noise)
        
        # 時間加權噪聲損失 - 考慮不同時間步的重要性
        time_weights = self.compute_time_weights(t)
        weighted_noise_loss = F.mse_loss(predicted_noise, true_noise, reduction='none')
        losses['weighted_noise_loss'] = (weighted_noise_loss * time_weights.view(-1, 1, 1)).mean()
        
        # Huber損失版本 - 更穩健的噪聲預測
        losses['huber_noise_loss'] = F.smooth_l1_loss(predicted_noise, true_noise)
        
        # 可選：x0預測損失（如果模型也預測原始數據）
        if hasattr(self.diffusion_model, 'predict_x0') and self.diffusion_model.predict_x0:
            predicted_x0 = self.diffusion_model.predict_original_from_noise(z_t, predicted_noise, t)
            losses['x0_loss'] = F.mse_loss(predicted_x0, z_0)
        
        # 可選：v-parameterization損失（先進的預測目標）
        if hasattr(self.diffusion_model, 'v_parameterization'):
            v_target = self.compute_v_target(z_0, true_noise, t)
            predicted_v = self.diffusion_model.predict_v(z_t, t)
            losses['v_loss'] = F.mse_loss(predicted_v, v_target)
        
        # 潛在空間正則化
        losses['latent_reg'] = torch.mean(z_t ** 2) * 1e-4
        
        return losses
    
    def compute_time_weights(self, t):
        """計算時間步權重 - 平衡不同噪聲水平的學習"""
        # SNR-based weighting: 低SNR(高噪聲)時間步權重更高
        alpha_t = self.diffusion_model.scheduler.alphas_cumprod[t]
        snr = alpha_t / (1 - alpha_t)
        
        # 反比於SNR，高噪聲時間步獲得更高權重
        weights = 1.0 / (snr + 1e-8)
        weights = weights / weights.max()  # 歸一化
        return weights
    
    def compute_v_target(self, x0, noise, t):
        """計算v-parameterization的目標"""
        alpha_t = self.diffusion_model.scheduler.alphas_cumprod[t].view(-1, 1, 1)
        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_one_minus_alpha_t = torch.sqrt(1 - alpha_t)
        
        v = sqrt_alpha_t * noise - sqrt_one_minus_alpha_t * x0
        return v
    
    def balance_diffusion_losses(self, losses, t):
        """動態權重平衡擴散損失"""
        progress = min(len(self.loss_history) / 3000, 1.0)
        
        # 擴散模型損失權重策略
        weights = {
            'noise_loss': 0.7,           # 主要噪聲預測損失
            'weighted_noise_loss': 0.2,  # 時間加權噪聲損失
            'huber_noise_loss': 0.05,    # 穩健噪聲損失
            'latent_reg': 0.02,          # 潛在空間正則化
        }
        
        # 如果有額外損失，調整權重
        if 'x0_loss' in losses:
            weights['x0_loss'] = 0.1 * progress  # 後期增加x0監督
            weights['noise_loss'] = 0.6  # 相應減少主要損失權重
        
        if 'v_loss' in losses:
            weights['v_loss'] = 0.15 * progress  # v-parameterization
            weights['noise_loss'] = 0.55
        
        total_loss = sum(weights.get(key, 0) * loss 
                        for key, loss in losses.items() 
                        if torch.is_tensor(loss))
        
        return total_loss

class CosineWarmupScheduler:
    """餘弦退火調度器（適用於擴散模型）"""
    def __init__(self, optimizer, warmup_steps=1000, max_steps=10000, base_lr=1e-4, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_step = 0
        
    def step(self):
        """執行學習率調度"""
        self.current_step += 1
        
        if self.current_step <= self.warmup_steps:
            # Warmup階段：線性增長
            lr = self.base_lr * self.current_step / self.warmup_steps
        else:
            # 餘弦退火階段
            progress = (self.current_step - self.warmup_steps) / (self.max_steps - self.warmup_steps)
            progress = min(progress, 1.0)
            lr = self.min_lr + (self.base_lr - self.min_lr) * \
                 (1 + np.cos(np.pi * progress)) / 2
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
        return lr

class DiffusionEarlyStopping:
    """針對擴散模型的早停機制"""
    def __init__(self, patience=15, min_delta=0.001, stability_threshold=10):
        self.patience = patience
        self.min_delta = min_delta
        self.stability_threshold = stability_threshold
        self.counter = 0
        self.best_loss = float('inf')
        self.unstable_epochs = 0
        
    def __call__(self, val_loss, grad_norm):
        # 檢查損失改善
        improved = val_loss < self.best_loss - self.min_delta
        
        if improved:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        
        # 檢查訓練穩定性（擴散模型對梯度更敏感）
        if grad_norm > 1.5:  # 更嚴格的梯度範數閾值
            self.unstable_epochs += 1
        else:
            self.unstable_epochs = max(0, self.unstable_epochs - 1)
        
        # 決定是否停止
        patience_exceeded = self.counter >= self.patience
        too_unstable = self.unstable_epochs >= self.stability_threshold
        
        return patience_exceeded or too_unstable

def compute_diffusion_losses_enhanced(predicted_noise, true_noise, z_t, z_0, t, 
                                    x_0, diffusion_model, timestep_weights=None):
    """增強的擴散損失計算函數"""
    losses = {}
    
    try:
        # === 核心擴散損失 ===
        # 1. 主要噪聲預測損失（MSE）
        losses['noise_mse'] = F.mse_loss(predicted_noise, true_noise)
        
        # 2. 穩健噪聲預測損失（Huber）
        losses['noise_huber'] = F.smooth_l1_loss(predicted_noise, true_noise)
        
        # 3. 時間加權噪聲損失
        if timestep_weights is not None:
            weighted_loss = F.mse_loss(predicted_noise, true_noise, reduction='none')
            losses['weighted_noise'] = (weighted_loss * timestep_weights.view(-1, 1, 1)).mean()
        
        # === 可選的高級損失 ===
        # 4. L1噪聲損失（提供額外的穩定性）
        losses['noise_l1'] = F.l1_loss(predicted_noise, true_noise)
        
        # 5. 潛在空間一致性損失
        losses['latent_consistency'] = torch.mean(torch.abs(z_t - z_0))
        
        # 6. 梯度範數懲罰（防止梯度爆炸）
        if predicted_noise.requires_grad:
            grad_norm = torch.norm(torch.autograd.grad(
                predicted_noise.sum(), predicted_noise, 
                create_graph=True, retain_graph=True)[0])
            losses['grad_penalty'] = grad_norm * 1e-4

        # 7.潛在空間 L1 正則化
        losses['latent_reg'] = torch.mean(torch.abs(z_t)) * 1e-4

        
    except Exception as e:
        print(f"擴散損失計算錯誤: {e}")
        device = predicted_noise.device
        losses = {
            'noise_mse': F.mse_loss(predicted_noise, true_noise),
            'noise_huber': F.smooth_l1_loss(predicted_noise, true_noise),
            'latent_consistency': torch.tensor(0.01, device=device, requires_grad=True),
            'latent_reg': torch.tensor(0.1, device=device, requires_grad=True),
        }
    
    return losses

def compute_weighted_diffusion_loss(losses, current_step, total_steps):
    """動態權重擴散損失計算"""
    progress = min(current_step / total_steps, 1.0)
    
    # 擴散模型損失權重配置
    weights = {
        # 主要損失 (78%)
        'noise_mse': 0.78,             # 核心噪聲預測

        # 輔助損失 (22%)  
        'weighted_noise': 0.10,        # 時間步加權噪聲損失
        'latent_consistency': 0.05,    # 潛在空間一致性
        'noise_huber': 0.04,           # 穩健性(少量輔助)
        'grad_penalty': 0.02,          # 梯度懲罰
        'latent_reg': 0.01,             # L1正則化
    }
    
    # 根據訓練進度調整權重
    if progress > 0.5:
        # 後期增加穩健性損失的權重
        weights['noise_huber'] *= 1.2
        weights['weighted_noise'] *= 1.3
    
    total_loss = sum(weights.get(key, 0) * loss 
                    for key, loss in losses.items() 
                    if key in weights and torch.is_tensor(loss))
    
    return total_loss

In [15]:
class SimpleGradientClipper:
    """簡化版梯度裁剪器，避免複雜的記憶體操作"""
    def __init__(self, model, max_norm=1.0):
        self.model = model
        self.max_norm = max_norm
    
    def clip_gradients_simple(self):
        """簡化版梯度裁剪"""
        try:
            # 使用 PyTorch 內建的梯度裁剪，更安全
            total_norm = torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), self.max_norm
            )
            return total_norm, total_norm
        except Exception as e:
            print(f"梯度裁剪錯誤: {e}")
            return 0.0, 0.0

輔助損失函數

In [16]:
def compute_distribution_regularization(z):
    """計算分佈正則化損失"""
    try:
        # 鼓勵潛在向量接近正態分佈
        z_flat = z.reshape(z.shape[0], -1)
        
        # KL散度正則化
        mean = z_flat.mean(dim=1, keepdim=True)
        std = z_flat.std(dim=1, keepdim=True) + 1e-6
        
        # 目標是標準正態分佈
        kl_loss = 0.5 * (mean.pow(2) + std.pow(2) - 2 * torch.log(std) - 1).mean()
        
        return kl_loss
    except:
        return torch.tensor(0.0, device=z.device, requires_grad=True)

def compute_alignment_consistency_loss(z_original, z_aligned, x_original, autoencoder):
    """計算對齊一致性損失"""
    try:
        with torch.no_grad():
            recon_original = autoencoder.decode(z_original)
        
        recon_aligned = autoencoder.decode(z_aligned)
        
        # 重建應該保持一致
        consistency_loss = F.mse_loss(recon_aligned, recon_original)
        
        return consistency_loss
    except:
        return torch.tensor(0.0, device=z_aligned.device, requires_grad=True)

In [17]:
def train_enhanced_diffusion_model(diffusion_model, train_dataloader, val_dataloader,
                                 num_epochs=100, lr=1e-4, device='cuda',
                                 save_path_diffusion="diffusion_model_v1.pth",
                                 log_dir="./logs/diffusion_training"):
    """
    增強版擴散模型訓練函數
    核心改動：專注於噪聲預測而非數據重建
    """
    
    # ================================
    # 初始化設置
    # ================================
    diffusion_model.to(device)
    
    # 創建日誌目錄
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(os.path.dirname(save_path_diffusion), exist_ok=True)
    
    # TensorBoard設置
    writer = SummaryWriter(log_dir=log_dir, flush_secs=30)
    
    # 擴散模型專用優化器設置
    optimizer = torch.optim.AdamW(
        diffusion_model.parameters(), 
        lr=lr, 
        betas=(0.9, 0.999),
        weight_decay=1e-4,
        eps=1e-8
    )
    
    # 擴散模型專用學習率調度
    total_steps = len(train_dataloader) * num_epochs
    scheduler = CosineWarmupScheduler(
        optimizer, 
        warmup_steps=total_steps // 10,  # 10% warmup
        max_steps=total_steps,
        base_lr=lr,
        min_lr=lr * 0.01
    )
    
    # 早停機制
    early_stopping = DiffusionEarlyStopping(patience=50, min_delta=0.0005)
    
    # 梯度裁剪器
    grad_clipper = StabilizedDiffusionGradientClipper(diffusion_model, max_norm=1.5)
    
    # 混合精度（可選）
    # scaler = GradScaler() if device == 'cuda' else None
    scaler = None
    
    # 訓練監控
    metrics_history = {
        'train': defaultdict(list),
        'val': defaultdict(list),
        'lr': [],
        'grad_norm': [],
        'best_metrics': {'epoch': 0, 'loss': float('inf')}
    }
    
    # ================================
    # 主訓練循環
    # ================================
    print("開始擴散模型訓練...")
    print(f"訓練數據批次: {len(train_dataloader)}, 驗證數據批次: {len(val_dataloader)}")
    print(f"設備: {device}, 學習率: {lr}, 訓練輪數: {num_epochs}")
    print(f"擴散時間步數: {diffusion_model.num_timesteps}")
    print("-" * 50)
    
    global_step = 0
    
    for epoch in range(num_epochs):
        
        # ============================
        # 訓練階段
        # ============================
        diffusion_model.train()
        train_metrics = train_diffusion_epoch(
            diffusion_model, train_dataloader, optimizer, scheduler,
            grad_clipper, device, epoch, num_epochs, scaler, global_step
        )
        
        global_step += len(train_dataloader)
        
        # ============================
        # 驗證階段
        # ============================
        diffusion_model.eval()
        val_metrics = validate_diffusion_epoch(
            diffusion_model, val_dataloader, device, epoch
        )
        
        # ============================
        # 記錄和監控
        # ============================
        current_lr = scheduler.optimizer.param_groups[0]['lr']
        
        # 保存指標
        for key, value in train_metrics.items():
            metrics_history['train'][key].append(value)
        for key, value in val_metrics.items():
            metrics_history['val'][key].append(value)
        
        metrics_history['lr'].append(current_lr)
        if 'grad_norm' in train_metrics:
            metrics_history['grad_norm'].append(train_metrics['grad_norm'])
        
        # TensorBoard日誌
        log_diffusion_metrics(writer, train_metrics, val_metrics, current_lr, epoch)
        
        # 控制台輸出
        print_diffusion_summary(epoch, num_epochs, train_metrics, val_metrics, current_lr)
        
        # ============================
        # 模型保存
        # ============================
        current_loss = val_metrics.get('total_loss', train_metrics.get('total_loss', float('inf')))
        if current_loss < metrics_history['best_metrics']['loss']:
            metrics_history['best_metrics']['loss'] = current_loss
            metrics_history['best_metrics']['epoch'] = epoch
            
            # 保存最佳模型
            best_model_path = save_path_diffusion.replace('.pth', '_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': diffusion_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.__dict__,
                'metrics_history': metrics_history,
                'loss': current_loss
            }, best_model_path)
            print(f"新的最佳擴散模型已保存: {best_model_path}")
        
        # 定期保存檢查點
        if (epoch + 1) % 100 == 0 or epoch == num_epochs - 1:
            checkpoint_path = save_path_diffusion.replace('.pth', f'_epoch_{epoch+1}.pth')
            torch.save(diffusion_model.state_dict(), checkpoint_path)
        
        # 早停檢查
        if early_stopping(val_metrics.get('total_loss', float('inf')), 
                         train_metrics.get('grad_norm', 0)):
            print(f"擴散模型訓練早停於 Epoch {epoch+1}")
            break
        
        # 顯存清理
        if device == "cuda":
            torch.cuda.empty_cache()
    
    # ================================
    # 訓練完成處理
    # ================================
    print("\n" + "="*80)
    print("擴散模型訓練完成!")
    
    # 保存最終模型
    torch.save({
        'model_state_dict': diffusion_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics_history': metrics_history,
    }, save_path_diffusion)
    
    # 生成訓練報告
    generate_diffusion_training_report(metrics_history, log_dir, num_epochs)
    
    # 繪製訓練曲線
    plot_diffusion_training_curves(metrics_history, log_dir)
    
    # 關閉TensorBoard
    writer.close()
    
    print(f"最佳模型 (Epoch {metrics_history['best_metrics']['epoch']}): "
          f"損失 = {metrics_history['best_metrics']['loss']:.6f}")
    print(f"擴散模型已保存至: {save_path_diffusion}")
    print(f"訓練日誌已保存至: {log_dir}")
    
    return extract_diffusion_metrics_for_return(metrics_history)

# ================================
# 核心訓練函數實現
# ================================

def train_diffusion_epoch(diffusion_model, train_dataloader, optimizer, scheduler,
                         grad_clipper, device, epoch, total_epochs, scaler=None, global_step=0):
    """擴散模型訓練epoch"""
    
    epoch_metrics = defaultdict(list)
    
    progress_bar = tqdm(train_dataloader, desc=f"擴散訓練 Epoch {epoch+1}/{total_epochs}")
    
    step = 0
    for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
        
        if fix_points is None:
            continue
        
        # 處理批次數據
        try:
            # 處理 fix_points
            fix_points = process_batch_data(fix_points, device)
            if fix_points is None:
                continue
                
            # 處理 break_points  
            break_points = process_batch_data(break_points, device)
            if break_points is None:
                continue
            
            # 額外確保數據在正確設備（雙重保險）
            fix_points = fix_points.to(device)
            break_points = break_points.to(device)
            
        except Exception as e:
            print(f"數據設備轉移錯誤: {e}")
            continue
        
        # 前向傳播
        optimizer.zero_grad()
        
        if scaler is not None:
            with autocast():
                # 獲得潛在表示
                with torch.no_grad():
                    z_0 = diffusion_model.autoencoder.encode(fix_points)
                
                # 擴散過程
                B = z_0.shape[0]
                device = z_0.device
                
                # 隨機時間步
                t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
                
                # 添加噪聲
                z_t, noise = diffusion_model.add_noise(z_0, t)
                
                # 預測噪聲
                predicted_noise = diffusion_model.forward(z_t, t, break_points)
                
                # 計算損失
                losses = compute_diffusion_losses_enhanced(
                    predicted_noise, noise, z_t, z_0, t, fix_points, diffusion_model
                )
                total_loss = compute_weighted_diffusion_loss(losses, global_step + step, 
                                                           len(train_dataloader) * total_epochs)
            
            # 檢查損失有效性
            if not torch.isfinite(total_loss):
                print(f"警告: 檢測到無效擴散損失 (Epoch {epoch}, Batch {batch_idx})")
                continue
            
            # 混合精度反向傳播
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), max_norm=1.5)
            
            # 檢測梯度異常
            if grad_norm > 10.0:
                print(f"警告: 擴散模型梯度爆炸 (Epoch {epoch}, Batch {batch_idx}, GradNorm: {grad_norm:.2f})")
                optimizer.zero_grad()
                continue
            
            scaler.step(optimizer)
            scaler.update()
        
        else:
            # 標準訓練
            with torch.no_grad():
                z_0 = diffusion_model.autoencoder.encode(fix_points)
            
            B = z_0.shape[0]
            t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
            
            z_t, noise = diffusion_model.add_noise(z_0, t)
            
            predicted_noise = diffusion_model.forward(z_t, t, break_points)
            
            losses = compute_diffusion_losses_enhanced(
                predicted_noise, noise, z_t, z_0, t, fix_points, diffusion_model
            )
            total_loss = compute_weighted_diffusion_loss(losses, global_step + step,
                                                       len(train_dataloader) * total_epochs)
            
            if not torch.isfinite(total_loss):
                continue
            
            total_loss.backward()
            grad_norm, clip_norm = grad_clipper.clip_gradients_enhanced()
            
            if grad_norm > 10.0:
                optimizer.zero_grad()
                continue
            
            optimizer.step()
        
        # 學習率調度
        scheduler.step()
        
        # 記錄指標
        batch_metrics = {
            'total_loss': total_loss.item(),
            'noise_loss': losses.get('noise_mse', total_loss).item(),
            'grad_norm': grad_norm,
            **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
        }
        
        for key, value in batch_metrics.items():
            epoch_metrics[key].append(value)
        
        # 更新進度條
        if batch_idx % 10 == 0:
            progress_bar.set_postfix({
                'Loss': f"{total_loss.item():.4f}",
                'Noise': f"{losses.get('noise_mse', total_loss).item():.4f}",
                'GradNorm': f"{grad_norm:.2f}"
            })
        
        step += 1
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}

def validate_diffusion_epoch(diffusion_model, val_dataloader, device, epoch):
    """擴散模型驗證epoch"""
    
    epoch_metrics = defaultdict(list)
    
    with torch.no_grad():
        progress_bar = tqdm(val_dataloader, desc=f"擴散驗證 Epoch {epoch+1}")
        
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue
            
            try:
                fix_points = process_batch_data(fix_points, device)
                if fix_points is None:
                    continue
                    
                break_points = process_batch_data(break_points, device)
                if break_points is None:
                    continue
                
                fix_points = fix_points.to(device)
                break_points = break_points.to(device)
                
            except Exception as e:
                print(f"驗證數據設備轉移錯誤: {e}")
                continue
            
            # 擴散模型前向傳播
            z_0 = diffusion_model.autoencoder.encode(fix_points)
            
            B = z_0.shape[0]
            t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
            
            z_t, noise= diffusion_model.add_noise(z_0, t)
            
            predicted_noise = diffusion_model.forward(z_t, t, break_points)
            
            # 計算損失
            losses = compute_diffusion_losses_enhanced(
                predicted_noise, noise, z_t, z_0, t, fix_points, diffusion_model
            )
            total_loss = compute_weighted_diffusion_loss(losses, 0, 1)  # 驗證時權重固定
            
            # 記錄指標
            batch_metrics = {
                'total_loss': total_loss.item(),
                'noise_loss': losses.get('noise_mse', total_loss).item(),
                **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
            }
            
            for key, value in batch_metrics.items():
                epoch_metrics[key].append(value)
            
            # 更新進度條
            if batch_idx % 10 == 0:
                progress_bar.set_postfix({
                    'Val_Loss': f"{total_loss.item():.4f}",
                    'Val_Noise': f"{losses.get('noise_mse', total_loss).item():.4f}"
                })
    
    return {key: np.mean(values) for key, values in epoch_metrics.items()}

# ================================
# 輔助函數
# ================================

def log_diffusion_metrics(writer, train_metrics, val_metrics, lr, epoch):
    """記錄擴散模型訓練指標到TensorBoard"""
    
    # 訓練指標
    for key, value in train_metrics.items():
        writer.add_scalar(f'Diffusion_Train/{key}', value, epoch)
    
    # 驗證指標
    for key, value in val_metrics.items():
        writer.add_scalar(f'Diffusion_Val/{key}', value, epoch)
    
    # 學習率
    writer.add_scalar('Diffusion_Learning_Rate', lr, epoch)
    
    # 損失對比
    if 'total_loss' in train_metrics and 'total_loss' in val_metrics:
        writer.add_scalars('Diffusion_Loss_Comparison', {
            'Train': train_metrics['total_loss'],
            'Val': val_metrics['total_loss']
        }, epoch)

def print_diffusion_summary(epoch, total_epochs, train_metrics, val_metrics, lr):
    """打印擴散模型訓練總結"""
    print(f"\n擴散模型 Epoch {epoch+1}/{total_epochs} 總結:")
    print(f"學習率: {lr:.2e}")
    
    # 訓練指標
    print(f"訓練 - 總損失: {train_metrics.get('total_loss', 0):.4f}, "
          f"噪聲損失: {train_metrics.get('noise_loss', 0):.4f}, "
          f"梯度範數: {train_metrics.get('grad_norm', 0):.2f}")
    
    # 驗證指標
    print(f"驗證 - 總損失: {val_metrics.get('total_loss', 0):.4f}, "
          f"噪聲損失: {val_metrics.get('noise_loss', 0):.4f}")

def generate_diffusion_training_report(metrics_history, log_dir, num_epochs):
    """生成擴散模型訓練報告"""
    report = {
        'training_summary': {
            'model_type': 'Diffusion Model',
            'total_epochs': num_epochs,
            'best_epoch': metrics_history['best_metrics']['epoch'],
            'best_loss': metrics_history['best_metrics']['loss'],
        },
        'final_metrics': {
            'train': {k: v[-1] if v else 0 for k, v in metrics_history['train'].items()},
            'val': {k: v[-1] if v else 0 for k, v in metrics_history['val'].items()}
        },
        'training_stability': {
            'loss_variance': np.var(metrics_history['train']['total_loss'][-20:]) if len(metrics_history['train']['total_loss']) >= 20 else 0,
            'avg_grad_norm': np.mean(metrics_history['grad_norm'][-20:]) if len(metrics_history['grad_norm']) >= 20 else 0
        }
    }
    
    with open(f'{log_dir}/diffusion_training_report.json', 'w') as f:
        json.dump(report, f, indent=2)

def plot_diffusion_training_curves(metrics_history, log_dir):
    """繪製擴散模型訓練曲線"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 總損失
    axes[0,0].plot(metrics_history['train']['total_loss'], label='Train Total', color='blue')
    axes[0,0].plot(metrics_history['val']['total_loss'], label='Val Total', color='red')
    axes[0,0].set_title('Total Loss (Diffusion)')
    axes[0,0].legend()
    axes[0,0].grid(True)
    
    # 噪聲損失
    if 'noise_loss' in metrics_history['train']:
        axes[0,1].plot(metrics_history['train']['noise_loss'], label='Train Noise', color='green')
        axes[0,1].plot(metrics_history['val']['noise_loss'], label='Val Noise', color='orange')
        axes[0,1].set_title('Noise Prediction Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)
    
    # MSE噪聲損失
    if 'noise_mse' in metrics_history['train']:
        axes[0,2].plot(metrics_history['train']['noise_mse'], label='Train MSE', color='purple')
        axes[0,2].plot(metrics_history['val']['noise_mse'], label='Val MSE', color='brown')
        axes[0,2].set_title('MSE Noise Loss')
        axes[0,2].legend()
        axes[0,2].grid(True)
    
    # 梯度範數
    axes[1,0].plot(metrics_history['grad_norm'], color='red', alpha=0.7)
    axes[1,0].set_title('Gradient Norm')
    axes[1,0].grid(True)
    
    # 學習率
    axes[1,1].plot(metrics_history['lr'], color='black')
    axes[1,1].set_title('Learning Rate')
    axes[1,1].grid(True)
    
    # Huber損失
    if 'noise_huber' in metrics_history['train']:
        axes[1,2].plot(metrics_history['train']['noise_huber'], label='Train Huber', color='cyan')
        axes[1,2].plot(metrics_history['val']['noise_huber'], label='Val Huber', color='magenta')
        axes[1,2].set_title('Huber Noise Loss')
        axes[1,2].legend()
        axes[1,2].grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{log_dir}/diffusion_training_curves.png', dpi=300, bbox_inches='tight')
    plt.close()

def extract_diffusion_metrics_for_return(metrics_history):
    """提取擴散模型訓練指標用於返回"""
    return (
        metrics_history['train']['total_loss'],     # train_total_diffusion
        metrics_history['train'].get('noise_loss', []), # train_noise_diffusion
        metrics_history['val']['total_loss'],       # val_total_diffusion
        metrics_history['val'].get('noise_loss', []),   # val_noise_diffusion
    )

In [18]:
class IntegratedMultiStageTrainingManager:
    """整合版多階段訓練管理器 - 基於現有優化流程"""
    def __init__(self, diffusion_model, train_dataloader, val_dataloader, device, log_dir):
        self.diffusion_model = diffusion_model
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.device = device
        self.log_dir = log_dir
        
        # 創建階段特定的日誌目錄
        self.stage_log_dirs = {
            'stage1': os.path.join(log_dir, 'stage1_alignment'),
            'stage2': os.path.join(log_dir, 'stage2_joint'), 
            'stage3': os.path.join(log_dir, 'stage3_diffusion')
        }
        
        for log_dir_path in self.stage_log_dirs.values():
            os.makedirs(log_dir_path, exist_ok=True)
        
        # TensorBoard writers for each stage
        self.writers = {}
        
        # 優化器和調度器
        self.optimizers = {}
        self.schedulers = {}
        self.early_stoppers = {}
        self.grad_clippers = {}
        
        # 訓練歷史記錄
        self.stage_metrics = {
            'stage1': {'train': defaultdict(list), 'val': defaultdict(list), 'lr': [], 'grad_norm': []},
            'stage2': {'train': defaultdict(list), 'val': defaultdict(list), 'lr': [], 'grad_norm': []},
            'stage3': {'train': defaultdict(list), 'val': defaultdict(list), 'lr': [], 'grad_norm': []}
        }
        
    def setup_stage_components(self):
        """設置各階段的訓練組件"""
        
        clip_values = {'stage1': 1.0, 'stage2': 1.5, 'stage3': 0.8}

        # Stage 1: 對齊訓練設置
        stage1_params = list(self.diffusion_model.latent_aligner.parameters())
        self.optimizers['stage1'] = torch.optim.AdamW(
            stage1_params, lr=2e-5, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8
        )
        
        # Stage 2: 聯合訓練設置  
        stage2_params = (list(self.diffusion_model.latent_aligner.parameters()) + 
                        list(self.diffusion_model.unet.parameters()) +
                        list(self.diffusion_model.encoder.parameters()))
        self.optimizers['stage2'] = torch.optim.AdamW(
            stage2_params, lr=1.5e-5, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8
        )
        
        # Stage 3: 純擴散訓練設置
        stage3_params = (list(self.diffusion_model.unet.parameters()) +
                        list(self.diffusion_model.encoder.parameters()))
        self.optimizers['stage3'] = torch.optim.AdamW(
            stage3_params, lr=8e-6, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8
        )
        
        # 為每個階段設置學習率調度器
        for stage in ['stage1', 'stage2', 'stage3']:
            epochs = {'stage1': 20, 'stage2': 80, 'stage3': 100}[stage]
            total_steps = len(self.train_dataloader) * epochs
            
            # 不同階段使用不同的調度策略
            if stage == 'stage1':
                # Stage1 使用線性 warmup + cosine decay
                self.schedulers[stage] = CosineWarmupScheduler(
                    self.optimizers[stage],
                    warmup_steps=total_steps // 5,  # 更長的 warmup
                    max_steps=total_steps,
                    base_lr=self.optimizers[stage].param_groups[0]['lr'],
                    min_lr=self.optimizers[stage].param_groups[0]['lr'] * 0.1
                )
            elif stage == 'stage2':
                # Stage2 使用標準 warmup + cosine decay
                self.schedulers[stage] = CosineWarmupScheduler(
                    self.optimizers[stage],
                    warmup_steps=total_steps // 10,
                    max_steps=total_steps,
                    base_lr=self.optimizers[stage].param_groups[0]['lr'],
                    min_lr=self.optimizers[stage].param_groups[0]['lr'] * 0.01
                )
            else:  # stage3
                # Stage3 使用較短 warmup + 更平緩的 decay
                self.schedulers[stage] = CosineWarmupScheduler(
                    self.optimizers[stage],
                    warmup_steps=total_steps // 20,
                    max_steps=total_steps,
                    base_lr=self.optimizers[stage].param_groups[0]['lr'],
                    min_lr=self.optimizers[stage].param_groups[0]['lr'] * 0.05
                )
            
            # 不同階段的早停策略
            patience = {'stage1': 15, 'stage2': 30, 'stage3': 25}[stage]
            min_delta = {'stage1': 0.005, 'stage2': 0.0002, 'stage3': 0.0001}[stage]
            self.early_stoppers[stage] = DiffusionEarlyStopping(
                patience=patience, min_delta=min_delta
            )

            # 梯度裁剪器
            # 不同的梯度裁剪閾值
            self.grad_clippers[stage] = SimpleGradientClipper(
                self.diffusion_model, max_norm=clip_values[stage]
            )
            
            # TensorBoard writer
            self.writers[stage] = SummaryWriter(
                log_dir=self.stage_log_dirs[stage], flush_secs=30
            )
    

    def _save_stage_checkpoint(self, stage, epoch, loss, is_final=False):
        """保存階段檢查點"""
        try:
            # 創建檢查點保存路徑
            best_path = os.path.join(self.stage_log_dirs[stage], f'{stage}_best.pth')
            final_path = os.path.join(self.stage_log_dirs[stage], f'{stage}_best.pth')
            # 保存模型狀態
            checkpoint_data = {
                # 'epoch': epoch + 1,
                # 'stage': stage,
                'model_state_dict': self.diffusion_model.state_dict(),
                # 'optimizer_state_dict': self.optimizers[stage].state_dict(),
                # 'scheduler_state_dict': self.schedulers[stage].state_dict(),
                # 'loss': float(loss),
                # 'metrics': self.stage_metrics[stage]
            }
            # 根據狀態儲存
            if is_final:
                torch.save(checkpoint_data, final_path)
                print(f"{stage.upper()} 最後模型已保存: Epoch {epoch+1}, Loss: {loss:.4f}")
                print(f"檢查點路徑: {final_path}")
            else:
                torch.save(checkpoint_data, best_path)
                print(f"{stage.upper()} 最佳模型已保存: Epoch {epoch+1}, Loss: {loss:.4f}")
                print(f"檢查點路徑: {best_path}")
            
        except Exception as e:
            print(f"保存檢查點錯誤: {e}")


    def _freeze_params_for_stage1(self):
        """只讓latent_aligner可訓練，其餘全部凍結"""
        for param in self.diffusion_model.autoencoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.unet.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.encoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.latent_aligner.parameters():
            param.requires_grad = True

    def _freeze_params_for_stage2(self):
        """讓latent_aligner、unet、encoder可訓練，AE凍結"""
        for param in self.diffusion_model.autoencoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.latent_aligner.parameters():
            param.requires_grad = True
        for param in self.diffusion_model.unet.parameters():
            param.requires_grad = True
        for param in self.diffusion_model.encoder.parameters():
            param.requires_grad = True

    def _freeze_params_for_stage3(self):
        """只讓unet和encoder可訓練，其餘全部凍結"""
        for param in self.diffusion_model.autoencoder.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.latent_aligner.parameters():
            param.requires_grad = False
        for param in self.diffusion_model.unet.parameters():
            param.requires_grad = True
        for param in self.diffusion_model.encoder.parameters():
            param.requires_grad = True

    def dynamic_loss_weights(self, epoch, total_epochs):
        progress = epoch / total_epochs
        alignment_weight = 0.80 - progress * 0.15   # 0.75 -> 0.60
        structure_weight = 0.10 + progress + 0.10   # 0.10 -> 0.20
        dist_reg_weight = 0.08  + progress + 0.05   # 0.08 -> 0.13
        condition_weight = 0.07                     
        return alignment_weight, structure_weight, dist_reg_weight, condition_weight


    def run_integrated_multi_stage_training(self, stage1_epochs=20, stage2_epochs=80, stage3_epochs=100):
        """執行整合版多階段訓練"""
        
        print("=== 開始整合版多階段擴散模型訓練 ===")
        self.setup_stage_components()
        
        # 階段1：潛在空間對齊
        print(f"\n--- 階段1：潛在空間對齊 ({stage1_epochs} epochs) ---")
        self._train_integrated_stage1(stage1_epochs)
        
        # # 階段2：聯合訓練
        # print(f"\n--- 階段2：聯合訓練 ({stage2_epochs} epochs) ---")
        # self._train_integrated_stage2(stage2_epochs)
        
        # # 階段3：純擴散訓練
        # print(f"\n--- 階段3：純擴散訓練 ({stage3_epochs} epochs) ---")
        # self._train_integrated_stage3(stage3_epochs)
        
        # 關閉所有TensorBoard writers
        for writer in self.writers.values():
            writer.close()
        
        # 生成綜合訓練報告
        self._generate_multi_stage_report()
        
        print("\n=== 整合版多階段訓練完成 ===")
        return self.stage_metrics


    def check_gpu_health(self):
        """檢查GPU健康狀態"""
        if not torch.cuda.is_available():
            return False
        
        try:
            # 測試簡單的GPU操作
            test_tensor = torch.randn(10, 10, device=self.device)
            result = test_tensor @ test_tensor.T
            
            if torch.isnan(result).any():
                return False
            
            # 清理測試張量
            del test_tensor, result
            torch.cuda.empty_cache()
            
            return True
            
        except Exception as e:
            print(f"GPU健康檢查失敗: {e}")
            return False


    # 階段1整合訓練函數
    def _train_integrated_stage1(self, num_epochs):
        """階段1：整合版對齊訓練"""
        
        # 訓練前GPU健康檢查
        if not self.check_gpu_health():
            print("GPU狀態異常，無法開始訓練")
            return

        # 凍結參數設置
        self._freeze_params_for_stage1()
        
        best_loss = float('inf')
        global_step = 0
        
        for epoch in range(num_epochs):
            
            # 訓練階段
            self.diffusion_model.train()
            train_metrics = self._train_stage1_epoch(epoch, num_epochs, global_step)
            global_step += len(self.train_dataloader)
            
            # 驗證階段
            self.diffusion_model.eval()
            val_metrics = self._validate_stage1_epoch(epoch, total_epochs=20)
            
            # 記錄指標
            self._log_stage_metrics('stage1', epoch, train_metrics, val_metrics)
            
            # 模型保存
            current_loss = val_metrics.get('total_loss', float('inf'))
            if current_loss < best_loss:
                best_loss = current_loss
                self._save_stage_checkpoint('stage1', epoch, current_loss, is_final=False)
            # 最後一個epoch時再存一次
            if epoch == num_epochs - 1:
                self._save_stage_checkpoint('stage1', epoch, current_loss, is_final=True)
            
            # 早停檢查
            if self.early_stoppers['stage1'](current_loss, train_metrics.get('grad_norm', 0)):
                print(f"階段1早停於 Epoch {epoch+1}")
                break
            
            # 記憶體清理
            if self.device == "cuda":
                torch.cuda.empty_cache()

    def _train_stage1_epoch(self, epoch, total_epochs, global_step):
        """階段1單個epoch訓練"""
        
        epoch_metrics = defaultdict(list)
        progress_bar = tqdm(self.train_dataloader, desc=f"Stage3-1: Epoch {epoch+1}/{total_epochs}")
        
        consecutive_errors = 0
        critical_error_count = 0
        step = 0
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            
            if fix_points is None:
                continue
            
            # 如果連續錯誤太多，中斷訓練
            if consecutive_errors > 3:  
                print(f"Continous error: ({consecutive_errors})，interrupt epoch{epoch+1}")
                break
            if critical_error_count > 1:
                print(f"Critical error: ({critical_error_count})，interrupt epoch{epoch+1}")
                break            

            # 使用現有的數據處理邏輯
            try:
                # 數據處理，增加錯誤檢查
                fix_points = process_batch_data(fix_points, self.device)
                break_points = process_batch_data(break_points, self.device)
                
                if fix_points is None or break_points is None:
                    consecutive_errors += 1
                    continue
                
                # 檢查張量有效性
                if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                    print(f"Invalid fix_points，skipping batch{batch_idx}")
                    consecutive_errors += 1
                    continue
                
                if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                    print(f"Invalid break_points，skipping batch{batch_idx}")
                    consecutive_errors += 1
                    continue
                
                # 前向傳播
                self.optimizers['stage1'].zero_grad()
            
                # 獲得潛在表示
                with torch.no_grad():
                    try:
                        z_original = self.diffusion_model.autoencoder.encode(fix_points)
                        condition_embedding = self.diffusion_model.encoder(break_points)
                        if torch.isnan(z_original).any():
                            print(f"AE enocder generates NaN，skipping batch{batch_idx}")
                            consecutive_errors += 1
                            continue
                    except Exception as e:
                        print(f"AE encodes error: {e}")
                        consecutive_errors += 1
                        continue

                # 應用對齊，Stage 1 專注於基礎對齊，不需要條件對齊
                try:
                    # 應用條件對齊
                    z_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_original)
                    if torch.isnan(z_aligned).any():
                        print(f"z_aligned contain NaN，skipping batch{batch_idx}")
                        consecutive_errors += 1
                        continue
                except Exception as e:
                    print(f"Aligning Error: {e}")
                    consecutive_errors += 1
                    continue

                # 計算對齊損失
                try:
                    recon_aligned = self.diffusion_model.autoencoder.decode(z_aligned)
                    if torch.isnan(recon_aligned).any():
                        print(f"recon_aligned contain NaN，skipping batch{batch_idx}")
                        consecutive_errors += 1
                        continue

                    alignment_weight, structure_weight, dist_reg_weight, condition_weight = self.dynamic_loss_weights(epoch, total_epochs)

                    alignment_loss = F.mse_loss(recon_aligned, fix_points) * alignment_weight
                    structure_loss = F.mse_loss(z_aligned, z_original.detach()) * structure_weight
                    dist_reg_loss = compute_distribution_regularization(z_aligned) * dist_reg_weight
                    condition_consistency = F.mse_loss(
                        self.diffusion_model.forward_encoder_only(z_aligned),
                        condition_embedding.detach()
                    ) * condition_weight

                    total_loss = alignment_loss + structure_loss + dist_reg_loss + condition_consistency


                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"Ivalid loss, skip batch{batch_idx}")
                        consecutive_errors += 1
                        continue
                    
                except Exception as e:
                    print(f"Computing loss error: {e}")
                    consecutive_errors += 1
                    continue
            
                
                # 反向傳播
                try:
                    total_loss.backward()
                    
                    max_grad_norm = 1.0
                    grad_norm = torch.nn.utils.clip_grad_norm_(self.diffusion_model.parameters(), max_grad_norm)
                    
                    # # 確保 grad_norm 是單一數值
                    # if isinstance(grad_norm, tuple):
                    #     grad_norm = grad_norm[0] if len(grad_norm) > 0 else 0.0
                    # elif torch.is_tensor(grad_norm):
                    #     grad_norm = grad_norm.item()

                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"total_loss is NaN/Inf，drop batch{batch_idx}")
                        self.optimizers['stage1'].zero_grad()
                        continue

                    # 梯度值健檢
                    all_grad_ok = True
                    for p in self.diffusion_model.parameters():
                        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                            all_grad_ok = False
                            break

                    if not all_grad_ok:
                        print(f"Found NaN/Inf-gradient，Drop batch{batch_idx}")
                        self.optimizers['stage1'].zero_grad()
                        continue
                            
                    self.optimizers['stage1'].step()
                    self.schedulers['stage1'].step()
                    
                    # 更新統計
                    self.diffusion_model.latent_aligner.update_statistics(z_original.detach())
                    
                    # 記錄指標
                    # 修正 batch_metrics 記錄
                    batch_metrics = {
                        'alignment_loss': self.safe_tensor_to_float(alignment_loss),
                        'structure_loss': self.safe_tensor_to_float(structure_loss),
                        'dist_reg_loss': self.safe_tensor_to_float(dist_reg_loss),
                        'condition_consistency': self.safe_tensor_to_float(condition_consistency),
                        'total_loss': self.safe_tensor_to_float(total_loss),
                        'grad_norm': self.safe_tensor_to_float(grad_norm)
                    }
                    for key, value in batch_metrics.items():
                        if torch.is_tensor(value):  # 雙重檢查
                            value = float(value.detach().cpu().numpy())
                        epoch_metrics[key].append(value)
                    
                    # 成功執行，重置錯誤計數
                    consecutive_errors = 0
                    critical_error_count = 0

                    # 更新進度條
                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Align_Loss': f"{alignment_loss.item():.4f}",
                            'Struct_Loss': f"{structure_loss.item():.4f}",
                            'Reg_Loss': f"{dist_reg_loss.item():.4f}",
                            'Condition_Loss': f"{condition_consistency.item():.4f}",
                            'Total_Loss': f"{grad_norm:.2f}",
                            'GradNorm': f"{grad_norm:.2f}",
                        })

                except RuntimeError as e:
                    if "CUDA" in str(e):
                        print(f"CUDA error in batch{batch_idx}: {str(e)[:100]}")
                        consecutive_errors += 1
                        
                        # CUDA 錯誤恢復策略
                        self.optimizers['stage1'].zero_grad()
                        
                        # 嘗試重置 CUDA 狀態
                        if self.device == "cuda":
                            try:
                                torch.cuda.synchronize()
                                torch.cuda.empty_cache()
                            except:
                                pass
                        
                        time.sleep(0.5)  # 短暫等待
                        continue
                    else:
                        raise e
                
                step += 1
                
            except Exception as e:
                print(f"Training error batch {batch_idx}: {e}")
                critical_error_count += 1  # 關鍵錯誤計數
                consecutive_errors += 1
                if critical_error_count > 1:
                    print("Too much critical error, interrupt training immediately!")
                    break

                self.optimizers['stage1'].zero_grad()
                if self.device == "cuda":
                    try:
                        torch.cuda.empty_cache()
                    except:
                        pass
                continue
        if len(epoch_metrics) == 0 or len(epoch_metrics['total_loss']) == 0:
            print("Warning: No successly trained batch")
            return {
                'total_loss': float('nan'), 
                'alignment_loss': float('nan'),
                'structure_loss': float('nan'),
                'grad_norm': float('nan')
            }
        
        return self.compute_epoch_metrics_safely(epoch_metrics)


    def _validate_stage1_epoch(self, epoch, total_epochs):
        """階段1驗證"""
        
        epoch_metrics = defaultdict(list)
        error_count = 0
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_dataloader, desc=f"Stage3-1: Val Epoch {epoch+1}")
            
            for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
                if fix_points is None:
                    continue
                
                if error_count > 2:
                    print("Too much valid error, valid early stop on validating")
                    break

                try:
                    fix_points = process_batch_data(fix_points, self.device)
                    break_points = process_batch_data(break_points, self.device)
                    if fix_points is None:
                        error_count += 1
                        continue
                    
                    fix_points = fix_points.to(self.device)
                    break_points = break_points.to(self.device)
                    
                    # 對齊驗證
                    z_original = self.diffusion_model.autoencoder.encode(fix_points)
                    condition_embedding = self.diffusion_model.encoder(break_points)
                    z_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_original)
                    recon_aligned = self.diffusion_model.autoencoder.decode(z_aligned)
                    
                    alignment_weight, structure_weight, dist_reg_weight, condition_weight = self.dynamic_loss_weights(epoch, total_epochs)

                    alignment_loss = F.mse_loss(recon_aligned, fix_points) * alignment_weight
                    structure_loss = F.mse_loss(z_aligned, z_original.detach()) * structure_weight
                    dist_reg_loss = compute_distribution_regularization(z_aligned) * dist_reg_weight
                    condition_consistency = F.mse_loss(
                        self.diffusion_model.forward_encoder_only(z_aligned),
                        condition_embedding.detach()
                    ) * condition_weight
                    
                    total_loss = alignment_loss + structure_loss + dist_reg_loss + condition_consistency
                    
                    batch_metrics = {
                        'alignment_loss': self.safe_tensor_to_float(alignment_loss),
                        'structure_loss': self.safe_tensor_to_float(structure_loss),
                        'dist_reg_loss': self.safe_tensor_to_float(dist_reg_loss),
                        'condition_consistency': self.safe_tensor_to_float(condition_consistency),
                        'total_loss': self.safe_tensor_to_float(total_loss)
                    }
                    
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)
                    
                    # 更新進度條
                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Val_Align': f"{alignment_loss.item():.4f}",
                            'Val_Struct': f"{structure_loss.item():.4f}",
                            'Val_DistReg': f"{dist_reg_loss.item():.4f}",
                            'Val_CondCons': f"{condition_consistency.item():.4f}"
                        })
                    
                except Exception as e:
                    print(f"Stage3-1 Error valid batch{batch_idx}: {e}")
                    if self.device == "cuda":
                        torch.cuda.empty_cache()
                    continue
        
        return self.compute_epoch_metrics_safely(epoch_metrics)


    # 階段2的函數
    def _train_integrated_stage2(self, num_epochs):
        """階段2：整合版聯合訓練"""
        
        self._freeze_params_for_stage2()
        
        best_loss = float('inf')
        global_step = 0
        
        for epoch in range(num_epochs):
            
            # 訓練階段 - 使用類似現有擴散訓練的邏輯
            self.diffusion_model.train()
            train_metrics = self._train_stage2_epoch(epoch, num_epochs, global_step)
            global_step += len(self.train_dataloader)
            
            # 驗證階段
            self.diffusion_model.eval()
            val_metrics = self._validate_stage2_epoch(epoch)
            
            # 記錄指標
            self._log_stage_metrics('stage2', epoch, train_metrics, val_metrics)
            
            # 模型保存
            current_loss = val_metrics.get('total_loss', float('inf'))
            if current_loss < best_loss:
                best_loss = current_loss
                self._save_stage_checkpoint('stage2', epoch, current_loss, is_final=False)
            # 最後一個epoch時再存一次
            if epoch == num_epochs - 1:
                self._save_stage_checkpoint('stage2', epoch, current_loss, is_final=True)
            
            # 早停檢查
            if self.early_stoppers['stage2'](current_loss, train_metrics.get('grad_norm', 0)):
                print(f"階段2早停於 Epoch {epoch+1}")
                break
            
            # 記憶體清理
            if self.device == "cuda":
                torch.cuda.empty_cache()

    def _train_stage2_epoch(self, epoch, total_epochs, global_step):
        """Stage 2 Improved epoch training: full error checking and professional logging."""
        epoch_metrics = defaultdict(list)
        progress_bar = tqdm(self.train_dataloader, desc=f"Stage3-2: Epoch {epoch+1}/{total_epochs}")

        consecutive_errors = 0
        critical_error_count = 0
        step = 0
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            # 跳過無效資料
            if fix_points is None:
                continue

            # 連續錯誤超限則提前終止
            if consecutive_errors > 3:
                print(f"[Epoch {epoch} Batch {batch_idx}] Consecutive error limit reached. Stopping epoch early.")
                break
            if critical_error_count > 1:
                print(f"[Epoch {epoch} Batch {batch_idx}] Too many critical errors. Stopping epoch early.")
                break

            try:
                # === 資料處理及數值檢查 ===
                fix_points = process_batch_data(fix_points, self.device)
                break_points = process_batch_data(break_points, self.device)
                if fix_points is None or break_points is None:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Data processing returned None, skipping batch.")
                    consecutive_errors += 1
                    continue
                if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] fix_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue
                if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] break_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue
                fix_points = fix_points.to(self.device)
                break_points = break_points.to(self.device)

                self.optimizers['stage2'].zero_grad()

                # === 前向 & 損失計算 ===
                with torch.no_grad():
                    try:
                        z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                        if torch.isnan(z_0).any():
                            print(f"[Epoch {epoch} Batch {batch_idx}] Latent encoding produced NaN, skipping batch.")
                            consecutive_errors += 1
                            continue
                    except Exception as e:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Error during latent encoding: {e}")
                        consecutive_errors += 1
                        continue

                try:
                    z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                    if torch.isnan(z_0_aligned).any():
                        print(f"[Epoch {epoch} Batch {batch_idx}] Aligned latent contains NaN, skipping batch.")
                        consecutive_errors += 1
                        continue

                    B = z_0_aligned.shape[0]
                    t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                    z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                    predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                    losses = compute_diffusion_losses_enhanced(
                        predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                    )
                    alignment_consistency = compute_alignment_consistency_loss(
                        z_0, z_0_aligned, fix_points, self.diffusion_model.autoencoder
                    ) * 0.1

                    total_loss = compute_weighted_diffusion_loss(
                        losses, global_step + step, len(self.train_dataloader) * total_epochs
                    )
                    total_loss = total_loss + alignment_consistency
                    
                    # 損失有效性檢查
                    if not torch.isfinite(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Loss is not finite, skipping batch.")
                        consecutive_errors += 1
                        continue
                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Error during forward/loss process: {e}")
                    consecutive_errors += 1
                    continue

                # === 反向 & 梯度檢查、優化 ===
                try:
                    total_loss.backward()

                    max_grad_norm = 1.0
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        self.diffusion_model.parameters(), max_grad_norm
                    )

                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Loss is NaN/Inf after backward, dropping batch.")
                        self.optimizers['stage2'].zero_grad()
                        consecutive_errors += 1
                        continue

                    grad_valid = True
                    for p in self.diffusion_model.parameters():
                        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                            grad_valid = False
                            break
                    if not grad_valid:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Invalid gradient detected, dropping batch.")
                        self.optimizers['stage2'].zero_grad()
                        consecutive_errors += 1
                        continue

                    self.optimizers['stage2'].step()
                    self.schedulers['stage2'].step()
               
                    # 記錄指標（與現有格式相同）
                    batch_metrics = {
                        'total_loss': total_loss.item(),
                        'noise_loss': losses.get('noise_mse', total_loss).item(),
                        'alignment_consistency': alignment_consistency.item(),
                        'grad_norm': grad_norm,
                        **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
                    }
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)

                    # 成功訓練批次後重置錯誤計數
                    consecutive_errors = 0
                    critical_error_count = 0

                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Loss': f"{total_loss.item():.4f}",
                            'Noise': f"{losses.get('noise_mse', total_loss).item():.4f}",
                            'GradNorm': f"{grad_norm:.2f}"
                        })
                    step += 1

                except RuntimeError as e:
                    if "CUDA" in str(e):
                        print(f"[Epoch {epoch} Batch {batch_idx}] CUDA runtime error: {str(e)[:100]}")
                        consecutive_errors += 1
                        self.optimizers['stage2'].zero_grad()
                        # CUDA error recovery
                        if self.device == "cuda":
                            try:
                                torch.cuda.synchronize()
                                torch.cuda.empty_cache()
                            except:
                                pass
                        time.sleep(0.5)
                        continue
                    else:
                        raise e

                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Unexpected training error: {e}")
                    consecutive_errors += 1
                    critical_error_count += 1
                    if critical_error_count > 1:
                        print(f"[Epoch {epoch}] Critical error limit reached, interrupting epoch.")
                        break
                    self.optimizers['stage2'].zero_grad()
                    if self.device == "cuda":
                        try:
                            torch.cuda.empty_cache()
                        except:
                            pass
                    continue
                
            except Exception as e:
                print(f"[Epoch {epoch} Batch {batch_idx}] Critical batch processing error: {e}")
                critical_error_count += 1
                consecutive_errors += 1
                if critical_error_count > 1:
                    print(f"[Epoch {epoch}] Critical error limit reached, interrupting epoch.")
                    break
                self.optimizers['stage2'].zero_grad()
                if self.device == "cuda":
                    try:
                        torch.cuda.empty_cache()
                    except:
                        pass
                continue

        if len(epoch_metrics) == 0 or len(epoch_metrics['total_loss']) == 0:
            print(f"[Epoch {epoch}] WARNING: No successfully trained batches this epoch.")
            return {
                'total_loss': float('nan'), 
                'noise_loss': float('nan'),
                'alignment_consistency': float('nan'),
                'grad_norm': float('nan')
            }

        return {key: np.mean(values) for key, values in epoch_metrics.items()}


    def _validate_stage2_epoch(self, epoch):
        """Stage 2 validation epoch (robust, professional log output)."""
        epoch_metrics = defaultdict(list)
        self.diffusion_model.eval()

        with torch.no_grad():
            progress_bar = tqdm(self.val_dataloader, desc=f"Stage2-Val-Epoch {epoch+1}")

            error_count = 0
            for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
                if fix_points is None:
                    continue
                try:
                    fix_points = process_batch_data(fix_points, self.device)
                    break_points = process_batch_data(break_points, self.device)

                    if fix_points is None or break_points is None:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation data processing produced None, skipping batch.")
                        error_count += 1
                        continue
                    if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation fix_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation break_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    fix_points = fix_points.to(self.device)
                    break_points = break_points.to(self.device)

                    # 推論流：AE -> latent_aligner -> 擴散流程
                    z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                    z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                    B = z_0_aligned.shape[0]
                    t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                    z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                    predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                    losses = compute_diffusion_losses_enhanced(
                        predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                    )
                    alignment_consistency = compute_alignment_consistency_loss(
                        z_0, z_0_aligned, fix_points, self.diffusion_model.autoencoder
                    ) * 0.1
                    total_loss = compute_weighted_diffusion_loss(
                        losses, 0, 1
                    ) + alignment_consistency

                    # Numeric safety check on validation loss
                    if not torch.isfinite(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Validation loss not finite, skipping batch.")
                        error_count += 1
                        continue

                    batch_metrics = {
                        'total_loss': total_loss.item(),
                        'noise_loss': losses.get('noise_mse', total_loss).item(),
                        'alignment_consistency': alignment_consistency.item(),
                        **{k: v.item() if torch.is_tensor(v) else v for k, v in losses.items()}
                    }
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)

                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Val_Loss': f"{total_loss.item():.4f}",
                            'Val_Noise': f"{losses.get('noise_mse', total_loss).item():.4f}"
                        })

                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Validation error: {e}")
                    error_count += 1
                    if self.device == "cuda":
                        torch.cuda.empty_cache()
                    continue

        if len(epoch_metrics) == 0 or len(epoch_metrics['total_loss']) == 0:
            print(f"[Epoch {epoch}] WARNING: No valid batches in validation.")
            return {
                'total_loss': float('nan'),
                'noise_loss': float('nan'),
                'alignment_consistency': float('nan')
            }

        return {key: np.mean(values) for key, values in epoch_metrics.items()}

    
    def _train_integrated_stage3(self, num_epochs):
        """階段3：整合版純擴散訓練"""
        
        # 訓練前GPU健康檢查
        if not self.check_gpu_health():
            print("GPU狀態異常，無法開始訓練")
            return

        # 凍結參數設置
        self._freeze_params_for_stage3()
        
        best_loss = float('inf')
        global_step = 0
        
        for epoch in range(num_epochs):
            
            # 訓練階段
            self.diffusion_model.train()
            train_metrics = self._train_stage3_epoch(epoch, num_epochs, global_step)
            global_step += len(self.train_dataloader)
            
            # 驗證階段
            self.diffusion_model.eval()
            val_metrics = self._validate_stage3_epoch(epoch)
            
            # 記錄指標
            self._log_stage_metrics('stage3', epoch, train_metrics, val_metrics)
            
            # 模型保存
            current_loss = val_metrics.get('total_loss', float('inf'))
            if current_loss < best_loss:
                best_loss = current_loss
                self._save_stage_checkpoint('stage3', epoch, current_loss, is_final=False)
            # 最後一個epoch時再存一次
            if epoch == num_epochs - 1:
                self._save_stage_checkpoint('stage3', epoch, current_loss, is_final=True)


            # 早停檢查
            if self.early_stoppers['stage3'](current_loss, train_metrics.get('grad_norm', 0)):
                print(f"階段3早停於 Epoch {epoch+1}")
                break
            
            # 記憶體清理
            if self.device == "cuda":
                torch.cuda.empty_cache()


    def _train_stage3_epoch(self, epoch, total_epochs, global_step):
        """階段3純擴散訓練epoch - 完整版本"""
        
        epoch_metrics = defaultdict(list)
        progress_bar = tqdm(self.train_dataloader, desc=f"Stage3-3: Epoch {epoch+1}/{total_epochs}")
        
        consecutive_errors = 0
        critical_error_count = 0
        step = 0
        
        for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
            if fix_points is None:
                continue
            
            # 錯誤控制檢查
            if consecutive_errors > 3:
                print(f"[Epoch {epoch} Batch {batch_idx}] Consecutive error limit reached. Stopping epoch early.")
                break
            if critical_error_count > 1:
                print(f"[Epoch {epoch} Batch {batch_idx}] Critical error limit reached, interrupting epoch.")
                break

            try:
                # === 資料處理及數值檢查 ===
                fix_points = process_batch_data(fix_points, self.device)
                break_points = process_batch_data(break_points, self.device)
                
                if fix_points is None or break_points is None:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Data processing returned None, skipping batch.")
                    consecutive_errors += 1
                    continue
                
                # NaN/Inf 檢查
                if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] fix_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue
                
                if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                    print(f"[Epoch {epoch} Batch {batch_idx}] break_points contains NaN/Inf, skipping batch.")
                    consecutive_errors += 1
                    continue

                self.optimizers['stage3'].zero_grad()

                # === 前向傳播及損失計算 ===
                with torch.no_grad():
                    try:
                        z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                        z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                        
                        # 檢查編碼結果
                        if torch.isnan(z_0).any() or torch.isnan(z_0_aligned).any():
                            print(f"[Epoch {epoch} Batch {batch_idx}] Encoded latents contain NaN, skipping batch.")
                            consecutive_errors += 1
                            continue
                    except Exception as e:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Error during encoding: {e}")
                        consecutive_errors += 1
                        continue

                try:
                    # 純擴散訓練
                    B = z_0_aligned.shape[0]
                    t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                    z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                    predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                    
                    # 計算擴散損失
                    losses = compute_diffusion_losses_enhanced(
                        predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                    )
                    
                    total_loss = compute_weighted_diffusion_loss(
                        losses, global_step + step, len(self.train_dataloader) * total_epochs
                    )
                    
                    # 損失有效性檢查
                    if not torch.isfinite(total_loss):
                        print(f"[Epoch {epoch} Batch {batch_idx}] Loss is not finite, skipping batch.")
                        consecutive_errors += 1
                        continue
                        
                except Exception as e:
                    print(f"[Epoch {epoch} Batch {batch_idx}] Error during forward/loss process: {e}")
                    consecutive_errors += 1
                    continue

                # === 反向傳播及梯度檢查 ===
                try:
                    total_loss.backward()
                    
                    # 標準梯度裁剪
                    max_grad_norm = 1.0
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        self.diffusion_model.parameters(), max_grad_norm
                    )
                    
                    # 梯度有效性檢查
                    grad_valid = True
                    for p in self.diffusion_model.parameters():
                        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()):
                            grad_valid = False
                            break
                    
                    if not grad_valid:
                        print(f"[Epoch {epoch} Batch {batch_idx}] Invalid gradient detected, dropping batch.")
                        self.optimizers['stage3'].zero_grad()
                        consecutive_errors += 1
                        continue
                    
                    self.optimizers['stage3'].step()
                    self.schedulers['stage3'].step()
                    
                    # === 記錄指標 ===
                    batch_metrics = {
                        'total_loss': self.safe_tensor_to_float(total_loss),
                        'noise_loss': self.safe_tensor_to_float(losses.get('noise_mse', total_loss)),
                        'grad_norm': self.safe_tensor_to_float(grad_norm),
                        **{k: self.safe_tensor_to_float(v) for k, v in losses.items()}
                    }
                    
                    # 成功訓練批次後重置錯誤計數
                    consecutive_errors = 0
                    critical_error_count = 0
                    
                    for key, value in batch_metrics.items():
                        epoch_metrics[key].append(value)
                    
                    # 更新進度條
                    if batch_idx % 10 == 0:
                        progress_bar.set_postfix({
                            'Loss': f"{total_loss.item():.4f}",
                            'Noise': f"{losses.get('noise_mse', total_loss).item():.4f}",
                            'GradNorm': f"{grad_norm:.2f}"
                        })
                    
                    step += 1
                    
                except RuntimeError as e:
                    if "CUDA" in str(e):
                        print(f"[Epoch {epoch} Batch {batch_idx}] CUDA runtime error: {str(e)[:100]}")
                        consecutive_errors += 1
                        self.optimizers['stage3'].zero_grad()
                        
                        # CUDA錯誤恢復
                        if self.device == "cuda":
                            try:
                                torch.cuda.synchronize()
                                torch.cuda.empty_cache()
                            except:
                                pass
                        time.sleep(0.5)
                        continue
                    else:
                        raise e
                        
            except Exception as e:
                print(f"[Epoch {epoch} Batch {batch_idx}] Unexpected training error: {e}")
                consecutive_errors += 1
                critical_error_count += 1
                if critical_error_count > 1:
                    print(f"[Epoch {epoch}] Critical error limit reached, interrupting epoch.")
                    break
                self.optimizers['stage3'].zero_grad()
                if self.device == "cuda":
                    try:
                        torch.cuda.empty_cache()
                    except:
                        pass
                continue
        
        # 檢查是否有成功的訓練批次
        if len(epoch_metrics) == 0 or len(epoch_metrics.get('total_loss', [])) == 0:
            print(f"[Epoch {epoch}] WARNING: No successfully trained batches this epoch.")
            return {
                'total_loss': float('nan'),
                'noise_loss': float('nan'), 
                'grad_norm': float('nan')
            }
        
        return self.compute_epoch_metrics_safely(epoch_metrics)


    def _validate_stage3_epoch(self, epoch):
        """Stage 3 Pure Diffusion Validation - Enhanced with comprehensive error checking"""
        
        epoch_metrics = defaultdict(list)
        error_count = 0
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_dataloader, desc=f"Stage3-Val: Epoch {epoch+1}")
            
            for batch_idx, (break_points, fix_points, norm_params) in enumerate(progress_bar):
                if fix_points is None:
                    continue
                
                # Error limit control
                if error_count > 2:
                    print(f"[Stage3-Val Epoch {epoch}] Too many validation errors, early termination.")
                    break
                
                try:
                    # === Data processing and validation ===
                    fix_points = process_batch_data(fix_points, self.device)
                    break_points = process_batch_data(break_points, self.device)
                    
                    if fix_points is None or break_points is None:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Data processing returned None, skipping batch.")
                        error_count += 1
                        continue
                    
                    # NaN/Inf validation checks
                    if torch.isnan(fix_points).any() or torch.isinf(fix_points).any():
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] fix_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    
                    if torch.isnan(break_points).any() or torch.isinf(break_points).any():
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] break_points contains NaN/Inf, skipping batch.")
                        error_count += 1
                        continue
                    
                    # Ensure tensors are on correct device
                    fix_points = fix_points.to(self.device)
                    break_points = break_points.to(self.device)
                    
                    # === Forward pass with pre-trained aligner ===
                    try:
                        z_0 = self.diffusion_model.autoencoder.encode(fix_points)
                        z_0_aligned = self.diffusion_model.latent_aligner.align_latent_space(z_0)
                        
                        # Validate encoded latents
                        if torch.isnan(z_0).any() or torch.isinf(z_0).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] z_0 contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                        
                        if torch.isnan(z_0_aligned).any() or torch.isinf(z_0_aligned).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] z_0_aligned contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                            
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during encoding: {e}")
                        error_count += 1
                        continue
                    
                    # === Diffusion forward process ===
                    try:
                        B = z_0_aligned.shape[0]
                        t = torch.randint(0, self.diffusion_model.num_timesteps, (B,), device=self.device)
                        z_t, noise = self.diffusion_model.add_noise(z_0_aligned, t)
                        predicted_noise = self.diffusion_model.forward(z_t, t, break_points)
                        
                        # Validate diffusion outputs
                        if torch.isnan(z_t).any() or torch.isinf(z_t).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] z_t contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                        
                        if torch.isnan(predicted_noise).any() or torch.isinf(predicted_noise).any():
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] predicted_noise contains NaN/Inf, skipping batch.")
                            error_count += 1
                            continue
                            
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during diffusion forward: {e}")
                        error_count += 1
                        continue
                    
                    # === Loss computation ===
                    try:
                        losses = compute_diffusion_losses_enhanced(
                            predicted_noise, noise, z_t, z_0_aligned, t, fix_points, self.diffusion_model
                        )
                        
                        total_loss = compute_weighted_diffusion_loss(losses, 0, 1)
                        
                        # Validate computed losses
                        if not torch.isfinite(total_loss):
                            print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] total_loss is not finite, skipping batch.")
                            error_count += 1
                            continue
                        
                        # Check individual loss components
                        for loss_name, loss_value in losses.items():
                            if torch.is_tensor(loss_value) and not torch.isfinite(loss_value):
                                print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] {loss_name} is not finite, skipping batch.")
                                error_count += 1
                                continue
                                
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during loss computation: {e}")
                        error_count += 1
                        continue
                    
                    # === Metrics recording ===
                    try:
                        batch_metrics = {
                            'total_loss': self.safe_tensor_to_float(total_loss),
                            'noise_loss': self.safe_tensor_to_float(losses.get('noise_mse', total_loss)),
                            **{k: self.safe_tensor_to_float(v) for k, v in losses.items()}
                        }
                        
                        # Validate metrics before recording
                        valid_metrics = True
                        for key, value in batch_metrics.items():
                            if not isinstance(value, (int, float)) or not np.isfinite(value):
                                print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Invalid metric {key}: {value}")
                                valid_metrics = False
                                break
                        
                        if not valid_metrics:
                            error_count += 1
                            continue
                        
                        # Record validated metrics
                        for key, value in batch_metrics.items():
                            epoch_metrics[key].append(value)
                        
                        # Reset error count on successful batch
                        if error_count > 0:
                            error_count = max(0, error_count - 1)  # Gradually reduce error count on success
                        
                        # Progress bar update
                        if batch_idx % 10 == 0:
                            progress_bar.set_postfix({
                                'Val_Loss': f"{total_loss.item():.4f}",
                                'Val_Noise': f"{losses.get('noise_mse', total_loss).item():.4f}"
                            })
                            
                    except Exception as e:
                        print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Error during metrics recording: {e}")
                        error_count += 1
                        continue
                    
                except Exception as e:
                    print(f"[Stage3-Val Epoch {epoch} Batch {batch_idx}] Unexpected validation error: {e}")
                    error_count += 1
                    if self.device == "cuda":
                        try:
                            torch.cuda.empty_cache()
                        except:
                            pass
                    continue
            
            # Final validation check
            if len(epoch_metrics) == 0 or len(epoch_metrics.get('total_loss', [])) == 0:
                print(f"[Stage3-Val Epoch {epoch}] WARNING: No valid batches in validation.")
                return {
                    'total_loss': float('nan'),
                    'noise_loss': float('nan')
                }
            
            print(f"[Stage3-Val Epoch {epoch}] Validation completed successfully with {len(epoch_metrics.get('total_loss', []))} valid batches.")
            return self.compute_epoch_metrics_safely(epoch_metrics)




    def safe_tensor_to_float(self, value):
        """安全地將任何值轉換為Python浮點數"""
        if torch.is_tensor(value):
            return float(value.detach().cpu().numpy())
        elif isinstance(value, (int, float)):
            return float(value)
        else:
            return float('nan')

    def compute_epoch_metrics_safely(self, epoch_metrics):
        """安全地計算epoch指標"""
        if not epoch_metrics:
            return {
                'total_loss': float('nan'),
                'alignment_loss': float('nan'), 
                'structure_loss': float('nan'),
                'grad_norm': float('nan')
            }
        
        result = {}
        for key, values in epoch_metrics.items():
            if not values:
                result[key] = float('nan')
            else:
                clean_values = [self.safe_tensor_to_float(v) for v in values]
                result[key] = float(np.mean(clean_values))
        
        return result
    

    # TensorBoard紀錄和報告生成
    def _log_stage_metrics(self, stage, epoch, train_metrics, val_metrics):
        """記錄階段指標到TensorBoard"""
        
        writer = self.writers[stage]
        
        # 記錄訓練指標
        for key, value in train_metrics.items():
            writer.add_scalar(f'{stage.capitalize()}_Train/{key}', value, epoch)
            self.stage_metrics[stage]['train'][key].append(value)
        
        # 記錄驗證指標
        for key, value in val_metrics.items():
            writer.add_scalar(f'{stage.capitalize()}_Val/{key}', value, epoch)
            self.stage_metrics[stage]['val'][key].append(value)
        
        # 記錄學習率
        current_lr = self.optimizers[stage].param_groups[0]['lr']
        writer.add_scalar(f'{stage.capitalize()}_Learning_Rate', current_lr, epoch)
        self.stage_metrics[stage]['lr'].append(current_lr)
        
        # 記錄梯度範數
        if 'grad_norm' in train_metrics:
            self.stage_metrics[stage]['grad_norm'].append(train_metrics['grad_norm'])
        
        # 損失對比圖
        if 'total_loss' in train_metrics and 'total_loss' in val_metrics:
            writer.add_scalars(f'{stage.capitalize()}_Loss_Comparison', {
                'Train': train_metrics['total_loss'],
                'Val': val_metrics['total_loss']
            }, epoch)
        
        # 打印總結
        print(f"\n{stage.upper()} Epoch {epoch+1} 總結:")
        print(f"學習率: {current_lr:.2e}")
        
        if stage == 'stage1':
            print(f"訓練 - 對齊損失: {train_metrics.get('alignment_loss', 0):.4f}, "
                f"結構損失: {train_metrics.get('structure_loss', 0):.4f}, "
                f"梯度範數: {train_metrics.get('grad_norm', 0):.2f}")
            print(f"驗證 - 對齊損失: {val_metrics.get('alignment_loss', 0):.4f}")
        else:
            print(f"訓練 - 總損失: {train_metrics.get('total_loss', 0):.4f}, "
                f"噪聲損失: {train_metrics.get('noise_loss', 0):.4f}, "
                f"梯度範數: {train_metrics.get('grad_norm', 0):.2f}")
            print(f"驗證 - 總損失: {val_metrics.get('total_loss', 0):.4f}, "
                f"噪聲損失: {val_metrics.get('noise_loss', 0):.4f}")

    def _plot_stage_training_curves(self, stage):
        """
        繪製訓練與驗證loss曲線，並自動儲存至對應目錄。
        """
        metrics = self.stage_metrics[stage]
        train = metrics['train']
        val = metrics['val']

        # 根據階段自動調整ylabel
        if stage == 'stage1':
            loss_key = 'alignment_loss'
            ylabel = 'Alignment Loss'
            title = 'Stage 1 Alignment Loss Curve'
        else:
            loss_key = 'total_loss'
            ylabel = 'Total Loss'
            title = f'Stage {stage[-1]} Diffusion Loss Curve'

        # 取對應損失（若key不存在會用空list）
        train_loss = train.get(loss_key, [])
        val_loss = val.get(loss_key, [])

        plt.figure(figsize=(8, 6))
        plt.plot(train_loss, label='Train')
        plt.plot(val_loss,   label='Validation')
        plt.xlabel('Epoch')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)

        save_dir = self.stage_log_dirs[stage]
        save_path = os.path.join(save_dir, f'{stage}_loss_curve.png')
        plt.savefig(save_path)
        plt.close()
        print(f"[{stage}] training loss curve saved: {save_path}")

    def _save_stage_report(self, stage):
        """
        儲存每個訓練階段的詳細總結報告（list形式每epoch所有主要指標）。
        格式：logs/stageX_alignment/stageX_report.json
        """
        metrics = self.stage_metrics[stage]
        report = {
            "train": dict(metrics['train']),
            "val": dict(metrics['val']),
            "lr": metrics['lr'],
            "grad_norm": metrics['grad_norm']
        }
        save_dir = self.stage_log_dirs[stage]
        save_path = os.path.join(save_dir, f'{stage}_report.json')

        # 轉存為簡單可移植的列表格式（確保 tensor/array 可序列化）
        # 對每個Key都自動將value轉為純Python型態
        for key in ["train", "val"]:
            for mname, mvalues in report[key].items():
                report[key][mname] = [float(v) for v in mvalues]

        report["lr"] = [float(v) for v in report["lr"]]
        report["grad_norm"] = [float(v) for v in report["grad_norm"]]

        with open(save_path, 'w') as f:
            json.dump(report, f, indent=2)
        print(f"[{stage}] 訓練詳細報告已儲存: {save_path}")


    def _generate_multi_stage_report(self):
        """生成多階段訓練綜合報告"""
        
        # 為每個階段生成訓練曲線
        for stage in ['stage1', 'stage2', 'stage3']:
            self._plot_stage_training_curves(stage)
            self._save_stage_report(stage)
        
        # 生成綜合報告
        comprehensive_report = {
            'multi_stage_training_summary': {
                'total_stages': 3,
                'training_completed': True,
                'stage_summaries': {}
            }
        }
        
        for stage in ['stage1', 'stage2', 'stage3']:
            metrics = self.stage_metrics[stage]
            if metrics['train'].get('total_loss') or metrics['train'].get('alignment_loss'):
                loss_key = 'total_loss' if 'total_loss' in metrics['train'] else 'alignment_loss'
                comprehensive_report['multi_stage_training_summary']['stage_summaries'][stage] = {
                    'epochs_completed': len(metrics['train'][loss_key]),
                    'best_train_loss': min(metrics['train'][loss_key]) if metrics['train'][loss_key] else 0,
                    'best_val_loss': min(metrics['val'][loss_key]) if metrics['val'][loss_key] else 0,
                    'final_lr': metrics['lr'][-1] if metrics['lr'] else 0
                }
        
        # 保存綜合報告
        with open(f'{self.log_dir}/multi_stage_comprehensive_report.json', 'w') as f:
            json.dump(comprehensive_report, f, indent=2)
        
        print(f"多階段訓練綜合報告已保存至: {self.log_dir}")


    def ensure_model_on_device(model, device):
        """確保模型及其所有組件都在指定設備上"""
        model = model.to(device)
        
        # 特別處理新添加的組件
        if hasattr(model, 'latent_aligner'):
            model.latent_aligner = model.latent_aligner.to(device)
            print(f"latent_aligner已移至設備: {device}")
        
        # 檢查所有參數是否在正確設備
        device_check = {}
        for name, module in model.named_modules():
            if len(list(module.parameters())) > 0:
                param_device = next(module.parameters()).device
                device_check[name] = param_device
        
        # print("設備檢查結果:")
        # for name, param_device in device_check.items():
        #     print(f"  {name}: {param_device}")
        
        return model

多階段訓練策略實作

In [19]:
def train_integrated_multi_stage_diffusion_model(diffusion_model, train_dataloader, val_dataloader,
                                                stage1_epochs=20, stage2_epochs=80, stage3_epochs=100,
                                                device='cuda', save_path_base="./models/diffusion_multistage",
                                                log_dir="./logs/integrated_multistage_training"):
    """整合版多階段擴散模型訓練主函數"""
    
    # 確保模型設備一致性
    diffusion_model = IntegratedMultiStageTrainingManager.ensure_model_on_device(diffusion_model, device)
    
    # 整合對齊機制
    diffusion_model = integrate_alignment_to_existing_model(diffusion_model, device)
    
    # 創建整合訓練管理器
    training_manager = IntegratedMultiStageTrainingManager(
        diffusion_model, train_dataloader, val_dataloader, device, log_dir
    )
    
    # 執行整合版多階段訓練
    stage_metrics = training_manager.run_integrated_multi_stage_training(
        stage1_epochs=stage1_epochs,
        stage2_epochs=stage2_epochs,
        stage3_epochs=stage3_epochs
    )
    
    # 保存最終模型
    final_model_path = f"{save_path_base}_integrated_final.pth"
    torch.save({
        'model_state_dict': diffusion_model.state_dict(),
        'stage_metrics': stage_metrics,
        'training_config': {
            'stage1_epochs': stage1_epochs,
            'stage2_epochs': stage2_epochs,
            'stage3_epochs': stage3_epochs
        }
    }, final_model_path)
    
    print(f"整合版多階段訓練完成，模型已保存至: {final_model_path}")
    
    return diffusion_model, stage_metrics

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Training Diffusion Model

test

In [20]:
print("CUDA可用:", torch.cuda.is_available())
try:
    test = torch.randn(100, 100).cuda()
    result = test @ test.T
    print("基本GPU操作正常")
    del test, result
    torch.cuda.empty_cache()
except Exception as e:
    print(f"基本GPU操作失敗: {e}")
    print("建議重啟Python kernel或系統")

CUDA可用: True
基本GPU操作正常


In [47]:
autoencoder = PointCloudAutoencoder(
    num_points=5000, 
    latent_dim=128,
    feature_dim=128
)
autoencoder = autoencoder.to(device)
diffusion_model = EnhancedConditionalDiffusionModel(
        autoencoder=autoencoder,
        feature_dim=128,
        num_points=5000
    )
diffusion_model = integrate_alignment_to_existing_model(diffusion_model, device)
diffusion_model = diffusion_model.to(device)
# 創建最小規模的測試
def minimal_alignment_test():
    """最小化對齊測試 - 修正版"""
    device = torch.device('cuda')
    
    # 極小的測試數據
    test_data = torch.randn(1, 5000, 3).to(device)
    
    # 測試AE編碼
    try:
        with torch.no_grad():
            z = autoencoder.encode(test_data)
        print(f"AE編碼成功: {z.shape}")
    except Exception as e:
        print(f"AE編碼失敗: {e}")
        return False
    
    # 測試對齊器
    try:
        z_aligned = diffusion_model.latent_aligner.align_latent_space(z)
        print(f"對齊成功: {z_aligned.shape}")
    except Exception as e:
        print(f"對齊失敗: {e}")
        return False
    
    # 測試擴散模型前向
    try:
        B = z_aligned.shape[0]
        t = torch.randint(0, diffusion_model.num_timesteps, (B,), device=device)
        z_t, noise = diffusion_model.add_noise(z_aligned, t)
        
        # 需要條件點雲
        condition_points = test_data  # 使用相同的測試數據作為條件
        predicted_noise = diffusion_model.forward(z_t, t, condition_points)
        
        print(f"擴散模型前向成功: predicted_noise {predicted_noise.shape}, noise {noise.shape}")
    except Exception as e:
        print(f"擴散模型前向失敗: {e}")
        return False
    
    return True

# 執行測試
if minimal_alignment_test():
    print("組件測試通過，問題可能在訓練循環")
else:
    print("組件測試失敗，GPU或模型有問題")

AE編碼成功: torch.Size([1, 128, 128])
對齊成功: torch.Size([1, 128, 128])


C:\Users\user\AppData\Local\Temp\ipykernel_22716\1934982287.py:73: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1807.)
  batch_std = z_flat.std(dim=0, keepdim=True) + 1e-6


擴散模型前向成功: predicted_noise torch.Size([1, 128, 128]), noise torch.Size([1, 128, 128])
組件測試通過，問題可能在訓練循環


In [56]:
# CUDA_LAUNCH_BLOCKING=1

In [21]:
torch.cuda.empty_cache()
# train_model(EnhancedConditionalDiffusionModel, dataloader)

In [22]:
if __name__ == "__main__":
    """擴散模型訓練"""
    
    autoencoder = PointCloudAutoencoder(
        num_points=5000, 
        latent_dim=128,
        feature_dim=128
    )
    
    # # 設定路徑
    # base_dir = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0824_0035")
    # models_dir = os.path.join(base_dir, "models")
    # os.makedirs(models_dir, exist_ok=True)  # 確保models目錄存在
    # save_path = os.path.join(models_dir, "autoencoder_pretrained_v8_0825_1146.pth")
    # log_dir = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0824_0035\logs\autoencoder_v8_0825_1146")
    
    # # 執行訓練
    # print("Step 1: 預訓練Autoencoder...")
    # train_results = train_enhanced_autoencoder(
    #     autoencoder=autoencoder,
    #     train_dataloader=train_dataloader,
    #     val_dataloader=val_dataloader,
    #     num_epochs=80,
    #     lr=2e-4,
    #     device='cuda' if torch.cuda.is_available() else 'cpu',
    #     save_path_ae=save_path,
    #     log_dir=log_dir
    # )
    # print("AutoEncoder預訓練完成！")

    print("Step 2: 載入AE預訓練權重...")
    pretrained_ae_path = converted_backslash(r"F:\Shawn\Diffusion\MTDC-A Mutilmodal Transformer Diffusion for Cranioplasty\LDM_training_v8_0913_1030\autoencoder_v8_0912_2137\autoencoder_pretrained_v8_0912_2137_best.pth")
    state_dict = torch.load(pretrained_ae_path, map_location="cuda" if torch.cuda.is_available() else 'cpu')
    if 'model_state_dict' in state_dict:
        autoencoder.load_state_dict(state_dict['model_state_dict'])
    else:
        autoencoder.load_state_dict(state_dict)
    # 凍結參數
    autoencoder.eval()
    for param in autoencoder.parameters():
        param.requires_grad = False
    

    diffusion_model = EnhancedConditionalDiffusionModel(
        autoencoder=autoencoder,
        feature_dim=128,
        num_points=5000
    )
    
    # 設定路徑
    models_dir = "./models"
    save_path = os.path.join(models_dir, "LDM_v8_0915_1610.pth")
    log_dir = "./logs/LDM_training_v8_0915_1610"
    
    # # 執行擴散模型訓練
    # print("Step 3: 訓練Diffusion Model...")
    # train_results = train_enhanced_diffusion_model(
    #     diffusion_model=diffusion_model,
    #     train_dataloader=train_dataloader,
    #     val_dataloader=val_dataloader,
    #     num_epochs=300,
    #     lr=1e-4,
    #     device='cuda' if torch.cuda.is_available() else 'cpu',
    #     save_path_diffusion=save_path,
    #     log_dir=log_dir
    # )

    # 執行擴散模型多階段訓練
    print("Step 3: 多階段訓練Diffusion Model...")
    trained_model, training_metrics = train_integrated_multi_stage_diffusion_model(
        diffusion_model=diffusion_model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        stage1_epochs=20,  # 潛在空間對齊
        stage2_epochs=80,  # 聯合訓練  
        stage3_epochs=100, # 純擴散訓練
        device='cuda' if torch.cuda.is_available() else 'cpu',
        save_path_base=save_path,
        log_dir=log_dir
    )

    print("擴散模型訓練流程設計完成！")
    print(f"訓練指標: {training_metrics}")

Step 2: 載入AE預訓練權重...
Step 3: 多階段訓練Diffusion Model...
=== 開始整合版多階段擴散模型訓練 ===

--- 階段1：潛在空間對齊 (20 epochs) ---


Stage3-1: Val Epoch 1: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it, Val_Align=0.0534, Val_Struct=0.0023, Val_DistReg=0.0004, Val_CondCons=0.0000]



STAGE1 Epoch 1 總結:
學習率: 5.00e-06
訓練 - 對齊損失: 0.0595, 結構損失: 0.0034, 梯度範數: 0.05
驗證 - 對齊損失: 0.0534
STAGE1 最佳模型已保存: Epoch 1, Loss: 0.0562
檢查點路徑: ./logs/LDM_training_v8_0915_1610\stage1_alignment\stage1_best.pth


Stage3-1: Val Epoch 2: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s, Val_Align=0.0431, Val_Struct=0.0026, Val_DistReg=0.0007, Val_CondCons=0.0000]



STAGE1 Epoch 2 總結:
學習率: 1.00e-05
訓練 - 對齊損失: 0.0501, 結構損失: 0.0028, 梯度範數: 0.03
驗證 - 對齊損失: 0.0431
STAGE1 最佳模型已保存: Epoch 2, Loss: 0.0464
檢查點路徑: ./logs/LDM_training_v8_0915_1610\stage1_alignment\stage1_best.pth


...


STAGE1 Epoch 15 總結:
學習率: 6.00e-06
訓練 - 對齊損失: 0.0354, 結構損失: 0.0064, 梯度範數: 0.07
驗證 - 對齊損失: 0.0375


Stage3-1: Val Epoch 16: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s, Val_Align=0.0396, Val_Struct=0.0074, Val_DistReg=0.0057, Val_CondCons=0.0008]



STAGE1 Epoch 16 總結:
學習率: 4.64e-06
訓練 - 對齊損失: 0.0327, 結構損失: 0.0063, 梯度範數: 0.07
驗證 - 對齊損失: 0.0396


Stage3-1: Val Epoch 17: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s, Val_Align=0.0372, Val_Struct=0.0081, Val_DistReg=0.0060, Val_CondCons=0.0010]



STAGE1 Epoch 17 總結:
學習率: 3.52e-06
訓練 - 對齊損失: 0.0311, 結構損失: 0.0068, 梯度範數: 0.06
驗證 - 對齊損失: 0.0372
階段1早停於 Epoch 17
[stage1] training loss curve saved: ./logs/LDM_training_v8_0915_1610\stage1_alignment\stage1_loss_curve.png
[stage1] 訓練詳細報告已儲存: ./logs/LDM_training_v8_0915_1610\stage1_alignment\stage1_report.json
[stage2] training loss curve saved: ./logs/LDM_training_v8_0915_1610\stage2_joint\stage2_loss_curve.png
[stage2] 訓練詳細報告已儲存: ./logs/LDM_training_v8_0915_1610\stage2_joint\stage2_report.json
[stage3] training loss curve saved: ./logs/LDM_training_v8_0915_1610\stage3_diffusion\stage3_loss_curve.png
[stage3] 訓練詳細報告已儲存: ./logs/LDM_training_v8_0915_1610\stage3_diffusion\stage3_report.json
多階段訓練綜合報告已保存至: ./logs/LDM_training_v8_0915_1610

=== 整合版多階段訓練完成 ===
整合版多階段訓練完成，模型已保存至: ./models\LDM_v8_0915_1610.pth_integrated_final.pth
擴散模型訓練流程設計完成！
訓練指標: {'stage1': {'train': defaultdict(<class 'list'>, {'alignment_loss': [0.05953712140520414, 0.05011482536792755, 0.05281660705804825, 0.050670009106

繪製圖表：AutoEncoder訓練損失

In [ ]:
# 可視化損失
import matplotlib.pyplot as plt
plt.plot(range(1, len(train_loss_ae) + 1), train_loss_ae, label="Train Total Loss",color="blue",alpha=0.7)
plt.plot(range(1, len(train_dcd_ae) + 1), train_dcd_ae, label="Train DCD Loss",color="orange", alpha=0.7)
plt.plot(range(1, len(train_scale_ae) + 1), train_scale_ae, label="Train Scale Loss",color='green', alpha=0.7)
plt.plot(range(1, len(train_normal_ae) + 1), train_normal_ae, label="Train Normal Loss",color='red', alpha=0.7)

# plt.plot(range(1, len(val_loss_ae) + 1), val_loss_ae, label="Valid Total Loss",color='blue', alpha=0.3)
plt.plot(range(1, len(val_dcd_ae) + 1), val_dcd_ae, label="Valid DCD Loss",color="orange", alpha=0.3)
plt.plot(range(1, len(val_scale_ae) + 1), val_scale_ae, label="Valid Scale Loss",color='green', alpha=0.3)
plt.plot(range(1, len(val_normal_ae) + 1), val_normal_ae, label="Valid Normal Loss",color='red', alpha=0.3)


plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale('log')  # 使用對數刻度
# plt.ylim(0,0.5) # 設置y軸上限
plt.legend()
plt.grid(True)
plt.title("Training for autoencoder_pretrained_v7_0823_1133")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 建立 2x3 子圖

# --- 第一列 ---
# 1. DCD Loss
axes[0,0].plot(range(1, len(train_dcd_ae)+1), train_dcd_ae, label="Train DCD", color="blue")
axes[0,0].plot(range(1, len(val_dcd_ae)+1), val_dcd_ae, label="Valid DCD", color="orange")
axes[0,0].set_title("DCD Loss")
axes[0,0].set_yscale("log")
axes[0,0].grid(True)
axes[0,0].legend()

# 2. Scale Loss
axes[0,1].plot(range(1, len(train_scale_ae)+1), train_scale_ae, label="Train Scale", color="blue")
axes[0,1].plot(range(1, len(val_scale_ae)+1), val_scale_ae, label="Valid Scale", color="orange")
axes[0,1].set_title("Scale Loss")
axes[0,1].set_yscale("log")
axes[0,1].grid(True)
axes[0,1].legend()

# 3. Normal Loss
axes[0,2].plot(range(1, len(train_normal_ae)+1), train_normal_ae, label="Train Normal", color="blue")
axes[0,2].plot(range(1, len(val_normal_ae)+1), val_normal_ae, label="Valid Normal", color="orange")
axes[0,2].set_title("Normal Loss")
axes[0,2].set_yscale("log")
axes[0,2].grid(True)
axes[0,2].legend()

# --- 第二列 ---
# 4. Total Loss
axes[1,0].plot(range(1, len(train_loss_ae)+1), train_loss_ae, label="Train Total", color="blue")
print(type(val_loss_ae))
print(val_loss_ae)
axes[1,0].plot(range(1, len(val_loss_ae)+1), val_loss_ae, label="Valid Total", color="orange")
axes[1,0].set_title("Total Loss")
axes[1,0].set_yscale("log")
axes[1,0].grid(True)
axes[1,0].legend()

# 5. All Train Losses
axes[1,1].plot(range(1, len(train_loss_ae)+1), train_loss_ae, label="Train Total", color="blue", alpha=0.7)
axes[1,1].plot(range(1, len(train_dcd_ae)+1), train_dcd_ae, label="Train DCD", color="orange", alpha=0.7)
axes[1,1].plot(range(1, len(train_scale_ae)+1), train_scale_ae, label="Train Scale", color="green", alpha=0.7)
axes[1,1].plot(range(1, len(train_normal_ae)+1), train_normal_ae, label="Train Normal", color="red", alpha=0.7)
axes[1,1].set_title("All Train Losses")
axes[1,1].set_yscale("log")
axes[1,1].grid(True)
axes[1,1].legend()

# 6. All Valid Losses
axes[1,2].plot(range(1, len(val_loss_ae)+1), val_loss_ae, label="Valid Total", color="blue", alpha=0.7)
axes[1,2].plot(range(1, len(val_dcd_ae)+1), val_dcd_ae, label="Valid DCD", color="orange", alpha=0.7)
axes[1,2].plot(range(1, len(val_scale_ae)+1), val_scale_ae, label="Valid Scale", color="green", alpha=0.7)
axes[1,2].plot(range(1, len(val_normal_ae)+1), val_normal_ae, label="Valid Normal", color="red", alpha=0.7)
axes[1,2].set_title("All Valid Losses")
axes[1,2].set_yscale("log")
axes[1,2].grid(True)
axes[1,2].legend()

# 總標題 & 美化
# fig.suptitle("Training for autoencoder_pretrained_v7_0819_0107", fontsize=16)
fig.suptitle("Training for LDM_v7_0823_1133", fontsize=16)
plt.tight_layout()
plt.show()

繪製圖表：Diffusion Model訓練損失

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 建立 2x3 子圖

train_losses, train_dcd_losses, train_scale_losses, train_normal_losses, val_losses, val_dcd_losses, val_scale_losses, val_normal_losses

# --- 第一列 ---
# 1. DCD Loss
axes[0,0].plot(range(1, len(train_dcd_losses)+1), train_dcd_losses, label="Train DCD", color="blue")
axes[0,0].plot(range(1, len(val_dcd_losses)+1), val_dcd_losses, label="Valid DCD", color="orange")
axes[0,0].set_title("DCD Loss")
axes[0,0].set_yscale("log")
axes[0,0].grid(True)
axes[0,0].legend()

# 2. Scale Loss
axes[0,1].plot(range(1, len(train_scale_losses)+1), train_scale_losses, label="Train Scale", color="blue")
axes[0,1].plot(range(1, len(val_scale_losses)+1), val_scale_losses, label="Valid Scale", color="orange")
axes[0,1].set_title("Scale Loss")
axes[0,1].set_yscale("log")
axes[0,1].grid(True)
axes[0,1].legend()

# 3. Normal Loss
axes[0,2].plot(range(1, len(train_normal_losses)+1), train_normal_losses, label="Train Normal", color="blue")
axes[0,2].plot(range(1, len(val_normal_losses)+1), val_normal_losses, label="Valid Normal", color="orange")
axes[0,2].set_title("Normal Loss")
axes[0,2].set_yscale("log")
axes[0,2].grid(True)
axes[0,2].legend()

# --- 第二列 ---
# 4. Total Loss
axes[1,0].plot(range(1, len(train_losses)+1), train_losses, label="Train Total", color="blue")
axes[1,0].plot(range(1, len(val_losses)+1), val_losses, label="Valid Total", color="orange")
axes[1,0].set_title("Total Loss")
axes[1,0].set_yscale("log")
axes[1,0].grid(True)
axes[1,0].legend()

# 5. All Train Losses
axes[1,1].plot(range(1, len(train_losses)+1), train_losses, label="Train Total", color="blue", alpha=0.7)
axes[1,1].plot(range(1, len(train_dcd_losses)+1), train_dcd_losses, label="Train DCD", color="orange", alpha=0.7)
axes[1,1].plot(range(1, len(train_scale_losses)+1), train_scale_losses, label="Train Scale", color="green", alpha=0.7)
axes[1,1].plot(range(1, len(train_normal_losses)+1), train_normal_losses, label="Train Normal", color="red", alpha=0.7)
axes[1,1].set_title("All Train Losses")
axes[1,1].set_yscale("log")
axes[1,1].grid(True)
axes[1,1].legend()

# 6. All Valid Losses
axes[1,2].plot(range(1, len(val_losses)+1), val_losses, label="Valid Total", color="blue", alpha=0.7)
axes[1,2].plot(range(1, len(val_dcd_losses)+1), val_dcd_losses, label="Valid DCD", color="orange", alpha=0.7)
axes[1,2].plot(range(1, len(val_scale_losses)+1), val_scale_losses, label="Valid Scale", color="green", alpha=0.7)
axes[1,2].plot(range(1, len(val_normal_losses)+1), val_normal_losses, label="Valid Normal", color="red", alpha=0.7)
axes[1,2].set_title("All Valid Losses")
axes[1,2].set_yscale("log")
axes[1,2].grid(True)
axes[1,2].legend()

# 總標題 & 美化
# fig.suptitle("Training for autoencoder_pretrained_v7_0819_0107", fontsize=16)
fig.suptitle("Training for LDM_v7_0823_1613", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 確保 dcd_ae 和 cd_losses_ae 是 numpy 陣列，並且長度一致
dcd_ae = np.array(dcd_ae)
mse_losses_ae = np.array(mse_losses_ae)
epochs = range(1, len(dcd_ae) + 1)

# 計算 DCD Loss 和 CD Loss 的差異
difference = dcd_ae - mse_losses_ae

# 創建圖表
fig, ax1 = plt.subplots()

# 主坐標軸繪製原始損失（DCD Loss 和 CD Loss）
ax1.plot(epochs, dcd_ae, label="DCD Loss", color="yellow", alpha=1)
ax1.plot(epochs, mse_losses_ae, label="CD Loss", color="blue", alpha=0.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (Log Scale)", color="blue")
ax1.set_yscale('log')  # 主坐標軸使用對數刻度
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend(loc="upper left")

# 創建次坐標軸繪製差異
ax2 = ax1.twinx()
ax2.plot(epochs, difference, label="DCD - CD Difference", color="red", linestyle="--")
ax2.set_ylabel("Difference", color="red")
ax2.legend(loc="upper right")

# 設置標題
plt.title("DCD Loss vs CD Loss with Difference")
plt.show()

# 可選：繪製相對差異（百分比）
relative_difference = (difference / cd_losses_ae) * 100  # 相對差異百分比
plt.figure()
plt.plot(epochs, relative_difference, label="Relative Difference (%)", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Relative Difference (%)")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.title("Relative Difference between DCD Loss and CD Loss")
plt.show()

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(range(1, len(dcd_ae)+1), dcd_ae, label="Total Loss", color="blue")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Total Loss", color = "tab:blue")
ax1.tick_params(axis="y", labelcolor = "tab:blue")
ax1.grid(True)

# 第二個 y 軸：MSE Loss 和 CD Loss（範圍較小）
ax2 = ax1.twinx()
ax2.plot(range(1, len(mse_losses_ae) + 1), mse_losses_ae, label="MSE Loss", color="tab:orange")
ax2.plot(range(1, len(cd_losses_ae) + 1), cd_losses_ae, label="CD Loss", color="tab:green")
ax2.set_ylabel("MSE & CD Loss", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

# 合併圖例
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.show()

In [ ]:
# def train_model(model, dataloader, num_epochs=200, lr=5e-5, save_path="ddpm_improved.pth"):
    # """Train the diffusion model"""
num_epochs = 200
lr = 5e-5
save_path="ddpm_v4_04_06.pth"
losses = [] # 建立一個空陣列來存放loss
accuracies = [] # 建立一個空陣列來記錄accuaracy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EnhancedConditionalDiffusionModel()  # 確保 model 是 `torch.nn.Module` 的實例
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for break_points, fix_points, norm_params in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_size = break_points.size(0)
        t = torch.randint(0, model.num_timesteps, (batch_size,), device=device)
        
        # Data augmentation
        break_points = break_points + torch.randn_like(break_points) * 0.01  # Add small noise
        fix_points = fix_points + torch.randn_like(fix_points) * 0.01
        
        x_t, noise = model.add_noise(fix_points, t)
        pred_noise = model(x_t, t, break_points)
        
        # Noise loss + Chamfer Distance loss
        noise_loss = F.mse_loss(pred_noise, noise)
        pred_points = model.sample(break_points, denormalize_params=norm_params)
        # cd_loss = chamfer_distance(pred_points, fix_points)
        # loss = noise_loss + 0.1 * cd_loss
        loss = noise_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(dataloader):.4f}")
    # Compute epoch-level loss and accuracy
    loss_per_epoch = total_loss / len(dataloader)
    losses.append(loss_per_epoch)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_per_epoch:.4f}")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"{save_path.split('.')[0]}_epoch{epoch+1}.pth"
        # torch.save(model.state_dict(), checkpoint_path)
        # 保存優化器和調度器狀態
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': loss_per_epoch
        }, checkpoint_path)
        print(f"Checkpoint saved at {checkpoint_path}")

Continue training from Epoch 50

In [ ]:
num_epochs = 100
continue_epochs = 50
lr = 5e-5
save_path="ddpm_v4_04_06.pth"
losses = [] # 建立一個空陣列來存放loss
accuracies = [] # 建立一個空陣列來記錄accuaracy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 載入模型
weight_path = converted_backslash(r"C:\Users\user\Desktop\小夜\GAN\cranioplasty_gan_2\LDM_training_v1_0428_1519\LDM_v1_0430_2021_epoch200.pth")
state_dict = torch.load(weight_path, map_location=device)
model = EnhancedConditionalDiffusionModel()  # 確保model是'torch.nn.Module' 的實例
model.to(device)
model.load_state_dict(state_dict)

# 初始化優化器和調度器
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
# 如果之前保存了優化器狀態（可選）
checkpoint = torch.load(weight_path, map_location=device)
# optimizer.load_state_dict(checkpoint['optimizer_state_dict']) # 幹我沒存到優化器狀態，下次記得要存!!
# scheduler.load_state_dict(checkpoint['scheduler_state_dict']) # 幹我沒存到調度器狀態，下次記得要存!!

In [ ]:
# 繼續訓練
for epoch in range(continue_epochs, num_epochs):
    model.train()
    total_loss = 0
    for break_points, fix_points, norm_params in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_size = break_points.size(0) # 批次大小從breakpoints的第一維提取
        t = torch.randint(0, model.num_timesteps, (batch_size,), device=device) # 隨機生成時間步長t(範圍0到500)
        
        # Data augmentation
        # 數據增強，添加均值為 0、標準差為 0.01 的隨機噪聲，模擬數據變異
        break_points = break_points + torch.randn_like(break_points) * 0.01 
        fix_points = fix_points + torch.randn_like(fix_points) * 0.01
        
        x_t, noise = model.add_noise(fix_points, t)
        pred_noise = model(x_t, t, break_points)
        
        
        # Noise loss + Chamfer Distance loss
        noise_loss = F.mse_loss(pred_noise, noise)
        pred_points = model.sample(break_points, denormalize_params=norm_params)
        cd_loss = chamfer_distance(pred_points, fix_points)
        loss = noise_loss + 0.1 * cd_loss

        optimizer.zero_grad() # 將優化器的梯度清零，為新一輪反向傳播準備
        loss.backward() # 計算loss對模型參數的梯度，進行反向傳播
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # 梯度裁剪到最大範數 1.0，防止梯度爆炸
        optimizer.step() # 更新模型參數
        
        total_loss += loss.item()
    
    scheduler.step() # 更新學習率調度器
    loss_per_epoch = total_loss / len(dataloader)
    losses.append(loss_per_epoch)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_per_epoch:.4f}")
    
    # 每 10 個 epoch 保存 checkpoint
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"{save_path.split('.')[0]}_epoch{epoch+1}.pth"
        # torch.save(model.state_dict(), checkpoint_path)
        # 保存優化器和調度器狀態
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': loss_per_epoch
        }, checkpoint_path)
        print(f"Checkpoint saved at {checkpoint_path}")

## ----------------------------------------------------------------------------------------

## -----------------------------------------------------------------------------------------

## Evaluation

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [35]:
break_points, fix_points, norm_params = dataset[00]

In [90]:
autoencoder = PointCloudAutoencoder(
        num_points=5000, 
        latent_dim=256,
        feature_dim=128
)

diffusion_model = EnhancedConditionalDiffusionModel(
    autoencoder=autoencoder,
    feature_dim=256,
    num_points=5000
)

weight_path = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107\models\LDM_v8_0904_1107_best.pth")
checkpoint = torch.load(weight_path, map_location=device)
diffusion_model.to(device)

# 提取模型參數
# 如果 checkpoint 是字典且包含 model_state_dict 鍵
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    diffusion_model.load_state_dict(checkpoint['model_state_dict']) # 從字典中取出真正的state_dict
else:
    # 否則直接載入，視為純 state_dict
    diffusion_model.load_state_dict(checkpoint)

# 創建模型實例
diffusion_model.eval()

EnhancedConditionalDiffusionModel(
  (autoencoder): PointCloudAutoencoder(
    (encoder): Sequential(
      (0): ResidualBlock(
        (conv1): Conv1d(3, 32, kernel_size=(1,), stride=(1,))
        (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU()
        (conv2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
        (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Conv1d(3, 32, kernel_size=(1,), stride=(1,))
      )
      (1): ResidualBlock(
        (conv1): Conv1d(32, 64, kernel_size=(1,), stride=(1,))
        (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU()
        (conv2): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Conv1d(32, 64, kernel_size=(1,), stride=(1,))
      )
      (2): ResidualBlo

create a method to show plot lines for loss ( Run Graph of Loss )

In [27]:
# def plot_lines(losses, start_epoch=1, step=10):
#   epochs = list(range(start_epoch, start_epoch + len(losses)*step, step))
#   arr = np.array(losses)

#   fig, ax = plt.subplots() # 生成圖表和對應的座標軸
#   ax.set_xlabel('Epoch') # 設x軸為Epoch(訓練週期)
#   ax.set_ylabel('Loss') # 設y軸為Loss(損失值)
#   ax.plot(epochs, arr, marker='', linestyle='-', label="loss")
#   ax.legend()
#   ax.grid(True)
#   # return fig

In [36]:
def plot_lines_per_10_epoch(losses1, step=10, title="title"):
    losses1 = np.array(losses1)
    # losses2 = np.array(losses2)
    # losses3 = np.array(losses3)
    epochs = np.arange(1, len(losses1) + 1)  # Create complete Epoch labels (1~100)
    
    # Calculate how many epochs we have
    max_epoch = len(losses1)
    
    # Create x-axis tick positions (0, 10, 20, ..., 100)
    x_ticks = np.arange(0, max_epoch + 1, step)
    
    # Create selected epochs for plotting loss values (1, 10, 20, ..., 100)
    # For epoch 1, we use index 0, for epoch 10, we use index 9, etc.
    selected_epochs = np.arange(10, max_epoch + 1, step)
    if selected_epochs[0] != 1:  # Make sure we include epoch 1
        selected_epochs = np.insert(selected_epochs, 0, 1)
        
    # Get corresponding losses for these selected epochs
    selected_losses = losses1[selected_epochs - 1]  # -1 because array is 0-indexed
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    
    # Plot all losses with light transparency
    ax.plot(epochs, losses1, linestyle="-", alpha=0.4, label="All Losses")
    
    # Plot selected epochs with blue markers
    ax.plot(selected_epochs, selected_losses, marker='o', color="orange", 
            linestyle="-", label="DCD Loss")
    
    ax.legend()
    ax.grid(True)  # Add grid for better readability
    
    # Set x-axis ticks to show 0, 10, 20, ..., 100
    plt.xticks(x_ticks)
    
    # Ensure the x-axis limits are set properly
    plt.xlim(0, max_epoch)
    
    # Optional: Make the figure look nicer
    title = f"Loss per 10 Epochs for Diffusion Model"
    plt.title(title)
    plt.tight_layout()
    return fig, selected_epochs  # Return the figure for further customization if needed

In [37]:
#-------------------------------------------------------------------------------
# Visualization Functions
#-------------------------------------------------------------------------------

def visualize_point_cloud_pair(brk=None, org = None,  gen = None, title="Point Cloud Comparison"):
    """Visualize broken skull and generated implant (and original if provided)"""
    # break_points = np.array(break_points, dtpye=np.float64).reshape(-1,3)
    # generated_points = np.array(generated_points, dtype=np.float64).reshape(-1,3)

    pcds = []
    
    if brk is not None:
        # Create point clouds
        break_pcd = o3d.io.read_point_cloud(brk)
        # break_pcd = o3d.geometry.PointCloud()
        # break_pcd.points = o3d.utility.Vector3dVector(brk)
        break_pcd.paint_uniform_color([0.7, 0.7, 0.7])  # Gray for broken skull
        pcds.append(break_pcd)
    
    if org is not None:
        org_pcd = o3d.io.read_point_cloud(org)
        # org_pcd = o3d.geometry.PointCloud()
        # org_pcd.points = o3d.utility.Vector3dVector(org)
        org_pcd.paint_uniform_color([0.8, 0, 0])  #Red for original fix
        pcds.append(org_pcd)

    if gen is not None:
        # gen_pcd = o3d.io.read_point_cloud(gen)
        gen_pcd = o3d.geometry.PointCloud()
        gen_pcd.points = o3d.utility.Vector3dVector(gen)
        gen_pcd.paint_uniform_color([0, 0.8, 0])  # Green for generated implant
        pcds.append(gen_pcd)
    
    # Visualize
    o3d.visualization.draw_geometries(pcds, window_name=title)

In [38]:
#-------------------------------------------------------------------------------
# Visualization Functions
#-------------------------------------------------------------------------------

def save_merged_point_cloud(output_path, brk=None, org = None,  gen = None, title="Point Cloud Comparison"):
    """Visualize broken skull and generated implant (and original if provided)"""
    # break_points = np.array(break_points, dtpye=np.float64).reshape(-1,3)
    # generated_points = np.array(generated_points, dtype=np.float64).reshape(-1,3)

    pcds = []
    
    if brk is not None:
        # Create point clouds
        break_pcd = o3d.io.read_point_cloud(brk)
        # break_pcd = o3d.geometry.PointCloud()
        # break_pcd.points = o3d.utility.Vector3dVector(brk)
        break_pcd.paint_uniform_color([0.7, 0.7, 0.7])  # Gray for broken skull
        pcds.append(break_pcd)
    
    if org is not None:
        org_pcd = o3d.io.read_point_cloud(org)
        # org_pcd = o3d.geometry.PointCloud()
        # org_pcd.points = o3d.utility.Vector3dVector(org)
        org_pcd.paint_uniform_color([0.8, 0, 0])  #Red for original fix
        pcds.append(org_pcd)

    if gen is not None:
        # gen_pcd = o3d.io.read_point_cloud(gen)
        gen_pcd = o3d.geometry.PointCloud()
        gen_pcd.points = o3d.utility.Vector3dVector(gen)
        gen_pcd.paint_uniform_color([0, 0.8, 0])  # Green for generated implant
        pcds.append(gen_pcd)
    
    if pcds is None:
        print("沒有可用點雲，合併錯誤")
        return
    
    # 合併所有點雲
    pcds_combined = pcds[0]
    for pcd in pcds[1:]:
        pcds_combined += pcd
    
    # 儲存.xyz檔案
    o3d.io.write_point_cloud(output_path, pcds_combined)
    print("合併點雲以成功儲存至指定路徑之檔案!!")

In [39]:
# 視覺化點雲
def visualize_xyz_concatenate_2D(break_file, fix_points, view_plane="xy"):
    # 載入點雲資料
    break_points = np.loadtxt(break_file, delimiter=" ")
    # fix_points = np.loadtxt(fix_file, delimiter=" ")
    
    # 設定視覺化的平面
    if view_plane == "xy":
        break_x, break_y = break_points[:, 0], break_points[:, 1]
        fix_x, fix_y = fix_points[:, 0], fix_points[:, 1]
        xlabel, ylabel = "X Axis", "Y Axis"
    elif view_plane == "xz":
        break_x, break_y = break_points[:, 0], break_points[:, 2]
        fix_x, fix_y = fix_points[:, 0], fix_points[:, 2]
        xlabel, ylabel = "X Axis", "Z Axis"
    else:
        raise ValueError("'view_plane 必須是xy或xz'")
    
    # 創建2D散點圖
    plt.figure(figsize=(10, 8))
    plt.scatter(break_x, break_y, s=1, c="gray", label="Break Skull")
    plt.scatter(fix_x, fix_y, s=1, c="blue", label="Fix Skull")
    
    # 設置標籤
    plt.xlabel('X Axis')
    plt.ylabel('Y Axis')
    plt.title('Break & Fix Skull')
    plt.legend()
    plt.grid(False)

    # 顯示圖形
    plt.show()

可視化自編碼器訓練結果

In [45]:
import open3d as o3d
def visualize_point_cloud(pred_points, fix_points=None, break_points=None):
    pred_pcd = o3d.geometry.PointCloud()
    pred_pcd.points = o3d.utility.Vector3dVector(pred_points.cpu().numpy()[0])
    pred_pcd.paint_uniform_color([0, 0.8, 0.5])  # 綠色
    
    gt_pcd = o3d.geometry.PointCloud()
    gt_pcd.points = o3d.utility.Vector3dVector(fix_points.cpu().numpy()[0])
    # gt_pcd.paint_uniform_color([0.7, 0.7, 0.7]) # 灰色
    gt_pcd.paint_uniform_color([1, 0, 0]) # 灰色
    
    # brk_pcd = o3d.geometry.PointCloud()
    # brk_pcd.points = o3d.utility.Vector3dVector(break_points.cpu().numpy()[0])
    # brk_pcd.paint_uniform_color([0.7, 0.7, 0.7]) # 灰色

    o3d.visualization.draw_geometries([pred_pcd, gt_pcd])
    # o3d.visualization.draw_geometries([pred_pcd, gt_pcd, brk_pcd])
    # o3d.visualization.draw_geometries([gt_pcd])
    # o3d.visualization.draw_geometries([pred_pcd])

In [41]:
autoencoder = PointCloudAutoencoder(num_points=5000, latent_dim=128, feature_dim=128)
autoencoder = autoencoder.to(device)
# 指定AE權重檔案路徑
weight_path = converted_backslash(r"C:\SHAWN\MTDC-A_Mutilmodal_Transformer_Diffusion_for_Cranioplasty\LDM_training_v8_0904_1107\models\autoencoder_pretrained_v8_0912_2137_best.pth")
# 載入權重
checkpoint = torch.load(weight_path, map_location=device)
# 如果 checkpoint 是字典且包含 model_state_dict 鍵
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    autoencoder.load_state_dict(checkpoint['model_state_dict'])
else:
    # 否則直接載入，視為純 state_dict
    autoencoder.load_state_dict(checkpoint)

In [55]:
autoencoder.eval()
with torch.no_grad():
    for _, fix_points, _ in train_dataloader:
        fix_points = fix_points.to(device)
        recon, _ = autoencoder(fix_points)
        visualize_point_cloud(recon, fix_points)
        break

可視化擴散模型訓練結果

In [121]:
with torch.no_grad():
        break_points_batch = break_points.to(device)  # [1, N, 3]
        if break_points.dim() == 2:
            break_points_batch = break_points.unsqueeze(0).to(device)
        else:
            break_points_batch = break_points.to(device)
        norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
        print("break_points_batch shape:", break_points_batch.shape)

        generated_points = diffusion_model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
        print("Raw generated_points shape:", generated_points.shape)
        
        # 安全調整形狀
        if generated_points.dim() == 3:  # [B, ?, ?]
            if generated_points.shape[0] == 1:  # [1, N, 3]
                generated_points = generated_points.squeeze(0)  # [N, 3]
            # elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:  # [1, 000, 3]
            #     generated_points = generated_points.squeeze(0)  # [5000, 3]
            # elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:  # [3, 5000, 3]
            #     # 假設第一維是多餘的，選擇第一個通道或平均
            #     generated_points = generated_points[0]  # [5000, 3]，取第一個通道
            #     # 或使用平均：generated_points = generated_points.mean(dim=0)  # [5000, 3]

# 轉換為 numpy
# break_points = break_points.cpu().numpy()
# fix_points = fix_points.cpu().numpy()
    
generated_points_00 = generated_points.cpu().numpy()

break_points_batch shape: torch.Size([1, 5000, 3])
condition_points shape inside model.sample(): torch.Size([1, 5000, 3])


Denoise Progress: 100%|██████████| 500/500 [00:05<00:00, 88.77it/s] 

Raw generated_points shape: torch.Size([5000, 3])


In [ ]:
generated_points_00 = generated_points

In [ ]:
generated_points_23 = generated_points

批次把訓練資料都拿來做預測

In [ ]:
gen_points_dict = {}
n = 99

for i in range(n+1):
    # 加載數據集
    break_points, fix_points, norm_params = dataset[i]  # 取第一個樣本
    # 生成修復點雲
    with torch.no_grad():
        break_points_batch = break_points.unsqueeze(0).to(device)  # [1, N, 3]
        norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
        # print("break_points_batch shape:", break_points_batch.shape)
        generated_points = model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
        print("Raw generated_points shape:", generated_points.shape)
        
        # 安全調整形狀
        if generated_points.dim() == 3:  # [B, ?, ?]
            if generated_points.shape[0] == 1:  # [1, N, 3]
                generated_points = generated_points.squeeze(0)  # [N, 3]
            elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:  # [1, 000, 3]
                generated_points = generated_points.squeeze(0)  # [5000, 3]
            elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:  # [3, 5000, 3]
                # 假設第一維是多餘的，選擇第一個通道或平均
                generated_points = generated_points[0]  # [5000, 3]，取第一個通道
                # 或使用平均：generated_points = generated_points.mean(dim=0)  # [5000, 3]

    # 轉換為 numpy
    # break_points = break_points.cpu().numpy()
    # fix_points = fix_points.cpu().numpy()
    
    gen_points_dict[i] = generated_points.squeeze(0).cpu().numpy()

In [ ]:
print(break_points.shape)
print(fix_points.shape)
print(generated_points.shape)

visualization

In [98]:
# 打印形狀和類型
print("break_points shape:", break_points.shape, "dtype:", break_points.dtype)
print("generated_points shape:", generated_points.shape, "dtype:", generated_points.dtype)
print("fix_points shape:", fix_points.shape, "dtype:", fix_points.dtype)

break_points shape: torch.Size([5000, 3]) dtype: torch.float32
generated_points shape: torch.Size([5000, 3]) dtype: torch.float32
fix_points shape: torch.Size([5000, 3]) dtype: torch.float32


In [111]:
file_path_break00_10000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_10000\000_break.xyz")
file_path_break23_10000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_10000\023_break.xyz")
file_path_break00_30000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_30000_7000\000_break.xyz")
file_path_break13_30000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_30000_7000\013_break.xyz")
file_path_break23_30000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_30000_7000\023_break.xyz")
file_path_break69_10000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_10000\069_break.xyz")
file_path_break88_30000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_30000_7000\088_break.xyz")
file_path_break99_30000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_30000_7000\099_break.xyz")
file_path_fix00_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\000_fix.xyz")
file_path_fix13_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\013_fix.xyz")
file_path_fix23_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\023_fix.xyz")
file_path_fix69_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\069_fix.xyz")
file_path_fix88_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\088_fix.xyz")
file_path_fix99_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\training_set\output_xyz_5000\099_fix.xyz")
# visualize_xyz_concatenate_2D(file_path_break00_5000, generated_points_calibration_v2, view_plane="xy")
# visualize_xyz_concatenate_2D(file_path_fit00_5000, generated_points_calibration_v2, view_plane="xz")

In [122]:
# 視覺化v1
visualize_point_cloud_pair(
    brk = file_path_break00_10000,
    org = file_path_fix00_5000,
    # gen = gen_points_dict[8],
    gen = generated_points_00,
    title="Skull Repair Visualization (Epoch 200)"
)

In [ ]:
output_dir = converted_backslash(r"C:\dataset\Skull Fix & Break\training_set\generated_output_xyz")
output_filename = f"ddpm_v3_0324_1421_epoch140_skull47_5000points.xyz"
save_path = os.path.join(output_dir, output_filename)

In [ ]:
save_merged_point_cloud(
    output_path = save_path,
    brk = file_path_break00_30000,
    gen = gen_points_dict[47],
)

In [ ]:
# # 視覺化v2
# visualize_point_cloud_pair_v2(
#     points1 = file_path_break00_30000,
#     points2 = file_path_fit00_5000,
#     points3 = generated_points,
#     # points2 = fix_points,
#     title="Skull Repair Visualization (Epoch 100)"
# )

## Visualizing the Model

In [ ]:
import torch
from torch.autograd import Variable
from torchviz import make_dot
# from EnhancedConditionalDiffusionModel import EnhancedConditionalDiffusionModel  # 假設程式碼在同一個文件中

# 確保設備配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 創建模型實例
model = EnhancedConditionalDiffusionModel(
    feature_dim=256,
    beta_schedule='cosine',
    num_points=5000,
    latent_dim=16,
    latent_feature_dim=64
).to(device)

# 設置模型為訓練模式
model.train()

# 創建隨機輸入數據
batch_size = 2
latent_dim = 16
latent_feature_dim = 64
num_points = 5000

# 模擬 z_t (latent representation)
z_t = torch.randn(batch_size, latent_dim, latent_feature_dim).to(device)
# 模擬 timestep
t = torch.randint(0, 500, (batch_size,), device=device).float() / 500  # 正規化到 [0, 1]
# 模擬 condition_points (point cloud)
condition_points = torch.randn(batch_size, num_points, 3).to(device)

# 將輸入轉為 Variable 以啟用梯度追蹤
z_t = Variable(z_t, requires_grad=True)
t = Variable(t, requires_grad=True)
condition_points = Variable(condition_points, requires_grad=True)

# 前向傳播
output = model(z_t, t, condition_points)

# 使用 torchviz 生成計算圖
dot = make_dot(output, params=dict(model.named_parameters()))

# 保存為 PDF 或 PNG 文件
dot.format = 'png'  # 也可以使用 'pdf' 或 'svg'
dot.render("enhanced_diffusion_model_architecture", view=True)  # view=True 會自動打開圖片

print("Model architecture visualization has been generated as 'enhanced_diffusion_model_architecture.png'.")

In [ ]:
# 確保設備配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 創建模型實例
model = EnhancedConditionalDiffusionModel(
    feature_dim=256,
    beta_schedule='cosine',
    num_points=5000,
    latent_dim=16,
    latent_feature_dim=64
).to(device)

# 設置模型為訓練模式
model.train()

# 創建精簡的隨機輸入數據
batch_size = 1  # 減少 batch size 以簡化圖表
latent_dim = 16
latent_feature_dim = 64
num_points = 500

# 模擬 z_t (latent representation)
z_t = torch.randn(batch_size, latent_dim, latent_feature_dim).to(device)
# 模擬單一時間步
t = torch.tensor([250], device=device).float() / 500  # 使用中間時間步，減少時間嵌入複雜性
# 模擬 condition_points (點雲，減少點數以簡化)
condition_points = torch.randn(batch_size, num_points, 3).to(device)

# 將輸入轉為 Variable 以啟用梯度追蹤
z_t = Variable(z_t, requires_grad=True)
t = Variable(t, requires_grad=True)
condition_points = Variable(condition_points, requires_grad=True)

# 前向傳播
output = model(z_t, t, condition_points)

# 使用 torchviz 生成計算圖，僅顯示主要參數
dot = make_dot(output, params={name: param for name, param in model.named_parameters() if 'weight' in name})

# 設置圖表屬性以提高易讀性
dot.attr(rankdir='TB')  # 從上到下佈局
dot.attr('node', shape='box', style='filled', fillcolor='lightblue')
dot.attr('edge', color='blue')

# 保存為 PNG 文件
dot.format = 'png'
dot.render("simplified_enhanced_diffusion_model_architecture", view=True)

print("Simplified model architecture visualization has been generated as 'simplified_enhanced_diffusion_model_architecture.png'.")

## Test

In [ ]:
test_path = r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz"
xyz_testing_data = converted_backslash(test_path)
print(xyz_testing_data)

In [ ]:
if not os.path.exists(xyz_testing_data):
    raise FileNotFoundError(f"錯誤資料夾路徑{xyz_testing_data}不存在，請檢查路徑!")
print(os.listdir(xyz_testing_data))

In [ ]:
# Creat DataLoader
dataset_test = SkullDataset2(data_dir = xyz_testing_data, num_points = 5000)
if len(dataset_test) == 0:
    raise ValueError("錯誤: SkullDataset2 載入的樣本數為 0，請檢查數據來源！")
dataloader = DataLoader(
    dataset_test, 
    batch_size = 8,
    shuffle = True, 
    collate_fn=collate_fn,
    # pin_memory=True, # 加速CPU -> GPU數據傳輸
    # num_workers=0
)
print(len(dataloader))

Single Data Testing

In [ ]:
break_points, fix_points, norm_params = dataset_test[00]  # 取單個樣本

Test AutoEncoder

In [ ]:
# AE test
model.autoencoder.eval()
with torch.no_grad():
    for _, fix_points, _ in dataloader:
        fix_points = fix_points.to(device)
        recon, _ = model.autoencoder(fix_points)
        visualize_point_cloud(recon, fix_points)
        break

Test Diffusion Model

In [ ]:
# 生成修復點雲
with torch.no_grad():
    break_points_batch = break_points.unsqueeze(0).to(device)  # [1, N, 3]
    norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
    # print("break_points_batch shape:", break_points_batch.shape)
    generated_points = model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
    print("Raw generated_points shape:", generated_points.shape)
    
    # 安全調整形狀
    if generated_points.dim() == 3:  # [B, ?, ?]
        if generated_points.shape[0] == 1:  # [1, N, 3]
            generated_points = generated_points.squeeze(0)  # [N, 3]
        elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:  # [1, 000, 3]
            generated_points = generated_points.squeeze(0)  # [5000, 3]
        elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:  # [3, 5000, 3]
            # 假設第一維是多餘的，選擇第一個通道或平均
            generated_points = generated_points[0]  # [5000, 3]，取第一個通道
            # 或使用平均：generated_points = generated_points.mean(dim=0)  # [5000, 3]

# 轉換為 numpy
# break_points = break_points.cpu().numpy()
# fix_points = fix_points.cpu().numpy()
gereated_points_test_00 = generated_points.squeeze(0).cpu().numpy()

Whole dataset Testing

In [ ]:
gen_points_test_dict = {}
n = 99

for i in range(n+1):
    # 加載數據集
    break_points, fix_points, norm_params = dataset_test[i]  # 取單個樣本
    # 生成修復點雲
    with torch.no_grad():
        break_points_batch = break_points.unsqueeze(0).to(device)  # [1, N, 3]
        norm_params_batch = {k: v.unsqueeze(0).to(device) for k, v in norm_params.items()}
        # print("break_points_batch shape:", break_points_batch.shape)
        generated_points = model.sample(break_points_batch, num_points=5000, denormalize_params=norm_params_batch)
        print("Raw generated_points shape:", generated_points.shape)
        
        # 安全調整形狀
        if generated_points.dim() == 3:  # [B, ?, ?]
            if generated_points.shape[0] == 1:  # [1, N, 3]
                generated_points = generated_points.squeeze(0)  # [N, 3]
            elif generated_points.shape[1] == 5000 and generated_points.shape[2] == 3:  # [1, 000, 3]
                generated_points = generated_points.squeeze(0)  # [5000, 3]
            elif generated_points.shape[0] == 3 and generated_points.shape[1] == 5000:  # [3, 5000, 3]
                # 假設第一維是多餘的，選擇第一個通道或平均
                generated_points = generated_points[0]  # [5000, 3]，取第一個通道
                # 或使用平均：generated_points = generated_points.mean(dim=0)  # [5000, 3]

    # 轉換為 numpy
    # break_points = break_points.cpu().numpy()
    # fix_points = fix_points.cpu().numpy()

    gen_points_test_dict[i] = generated_points.squeeze(0).cpu().numpy()

In [ ]:
print(gen_points_test_dict)

In [ ]:
generated_points_test_99 = generated_points

In [ ]:
generated_points_test_00 = generated_points

In [ ]:
test_00_5000 = converted_backslash(r"F:\Shawn\Dataset & Image\Skull Fix & Break\test_set_for_participants\output_xyz\000.xyz")
test_23_5000 = converted_backslash(r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz\023.xyz")
test_50_5000 = converted_backslash(r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz\050.xyz")
test_99_5000 = converted_backslash(r"C:\dataset\Skull Fix & Break\test_set_for_participants\output_xyz\099.xyz")

In [ ]:
# 測試集顱骨視覺化
visualize_point_cloud_pair(
    brk = test_00_5000,
    # org = file_path_fix00_5000,
    # gen = gen_points_test_dict[00],
    gen = gereated_points_test_00,
    title="Skull Repair Visualization (Epoch 100)"
)

In [ ]:
def evaluate_model(model, dataset, num_samples=5):
    """Evaluate model on test samples and visualize results"""
    model.eval()
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            break_points, fix_points, norm_params = dataset[i]
            
            # Add batch dimension
            break_points = break_points.unsqueeze(0).to(device)  # [1, N, 3]
            fix_points = fix_points.unsqueeze(0).to(device)      # [1, N, 3]
            norm_params = {k: v.unsqueeze(0).to(device) for k,v in norm_params.keys()}
            
            # 生成修復點雲
            generated_points = model.sample(break_points, num_points=5000, denormalize_params=norm_params)
            generated_points = generated_points.squeeze(0).cpu().numpy()  # [N, 3]
            
            # 轉換為 numpy 格式
            break_points_np = break_points.squeeze(0).cpu().numpy()  # [N, 3]
            fix_points_np = fix_points.squeeze(0).cpu().numpy()      # [N, 3]
            
            # 視覺化
            visualize_point_cloud_pair(
                brk=break_points_np,
                org=fix_points_np,
                gen=generated_points,
                title=f"Sample {i+1}: Skull Repair Comparison"
            )
# 呼叫 evaluate_model
model = EnhancedConditionalDiffusionModel(feature_dim=256, beta_schedule='linear')
model.load_state_dict(state_dict)
model.to(device)
dataset = SkullDataset2(data_dir=xyz_training_data, num_points=5000)
evaluate_model(model, dataset, num_samples=3)  # 評估 3 個樣本